# BLAZE — configuração e ensaios do protótipo

Execute as seções em ordem: conexão, visão, controle, hardware, combinação oficial, firmware, testes físicos e central do operador.


## 1. Verificar ambiente

**Objetivo:** confirmar que o kernel preparado foi selecionado e carregar as funções compartilhadas com o runtime.

**Entrada:** Kernel de `.bootstrap/python312` selecionado no VS Code.

**Resultado:** caminho do Python em `.bootstrap/python312`. Esta célula não acessa hardware.

**Código chamado:** `scripts/preparar_notebooks.py`, as funções deste notebook, `pc_operador/runtime.py`.


In [ ]:
from pathlib import Path
import sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'pc_operador' / 'runtime.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Abra a pasta blaze_prototipo_2026_10 no VS Code.')
EXPECTED = ROOT / '.bootstrap' / 'python312'
if not Path(sys.executable).resolve().is_relative_to(EXPECTED.resolve()):
    raise RuntimeError('Rode PRIMEIRO scripts/preparar_notebooks_windows.cmd no Windows ou a tarefa de preparação no Ubuntu; então selecione o kernel em .bootstrap/python312.')
from IPython.display import display, Image
import ipywidgets as widgets
sys.path.insert(0, str(ROOT / 'pc_operador'))
from runtime import AppConfig, discover_services, estimate
results = {}
print('Kernel pronto:', sys.executable)

In [ ]:
import copy, datetime as _setup_datetime, hashlib, html, re
import ipywidgets as widgets
from pathlib import Path

# Setups editáveis e combinação oficial usados pelos dois notebooks.
SETUPS_DIR = ROOT / 'pc_operador' / 'parameters_setups'
SETUP_SECTIONS = ('connection', 'vision', 'control', 'hardware')

def _setup_name(name):
    name = str(name).strip()
    if not name or any(ord(ch)<32 for ch in name) or '/' in name or '\\' in name:
        raise ValueError('Informe um nome sem barras ou caracteres de controle.')
    return name

def auto_setup_name(section, payload):
    """Nome descritivo e estável, sem endereço de arquivo ou digitação pelo operador."""
    def token(value):
        return re.sub(r'[^A-Za-z0-9]+', '-', str(value)).strip('-')[:24] or 'Default'
    def model(value):
        return token(Path(str(value).replace('\\', '/')).stem)
    if section == 'connection':
        esp=payload.get('esp',{});geo=payload.get('geometry',{})
        part=f"ESP-{token(esp.get('transport','auto'))}_Tr{float(geo.get('track_width_m') or 0):g}m_R{float(geo.get('sprocket_radius_m') or 0):g}m"
    elif section == 'vision':
        v=payload['vision'];src=v.get('camera_source',0)
        source=('Video-'+model(src) if v.get('source_type')=='video'
                else 'CameraUSB-'+token(src))
        part=f"Visao-{token(v.get('combat_mode','simulation'))}_{source}_F-{model(v.get('fire_model','fogo'))}_P-{model(v.get('person_model','pessoas'))}"
    elif section == 'control':
        c=payload.get('control',{})
        part=f"Xbox-{token(payload.get('xbox_index',0))}_{int(c.get('speed_limit_rpm',80))}RPM_Camera-{int(c.get('camera_recenter_step_deg',20))}graus"
    elif section == 'hardware':
        h=payload['hardware'];motor={'dc_3_6v':'DC3-6V','bts7960_12v':'Mabuchi12V'}.get(h.get('drive_kind'),h.get('drive_kind','DC'))
        part=f"ESP-{token(motor)}_{token(h.get('drive_driver_id','BTS7960'))}_P-{token(h.get('pan_servo_kind',h.get('cannon_motor_id','mg996r')))}_T-{token(h.get('tilt_servo_kind','mg996r'))}_C-{token(h.get('camera_servo_kind','sg90'))}_F-SG90_{'Fisico' if h.get('physical_outputs') else 'Simulacao'}"
        if h.get('jet_output_enabled'):part+='_Jato'
    else:raise ValueError('Grupo de setup desconhecido.')
    digest=hashlib.sha256(json.dumps(payload,sort_keys=True,ensure_ascii=False,
                        default=str).encode('utf-8')).hexdigest()[:6]
    return part[:112].rstrip('-_')+'_'+digest

def _setup_path(section):
    if section not in (*SETUP_SECTIONS, 'combinations'):
        raise ValueError('Tipo de setup desconhecido.')
    return SETUPS_DIR / 'user' / (section + '.json')

def _json_atomic(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)

def setup_all(section):
    path = _setup_path(section)
    if not path.is_file(): return {}
    data = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(data, dict): raise ValueError(f'JSON de setups inválido: {path}')
    return data

def setup_names(section):
    return sorted(setup_all(section), key=str.casefold)

def setup_get(section, name):
    name = _setup_name(name)
    return copy.deepcopy(setup_all(section)[name])

def setup_save(section, name, payload):
    name = _setup_name(name)
    if section != 'combinations' and '*' in name:
        raise ValueError('O caractere * separa os grupos na combinação; escolha outro caractere para o nome do setup.')
    records = setup_all(section)
    payload = copy.deepcopy(payload)
    if section != 'combinations':
        payload['_meta'] = {'saved_at': _setup_datetime.datetime.now().astimezone().isoformat()}
    records[name] = payload
    _json_atomic(_setup_path(section), records)
    return name

def setup_delete(section, name):
    name = _setup_name(name)
    records = setup_all(section)
    if name not in records: raise KeyError(name)
    if section == 'combinations' and official_name() == name:
        raise ValueError('Este é o setup oficial. Ative outro antes de excluir.')
    for combo_name, combo in setup_all('combinations').items():
        if section in SETUP_SECTIONS and combo.get('sections', {}).get(section) == name:
            raise ValueError(f'Está em uso na combinação {combo_name!r}; edite ou exclua a combinação primeiro.')
    del records[name]
    _json_atomic(_setup_path(section), records)

def official_name():
    path = SETUPS_DIR / 'official' / 'active.json'
    if not path.is_file(): return ''
    return str(json.loads(path.read_text(encoding='utf-8')).get('_official', {}).get('name', ''))

def assemble_combination(selections):
    base_path = ROOT / 'pc_operador' / 'config.example.json'
    data = json.loads(base_path.read_text(encoding='utf-8'))
    for section in SETUP_SECTIONS:
        selected = selections.get(section, '')
        payload = (setup_get(section, selected) if selected else
                   json.loads((SETUPS_DIR/'defaults'/f'{section}.json').read_text(encoding='utf-8')))
        for key, value in payload.items():
            if key in ('_meta', 'saved_at'): continue
            if isinstance(value, dict) and isinstance(data.get(key), dict):
                data[key].update(copy.deepcopy(value))
            else:
                data[key] = copy.deepcopy(value)
    if selections.get('vision'):
        data.setdefault('camera', {})['source'] = data['vision'].get('camera_source', 0)
    rpm = data.get('control', {}).get('speed_limit_rpm', 0)
    max_rpm = data.get('geometry', {}).get('output_max_rpm', 0)
    if not 10 <= float(rpm) <= 80:
        raise ValueError('O limite de velocidade deve ficar entre 10 e 80 RPM.')
    if float(max_rpm) < 0:
        raise ValueError('RPM máxima calibrada não pode ser negativa.')
    return data

def activate_combination(name):
    name = _setup_name(name)
    combo = setup_get('combinations', name)
    selections = combo['sections']
    data = assemble_combination(selections)
    data['_official'] = {'name': name, 'sections': selections,
                         'activated_at': _setup_datetime.datetime.now(_setup_datetime.timezone.utc).isoformat()}
    _json_atomic(SETUPS_DIR / 'official' / 'active.json', data)
    return data

def status_box(title, state, description):
    colors = {'OK': '#17803d', 'FALHA': '#b42318', 'PENDENTE': '#9a6700', 'TESTANDO': '#1e65aa'}
    color = colors.get(state, '#6b7280')
    return widgets.HTML('<div style="border:1px solid #d1d5db;border-radius:7px;padding:8px;margin:3px">'
                        f'<b style="color:{color};display:inline-block;width:78px">{html.escape(state)}</b>'
                        f'<strong>{html.escape(title)}</strong><span style="margin-left:10px;color:#444">'
                        f'{html.escape(description)}</span></div>')


## 2. Conexão do ESP32 e geometria

Selecione USB ou Wi-Fi local. A porta USB aparece na lista depois de conectar o ESP32 e clicar **Atualizar COM**; não é preciso digitá-la. Informe as medidas conhecidas das esteiras e salve. O nome do setup é gerado dos parâmetros. O vídeo é escolhido na seção 3.


In [ ]:
# Conexão e geometria: a fonte de vídeo é definida somente na seção de visão.
import json, html
import ipywidgets as widgets
from IPython.display import display
from serial.tools import list_ports
config_path=ROOT/'pc_operador'/'config.json'
cfg=AppConfig(config_path)
results={}
FORM_SAVED=config_path.is_file()
controller=widgets.Dropdown(options=[('Xbox principal',0)],value=int(cfg.data.get('xbox_index',0)),
                           layout=widgets.Layout(display='none'))
conn_pick=widgets.Dropdown(description='Setups:',options=[('Default','')],layout=widgets.Layout(width='520px'))
conn_label=widgets.HTML('<b>Nome automático:</b> calculado ao salvar.')
transport=widgets.Dropdown(options=[('USB','usb'),('Wi-Fi local do ESP32','wifi_local')],
                            value=cfg.data.get('esp',{}).get('transport') if cfg.data.get('esp',{}).get('transport') in ('usb','wifi_local') else 'wifi_local',
                            description='Comunicação:',layout=widgets.Layout(width='440px'))
port_pick=widgets.Dropdown(description='Porta ESP32:',options=[('Detectar','auto')],
                           layout=widgets.Layout(width='440px'))
conn_host=widgets.HTML('Rede criada pelo próprio ESP32: <b>192.168.4.1:8765</b>. '
                       'Compile e grave as credenciais na seção 7.')
width=widgets.FloatText(value=float(cfg.data.get('geometry',{}).get('track_width_m') or 0),
                        description='Entre esteiras (m):',style={'description_width':'155px'})
radius=widgets.FloatText(value=float(cfg.data.get('geometry',{}).get('sprocket_radius_m') or 0),
                         description='Raio motriz (m):',style={'description_width':'155px'})
rpm=widgets.FloatText(value=float(cfg.data.get('geometry',{}).get('output_max_rpm') or 0),
                      description='RPM saída:',style={'description_width':'155px'})
conn_status=widgets.HTML()

def refresh_coms(_=None):
    found=list(list_ports.comports())
    current=str(cfg.data.get('esp',{}).get('serial_port','auto'))
    options=[('Detectar automaticamente','auto')]+[(f'{p.device} · {p.description}',p.device) for p in found]
    port_pick.options=options
    port_pick.value=current if current in [v for _,v in options] else 'auto'
    conn_status.value=status_box('Portas USB','OK' if found else 'PENDENTE',
        'Escolha a porta do conversor USB serial da placa.' if found else 'Nenhuma COM detectada; conecte o ESP32 e atualize.').value

def show_transport(_=None):
    usb=transport.value == 'usb'
    port_pick.layout.display='' if usb else 'none'
    conn_host.layout.display='' if transport.value == 'wifi_local' else 'none'
transport.observe(show_transport,names='value')

def refresh_conn(selected=''):
    names=setup_names('connection')
    conn_pick.options=[('Default','')]+[(x,x) for x in names]
    conn_pick.value=selected if selected in names else ''

def load_conn(_):
    try:
        if not conn_pick.value: raise ValueError('Selecione um setup salvo; Default usa os campos atuais.')
        data=setup_get('connection',conn_pick.value)
        esp=data.get('esp',{}); geometry=data.get('geometry',{})
        transport.value=esp.get('transport','wifi_local') if esp.get('transport') in ('usb','wifi_local') else 'wifi_local'
        port=esp.get('serial_port','auto')
        port_pick.value=port if port in [v for _,v in port_pick.options] else 'auto'
        width.value=float(geometry.get('track_width_m') or 0)
        radius.value=float(geometry.get('sprocket_radius_m') or 0)
        rpm.value=float(geometry.get('output_max_rpm') or 0)
        show_transport()
        conn_status.value=status_box('Setup','OK','Carregado para edição.').value
    except Exception as exc:conn_status.value=status_box('Setup','FALHA',str(exc)).value

def save_conn(_):
    global cfg,FORM_SAVED
    try:
        if min(width.value,radius.value,rpm.value)<0:raise ValueError('Medidas e RPM não podem ser negativos.')
        esp={'transport':transport.value,'serial_port':port_pick.value,
             'baudrate':460800,'host':'192.168.4.1','port':8765,'reconnect_seconds':1.5}
        payload={'esp':esp,'geometry':{'track_width_m':width.value,
                 'sprocket_radius_m':radius.value,'output_max_rpm':rpm.value}}
        name=setup_save('connection',auto_setup_name('connection',payload),payload)
        updated=dict(cfg.data)
        updated['esp']={**updated.get('esp',{}),**esp}
        updated['geometry']=payload['geometry']
        _json_atomic(config_path,updated)
        cfg=AppConfig(config_path);FORM_SAVED=True
        refresh_conn(name)
        conn_status.value=status_box('Conexão','OK',f'{name} salvo. Escolha vídeo e modelos na seção 3.').value
    except Exception as exc:conn_status.value=status_box('Conexão','FALHA',str(exc)).value

def delete_conn(_):
    try:
        if not conn_pick.value:raise ValueError('Default não é um setup salvo.')
        setup_delete('connection',conn_pick.value)
        refresh_conn()
        conn_status.value=status_box('Setup','OK','Excluído.').value
    except Exception as exc:conn_status.value=status_box('Setup','FALHA',str(exc)).value
b_refresh=widgets.Button(description='Atualizar COM');b_refresh.on_click(refresh_coms)
b_load=widgets.Button(description='Carregar para editar');b_load.on_click(load_conn)
b_save=widgets.Button(description='Salvar setup',button_style='success');b_save.on_click(save_conn)
b_delete=widgets.Button(description='Excluir salvo',button_style='danger');b_delete.on_click(delete_conn)
refresh_conn();refresh_coms();show_transport()
display(widgets.VBox([widgets.HBox([conn_pick,conn_label]),widgets.HBox([b_load,b_save,b_delete]),
    widgets.HBox([transport,b_refresh]),port_pick,conn_host,
    widgets.HBox([width,radius,rpm]),conn_status]))


## 3. Visão RGB e escolha do vídeo

Teste aqui a câmera USB ou o arquivo de vídeo. Selecione os modelos de fogo e pessoas (obrigatórios), ajuste a visão e salve. O nome resulta do modo, da fonte e dos modelos. A visão em simulação aceita o cadastro de componentes físicos futuros, mas não aciona saídas do ESP32.


In [ ]:
# SEÇÃO 3 — substitua a célula inteira por este código.
import cv2, numpy as np, ipywidgets as widgets
import html, json, math, os, shutil, subprocess, sys, threading, time
from datetime import datetime as _vision_datetime
from pathlib import Path
from IPython.display import display

PC_DIR = ROOT / 'pc_operador'
DATA_DIR, MODELS_DIR, RUNS_DIR = (PC_DIR / name for name in ('data', 'models', 'runs'))
VIDEO_DIR = PC_DIR / 'videos_input'
LEGACY_DIR = PC_DIR / 'parameters_setups' / 'user'
VISION_FILE = LEGACY_DIR / 'rgb_monocular_fire_monitor_setups.json'
EXPORT_DIR = ROOT / 'output' / 'visao'
for folder in (DATA_DIR, MODELS_DIR, RUNS_DIR, VIDEO_DIR, LEGACY_DIR, EXPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# São exemplos de estrutura, não datasets anotados. Nenhum treinamento começa automaticamente.
for kind, class_name in (('fire', 'fire'), ('person', 'person'), ('firefighter', 'firefighter')):
    base = DATA_DIR / kind
    for split in ('train', 'val'):
        for sub in ('images', 'labels'):
            (base / sub / split).mkdir(parents=True, exist_ok=True)
    template = base / 'dataset.yaml'
    if not template.exists():
        template.write_text(
            'path: .\ntrain: images/train\nval: images/val\nnames:\n  0: ' + class_name +
            ('\n  1: helmet\n  2: protective_clothing' if kind == 'firefighter' else '') + '\n',
            encoding='utf-8')

def note(control, message):
    return widgets.HBox([control, widgets.HTML(html.escape(message),
                         layout=widgets.Layout(width='560px'))])

def status(label, state, message):
    return status_box(label, state, message).value

def within_package(path):
    try:
        return str(Path(path).resolve().relative_to(PC_DIR.resolve()))
    except ValueError:
        return str(Path(path).resolve())

def camera_value(value):
    value = str(value).strip()
    return int(value) if value.isdecimal() else value

def choices(paths, label):
    return [(within_package(path), str(path.resolve())) for path in sorted(set(paths))]

def model_paths():
    return sorted(MODELS_DIR.glob('*.pt'))

def data_paths():
    return sorted(set(DATA_DIR.rglob('*.yaml')) | set(DATA_DIR.rglob('*.yml')))

def read_legacy():
    try:
        result = json.loads(VISION_FILE.read_text(encoding='utf-8'))
        return result if isinstance(result, dict) else {}
    except (OSError, ValueError):
        return {}

def refresh_dropdown(dropdown, values, preferred=None):
    old = preferred if preferred is not None else dropdown.value
    dropdown.options = [('Selecione...', '')] + values
    allowed = [value for _, value in dropdown.options]
    dropdown.value = old if old in allowed else ''

def resolved_path(dropdown, manual=''):
    value = manual.strip() if manual else dropdown.value
    if not value:
        return None
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = PC_DIR / path
    if not path.is_file():
        basename = str(value).replace('\\', '/').split('/')[-1]
        for folder in (VIDEO_DIR, MODELS_DIR):
            candidate = folder / basename
            if candidate.is_file():
                return candidate.resolve()
    return path.resolve()

def open_capture(source, stop, mode='camera', requested_fps=None):
    backends = ([('DirectShow', cv2.CAP_DSHOW), ('Media Foundation', cv2.CAP_MSMF),
                 ('automático', cv2.CAP_ANY)] if sys.platform == 'win32' and
                mode == 'camera' and isinstance(source, int) else [('automático', cv2.CAP_ANY)])
    errors = []
    for backend_name, backend in backends:
        if stop.is_set():
            break
        cap = None
        try:
            cap = (cv2.VideoCapture(source, backend, _blaze_stop=stop)
                   if isinstance(source,str) and source.lower().startswith(('rtsp://','rtsps://'))
                   else cv2.VideoCapture(source, backend))
            if cap.isOpened():
                if requested_fps and mode == 'camera':
                    cap.set(cv2.CAP_PROP_FPS, float(requested_fps))
                deadline = time.monotonic() + 3
                while time.monotonic() < deadline and not stop.is_set():
                    ok, sample = cap.read()
                    if ok and sample is not None and sample.size:
                        return cap, sample, backend_name, errors
            detail=getattr(cap,'error','') if cap is not None else ''
            errors.append(f'{backend_name}: {detail or "não forneceu quadros"}')
        except Exception as exc:
            errors.append(f'{backend_name}: {type(exc).__name__}: {exc}')
        finally:
            if cap is not None and (not cap.isOpened() or errors and
                                    errors[-1].startswith(backend_name + ':')):
                cap.release()
    return None, None, None, errors

def dataset_report(path, kind):
    if path is None or not path.is_file():
        raise ValueError(f'Selecione o dataset YAML de {kind}.')
    import yaml
    specification = yaml.safe_load(path.read_text(encoding='utf-8'))
    if not isinstance(specification, dict):
        raise ValueError('YAML inválido.')
    names = specification.get('names', {})
    if isinstance(names, dict):
        class_names = [str(name).lower() for name in names.values()]
    elif isinstance(names, list):
        class_names = [str(name).lower() for name in names]
    else:
        raise ValueError('O YAML precisa declarar names.')
    terms = (('fire', 'flame', 'fogo', 'chama') if kind == 'fire' else
             ('person', 'pessoa', 'human') if kind == 'person' else
             ('firefighter', 'fireman', 'bombeiro', 'helmet', 'capacete', 'turnout', 'protective', 'ppe', 'epi'))
    if not any(any(word in name for word in terms) for name in class_names):
        raise ValueError(f'O dataset {kind} precisa de uma classe com nome {terms}.')
    root = Path(specification.get('path') or path.parent)
    if not root.is_absolute():
        root = (path.parent / root).resolve()
    counts = {}
    for split in ('train', 'val'):
        location = specification.get(split)
        if not location or isinstance(location, list):
            raise ValueError(f'Informe um diretório {split}: images/{split} no YAML.')
        image_dir = Path(location)
        if not image_dir.is_absolute():
            image_dir = root / image_dir
        image_dir = image_dir.resolve()
        images = [p for p in image_dir.rglob('*') if p.suffix.lower() in
                  ('.jpg', '.jpeg', '.png', '.bmp', '.webp')]
        label_dir = Path(str(image_dir).replace(os.sep + 'images' + os.sep,
                                                 os.sep + 'labels' + os.sep))
        labels = [p for p in label_dir.rglob('*.txt')] if label_dir.is_dir() else []
        if not images or not labels:
            raise ValueError(f'{split}: faltam imagens ou arquivos YOLO .txt em {image_dir} / {label_dir}.')
        counts[split] = (len(images), len(labels))
    return counts

def load_model(path, kind):
    if path is None or not path.is_file():
        raise ValueError(f'Selecione um modelo .pt já treinado para {kind}.')
    from ultralytics import YOLO
    model = YOLO(str(path))
    names = getattr(model, 'names', {})
    names = list(names.values()) if isinstance(names, dict) else list(names)
    names = [str(name).lower() for name in names]
    terms = (('fire', 'flame', 'fogo', 'chama') if kind == 'fire' else
             ('person', 'pessoa', 'human') if kind == 'person' else
             ('firefighter', 'fireman', 'bombeiro', 'helmet', 'capacete', 'turnout', 'protective', 'ppe', 'epi'))
    if not any(any(term in name for term in terms) for name in names):
        raise ValueError(f'{path.name}: classes {names} sem classe {kind}.')
    return model

def detections(model, image, confidence, image_size, words):
    names = model.names
    names = dict(enumerate(names)) if isinstance(names, list) else names
    ids = [int(key) for key, name in names.items() if
           any(word in str(name).lower() for word in words)]
    result = model.predict(image, conf=float(confidence), imgsz=int(image_size),
                           classes=ids, max_det=100, verbose=False)[0]
    output = []
    if result.boxes is not None:
        for box, score in zip(result.boxes.xyxy.cpu().numpy(),
                              result.boxes.conf.cpu().numpy()):
            x1, y1, x2, y2 = map(int, box)
            if x2 > x1 and y2 > y1:
                output.append({'box': (x1, y1, x2, y2), 'conf': float(score),
                               'center': ((x1+x2)/2, (y1+y2)/2),
                               'area': (x2-x1)*(y2-y1)})
    return output

def merge_overlapping_boxes(boxes, iou_threshold):
    """Agrupa caixas da mesma classe por sobreposição, com união transitiva."""
    pending=sorted(boxes,key=lambda item:item['conf'],reverse=True)
    merged=[]
    while pending:
        group=[pending.pop(0)]
        changed=True
        while changed:
            changed=False
            remaining=[]
            for candidate in pending:
                if any(box_iou(candidate['box'],other['box']) >= iou_threshold for other in group):
                    group.append(candidate)
                    changed=True
                else:remaining.append(candidate)
            pending=remaining
        x1=min(item['box'][0] for item in group)
        y1=min(item['box'][1] for item in group)
        x2=max(item['box'][2] for item in group)
        y2=max(item['box'][3] for item in group)
        merged.append({'box':(x1,y1,x2,y2),'conf':max(item['conf'] for item in group),
                       'center':((x1+x2)/2,(y1+y2)/2),'area':(x2-x1)*(y2-y1)})
    return merged

def box_iou(a,b):
    overlap=max(0,min(a[2],b[2])-max(a[0],b[0]))*max(0,min(a[3],b[3])-max(a[1],b[1]))
    area_a=max(0,a[2]-a[0])*max(0,a[3]-a[1])
    area_b=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    return overlap/max(1,area_a+area_b-overlap)

from runtime import pixel_to_servos as runtime_pixel_to_servos

def pixel_to_servos(point, shape, calibration, params):
    # A janela de teste não aciona servos; a calibração física por 12 pontos
    # pertence ao runtime e usa a posição comandada da plataforma.
    return runtime_pixel_to_servos(point, shape, calibration, params, 90.)

class VisionSession:
    def __init__(self, fire_model, person_model, params, calibration, mode, source,
                 firefighter_model=None):
        self.fire_model, self.person_model = fire_model, person_model
        self.firefighter_model = firefighter_model
        self.params, self.calibration, self.mode, self.source = params, calibration, mode, source
        self.journal_path = EXPORT_DIR / f'trajetoria_{_vision_datetime.now():%Y%m%d_%H%M%S}.jsonl'
        self.journal = self.journal_path.open('w', encoding='utf-8')
        self.reset()

    def close(self):
        self.journal.close()

    def reset(self):
        self.fires, self.people, self.firefighters = [], [], []
        self.regions, self.route = [], []
        self.impact, self.last_step = None, None
        self.last_person, self.last_detection = -1e10, -1e10
        self.trail, self.visited_cells = [], set()
        self.target = None

    def _painted_frame(self, frame, now):
        canvas=frame.copy()
        if self.mode != 'simulation':return canvas
        lifetime=max(.1,float(self.params['restore_seconds']))
        radius=int(self.params['erase_radius_px'])
        self.trail=[point for point in self.trail if now-point[0]<lifetime]
        for instant,px,py in self.trail:
            alpha=max(0.,min(1.,1.-(now-instant)/lifetime))
            cx,cy=int(round(px)),int(round(py))
            x0,x1=max(0,cx-radius),min(canvas.shape[1],cx+radius+1)
            y0,y1=max(0,cy-radius),min(canvas.shape[0],cy+radius+1)
            if x0>=x1 or y0>=y1:continue
            roi=canvas[y0:y1,x0:x1]
            mask=np.zeros(roi.shape[:2],dtype=np.uint8)
            cv2.circle(mask,(cx-x0,cy-y0),radius,255,-1)
            roi[mask>0]=(roi[mask>0].astype(np.float32)*(1.-alpha)).astype(np.uint8)
        return canvas

    def _fire_mask(self, frame, raw, cell):
        """Refina as caixas YOLO com pixels quentes, mantendo fallback para chama atípica."""
        height,width=frame.shape[:2]
        mask=np.zeros((height,width),dtype=np.uint8)
        for fire in raw:
            x1,y1,x2,y2=fire['box']
            x1,y1=max(0,x1),max(0,y1)
            x2,y2=min(width,x2),min(height,y2)
            if x2<=x1 or y2<=y1:continue
            roi=frame[y1:y2,x1:x2]
            hsv=cv2.cvtColor(roi,cv2.COLOR_BGR2HSV)
            hue,sat,val=cv2.split(hsv)
            warm=(((hue<=42)|(hue>=170)) & (sat>=65) & (val>=115)).astype(np.uint8)*255
            # O branco muito luminoso só é aceito perto de pixels quentes.
            bright=((val>=205)&(sat<=95)).astype(np.uint8)*255
            nearby=cv2.dilate(warm,np.ones((max(3,cell//4),)*2,np.uint8))
            support=cv2.bitwise_or(warm,cv2.bitwise_and(bright,nearby))
            coverage=cv2.countNonZero(support)
            if coverage>=max(12,int(.008*roi.shape[0]*roi.shape[1])):
                kernel=np.ones((max(3,cell//6),)*2,np.uint8)
                support=cv2.morphologyEx(support,cv2.MORPH_CLOSE,kernel)
                support=cv2.dilate(support,np.ones((max(3,cell//5),)*2,np.uint8))
                mask[y1:y2,x1:x2]=cv2.bitwise_or(mask[y1:y2,x1:x2],support)
            else:
                # Mesmo no fallback, pixels ainda escuros pelo jato ficam de fora.
                visible=(np.max(roi,axis=2)>35).astype(np.uint8)*255
                mask[y1:y2,x1:x2]=cv2.bitwise_or(mask[y1:y2,x1:x2],visible)
        return mask

    def _small_boxes_and_regions(self, frame, raw):
        height,width=frame.shape[:2]
        cell=max(4,int(math.floor(math.sqrt(float(self.params['max_fire_box_area_px'])))))
        mask=self._fire_mask(frame,raw,cell)
        boxes=[]
        coverage=np.zeros((height,width),dtype=np.uint8)
        # Cada caixa fica contida em uma célula fixa: nunca há sobreposição.
        for y0 in range(0,height,cell):
            for x0 in range(0,width,cell):
                cropped=mask[y0:min(height,y0+cell),x0:min(width,x0+cell)]
                if not cv2.countNonZero(cropped):continue
                ys,xs=np.nonzero(cropped)
                x1,x2=x0+int(xs.min()),x0+int(xs.max())+1
                y1,y2=y0+int(ys.min()),y0+int(ys.max())+1
                if (x2-x1)*(y2-y1)>self.params['max_fire_box_area_px']:
                    # O ajuste abaixo também cobre limites de área não quadrados.
                    x2=min(x2,x1+cell)
                    y2=min(y2,y1+max(1,int(self.params['max_fire_box_area_px']//(x2-x1))))
                overlaps=[f['conf'] for f in raw if box_iou((x1,y1,x2,y2),f['box'])>0]
                conf=max(overlaps,default=max((f['conf'] for f in raw),default=0.))
                index=len(boxes)
                boxes.append({'box':(x1,y1,x2,y2),'conf':conf,
                              'center':((x1+x2)/2.,(y1+y2)/2.),
                              'area':(x2-x1)*(y2-y1),
                              'cell':(x0//cell,y0//cell),'region':None})
                coverage[y1:y2,x1:x2]=255
        if not boxes:return [],[]
        count,labels,stats,_=cv2.connectedComponentsWithStats(coverage,connectivity=8)
        regions=[]
        for component in range(1,count):
            if stats[component,cv2.CC_STAT_AREA]<2:continue
            ox,oy,ow,oh=map(int,stats[component,:4])
            part=np.where(labels[oy:oy+oh,ox:ox+ow]==component,255,0).astype(np.uint8)
            contours,_=cv2.findContours(part,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
            if not contours:continue
            contour=max(contours,key=cv2.contourArea).copy()
            contour[:,0,0]+=ox
            contour[:,0,1]+=oy
            reduced=cv2.approxPolyDP(contour,max(1.,cv2.arcLength(contour,True)*.002),True)
            regions.append({'contour':contour,'export':reduced.reshape(-1,2).tolist(),
                            'component':component,'box_indices':[]})
        component_to_region={region['component']:i for i,region in enumerate(regions)}
        for i,box in enumerate(boxes):
            cx=min(width-1,int(box['center'][0]))
            cy=min(height-1,int(box['center'][1]))
            component=int(labels[cy,cx])
            region_index=component_to_region.get(component)
            if region_index is not None:
                box['region']=region_index
                regions[region_index]['box_indices'].append(i)
        return boxes,regions

    def _detect(self, painted, now, force=False):
        p=self.params
        fps=max(1.,float(p.get('input_fps',25)))
        if not force and now-self.last_detection<max(.02,p['detect_every_n_frames']/fps):
            return
        self.people=merge_overlapping_boxes(
            detections(self.person_model,painted,p['person_conf'],p['imgsz_infer'],
                       ('person','pessoa','human')),p['merge_iou'])
        self.firefighters=(merge_overlapping_boxes(
            detections(self.firefighter_model,painted,p['firefighter_conf'],p['imgsz_infer'],
                       ('firefighter','fireman','bombeiro','helmet','capacete','turnout',
                        'protective','ppe','epi')),p['merge_iou'])
            if self.firefighter_model is not None else [])
        raw=detections(self.fire_model,painted,p['fire_conf'],p['imgsz_infer'],
                       ('fire','flame','fogo','chama'))
        self.fires,self.regions=self._small_boxes_and_regions(painted,raw)
        self.last_detection=time.monotonic()
        if self.people or self.firefighters:self.last_person=self.last_detection
        if not self.fires:self.visited_cells.clear()

    def process(self, frame, engaged=True):
        now=time.monotonic()
        p=self.params
        height,width=frame.shape[:2]
        painted=self._painted_frame(frame,now)
        # Em simulação a inferência recebe o vídeo já pintado, sem desenhos de interface.
        self._detect(painted,now)
        blocked=now-self.last_person<=p['person_memory_seconds']
        if self.impact is None:self.impact=(width/2.,height/2.)
        self.route=[tuple(box['center']) for box in self.fires]
        self.target=None
        if engaged and not blocked and self.fires:
            candidates=[box for box in self.fires if box['cell'] not in self.visited_cells]
            if not candidates:
                self.visited_cells.clear()
                candidates=self.fires
            self.target=min(candidates,key=lambda box:math.dist(self.impact,box['center']))
        if blocked or not self.fires:
            self.impact=None
            self.last_step=None
            if blocked:self.trail=[]
        fps=max(1.,float(p.get('input_fps',25)))
        dt=min(.25,max(1./fps,now-(self.last_step if self.last_step is not None else now)))
        self.last_step=now if self.target is not None else None
        arrived=False
        if self.target is not None:
            waypoint=self.target['center']
            distance=math.dist(self.impact,waypoint)
            step=float(p['route_speed_px_frame'])*fps*dt
            if distance<=step:
                self.impact=tuple(map(float,waypoint))
                arrived=True
                self.visited_cells.add(self.target['cell'])
                # A próxima imagem exige inferência nova antes de escolher outro centro.
                self.last_detection=-1e10
            elif distance:
                self.impact=(self.impact[0]+(waypoint[0]-self.impact[0])*step/distance,
                             self.impact[1]+(waypoint[1]-self.impact[1])*step/distance)
        active_region=None
        active_region_index=None
        if self.impact is not None and not blocked:
            for index,region in enumerate(self.regions):
                if cv2.pointPolygonTest(region['contour'],self.impact,False)>=0:
                    active_region=region
                    active_region_index=index
                    break
        enabled=active_region is not None and engaged
        if enabled and self.mode=='simulation':
            self.trail.append((now,*self.impact))
            self.trail=self.trail[-max(100,int(p['restore_seconds']*fps*2)):]
            painted=self._painted_frame(frame,now)
        aim=(self.impact[0],max(0.,self.impact[1]-p['jet_drop_px'])) if self.target else None
        pan,tilt=pixel_to_servos(aim,frame.shape,self.calibration,p) if aim else (None,None)
        record={'t_unix':time.time(),'mode':self.mode,'camera':str(self.source),
                'frame_width':width,'frame_height':height,'strategy':'centroides_em_regioes',
                'route_index':self.fires.index(self.target) if self.target else None,
                'route_pixels':[list(point) for point in self.route],
                'small_boxes':[list(box['box']) for box in self.fires],
                'regions':[region['export'] for region in self.regions],
                'impact_px':list(self.impact) if self.impact else None,
                'aim_px':list(aim) if aim else None,'pan_deg':pan,'tilt_deg':tilt,
                'target_valid':bool(self.target),'centroid_reached':bool(arrived),
                'person_detected':bool(self.people),'firefighter_detected':bool(self.firefighters),
                'blocked':bool(blocked),'jet_requested':bool(self.target and not blocked),
                'jet_permitted':bool(enabled),'region_index':active_region_index}
        if self.mode=='real' or enabled or blocked:
            self.journal.write(json.dumps(record,ensure_ascii=False)+'\n')
            self.journal.flush()
        canvas=painted
        label_scale=max(.45,min(.9,width/1500.))
        label_stroke=max(1,int(round(label_scale*2)))
        for box in self.fires:
            bx1,by1,bx2,by2=box['box']
            cv2.rectangle(canvas,(bx1,by1),(bx2,by2),(0,150,255),1)
            cv2.circle(canvas,tuple(map(int,box['center'])),3,(0,200,255),-1)
        for region in self.regions:
            cv2.polylines(canvas,[region['contour']],True,(220,40,220),2,cv2.LINE_AA)
            indices=region['box_indices']
            if indices:
                mean_conf=sum(self.fires[index]['conf'] for index in indices)/len(indices)
                label=f'FOGO {mean_conf:.0%}'
                points=region['contour'].reshape(-1,2)
                x=int(points[:,0].min())
                y=max(16,int(points[:,1].min())-4)
                small_scale=max(.35,min(.6,width/2200.))
                cv2.putText(canvas,label,(x,y),cv2.FONT_HERSHEY_SIMPLEX,
                            small_scale,(0,0,0),3)
                cv2.putText(canvas,label,(x,y),cv2.FONT_HERSHEY_SIMPLEX,
                            small_scale,(220,40,220),1)
        for kind,items,color in (('PESSOA',self.people,(0,0,255)),
                                 ('BOMBEIRO/EPI',self.firefighters,(255,200,0))):
            for found in items:
                x1,y1,x2,y2=found['box']
                margin=int(p['human_safety_radius_px'])
                cv2.rectangle(canvas,(max(0,x1-margin),max(0,y1-margin)),
                              (min(width-1,x2+margin),min(height-1,y2+margin)),color,1)
                cv2.rectangle(canvas,(x1,y1),(x2,y2),color,2)
                label=f'{kind} {found["conf"]:.0%}'
                y=max(18,y1-3)
                cv2.putText(canvas,label,(x1,y),cv2.FONT_HERSHEY_SIMPLEX,
                            label_scale,(0,0,0),label_stroke+2)
                cv2.putText(canvas,label,(x1,y),cv2.FONT_HERSHEY_SIMPLEX,
                            label_scale,color,label_stroke)
        if blocked:
            cv2.rectangle(canvas,(0,0),(width,max(55,int(width*.085))),(0,0,170),-1)
            cv2.putText(canvas,'PESSOA/BOMBEIRO: JATO BLOQUEADO',
                        (10,max(40,int(width*.06))),cv2.FONT_HERSHEY_SIMPLEX,
                        max(1.,width/640.),(255,255,255),3)
        if self.target is not None and not blocked and self.impact is not None:
            hit=(int(round(self.impact[0])),int(round(self.impact[1])))
            aim_pt=(int(round(aim[0])),int(round(aim[1])))
            cv2.circle(canvas,aim_pt,9,(0,200,0),-1)
            cv2.circle(canvas,hit,7,(0,255,255),2)
            if enabled:
                drop=float(p['jet_drop_px'])
                start=(width//2,height-1)
                path=[]
                for t in np.linspace(0.,1.,40):
                    x=start[0]+(hit[0]-start[0])*t
                    y=start[1]+(hit[1]-start[1]-drop)*t+drop*t*t
                    path.append((int(round(x)),int(round(y))))
                cv2.polylines(canvas,[np.array(path,dtype=np.int32)],
                              False,(255,110,0),max(5,width//120),cv2.LINE_AA)
                cv2.circle(canvas,aim_pt,9,(0,200,0),-1)
                cv2.circle(canvas,hit,9,(255,255,255),2)
        return canvas,record

# Encerra a sessão anterior ao reexecutar a célula.
if 'vision_stop' in globals(): vision_stop.set()
if 'vision_thread' in globals() and vision_thread is not None: vision_thread.join(timeout=5)
vision_stop = threading.Event()
vision_thread = None
vision_session = None
train_process = None
if 'camera_test_stop' in globals():camera_test_stop.set()
if 'camera_test_thread' in globals() and camera_test_thread is not None:
    camera_test_thread.join(timeout=4)
camera_test_stop=threading.Event()
camera_test_thread=None
vision_status = widgets.HTML('Configure as fontes e modelos antes de iniciar.')
training_status = widgets.HTML('Nenhum treinamento iniciado.')
setup_status = widgets.HTML('')
vision_display = widgets.Image(format='jpeg',layout=widgets.Layout(width='850px'))

mode_select = widgets.ToggleButtons(options=[('Simulação','simulation'),
                                             ('Combate real','real')],
                                     description='Modo:',style={'description_width':'initial'},
                                     layout=widgets.Layout(width='620px'))
saved_camera=cfg.camera_source()
initial_input='local' if isinstance(saved_camera,int) else 'video'
import runtime as network_runtime
input_mode = widgets.Dropdown(options=[('Câmera USB/local','local'),
                                       ('Arquivo de vídeo','video')],value=initial_input,
                              description='Entrada:',layout=widgets.Layout(width='390px'))
local_indices=sorted(set(range(6)) | ({saved_camera} if isinstance(saved_camera,int) else set()))
camera_list = widgets.Dropdown(options=[(f'Câmera local {i}',i) for i in local_indices],
                               value=saved_camera if isinstance(saved_camera,int) else 0,
                               description='Índice:',layout=widgets.Layout(width='340px'))
video_list = widgets.Dropdown(description='Vídeo:',layout=widgets.Layout(width='580px'))
input_fps=widgets.BoundedIntText(value=25,min=1,max=120,
    description='FPS desejado:',style={'description_width':'125px'},
    layout=widgets.Layout(width='240px'))
vision_playback_state={'seek':None,'internal':False}
vision_playback_lock=threading.Lock()
video_timeline=widgets.FloatSlider(value=0,min=0,max=1,step=.1,
    description='Tempo:',continuous_update=False,readout=False,
    layout=widgets.Layout(width='520px'))
video_timeline.disabled=True
video_time_label=widgets.HTML('00:00 / 00:00')
video_back=widgets.Button(description='Voltar 5 s',layout=widgets.Layout(width='110px'))
video_forward=widgets.Button(description='Avançar 5 s',layout=widgets.Layout(width='120px'))
video_pause=widgets.ToggleButton(value=False,description='Pausar vídeo',
    button_style='warning',layout=widgets.Layout(width='140px'))
video_controls=widgets.VBox([widgets.HBox([video_back,video_pause,video_forward]),
                              widgets.HBox([video_timeline,video_time_label])])
test_local_button=widgets.Button(description='Testar câmera',button_style='info',
                                 layout=widgets.Layout(width='155px'))
stop_camera_test_button=widgets.Button(description='Parar teste',button_style='warning',
                                       layout=widgets.Layout(width='130px'))
camera_test_image=widgets.Image(format='jpeg',layout=widgets.Layout(width='320px',height='240px'))
camera_test_status=widgets.HTML('Prévia parada.')
local_input_panel=widgets.HBox([camera_list,test_local_button])
video_input_panel=widgets.HBox([video_list])
camera_input_controls=widgets.VBox([local_input_panel,video_input_panel,
                                    note(input_fps,'No arquivo, define a velocidade de reprodução; na câmera, solicita esta taxa à captura. Padrão: 25 FPS.'),
                                    stop_camera_test_button,camera_test_status])
camera_test_preview=widgets.VBox([camera_test_image],layout=widgets.Layout(width='330px'))

def update_input_panel(_=None):
    local_input_panel.layout.display='' if input_mode.value=='local' else 'none'
    video_input_panel.layout.display='' if input_mode.value=='video' else 'none'
    video_controls.layout.display='' if input_mode.value=='video' else 'none'
    camera_test_preview.layout.display='' if input_mode.value!='video' else 'none'
    stop_camera_test_button.layout.display='' if input_mode.value!='video' else 'none'
    camera_test_status.layout.display='' if input_mode.value!='video' else 'none'
    if input_mode.value=='video':camera_test_stop.set()
    if 'update_calibration_panel' in globals():update_calibration_panel()

input_mode.observe(update_input_panel,names='value')
update_input_panel()

def format_video_time(seconds):
    seconds=max(0,int(seconds))
    return f'{seconds//3600:02d}:{seconds//60%60:02d}:{seconds%60:02d}' if seconds>=3600 else f'{seconds//60:02d}:{seconds%60:02d}'

def request_video_seek(seconds):
    if video_timeline.disabled:return
    with vision_playback_lock:vision_playback_state['seek']=max(0,min(float(video_timeline.max),float(seconds)))

def on_video_timeline(change):
    if change['name']=='value' and not vision_playback_state['internal']:
        request_video_seek(change['new'])

def on_video_pause(change):
    video_pause.description='Retomar vídeo' if change['new'] else 'Pausar vídeo'
    video_pause.button_style='success' if change['new'] else 'warning'

video_timeline.observe(on_video_timeline,names='value')
video_pause.observe(on_video_pause,names='value')
video_back.on_click(lambda _:request_video_seek(video_timeline.value-5))
video_forward.on_click(lambda _:request_video_seek(video_timeline.value+5))
fire_list = widgets.Dropdown(description='Modelo fogo:',layout=widgets.Layout(width='680px'))
person_list = widgets.Dropdown(description='Modelo pessoa:',layout=widgets.Layout(width='680px'))
firefighter_list=widgets.Dropdown(description='Bombeiro/EPI:',layout=widgets.Layout(width='680px'))
fire_strategy = widgets.RadioButtons(options=[('Usar modelo já treinado','existing'),
                                              ('Treinar novo modelo com dataset','train')],
                                     value='existing',description='Fogo:',
                                     style={'description_width':'80px'},
                                     layout=widgets.Layout(width='440px'))
person_strategy = widgets.RadioButtons(options=[('Usar modelo já treinado','existing'),
                                                ('Treinar novo modelo com dataset','train')],
                                       value='existing',description='Pessoas:',
                                       style={'description_width':'80px'},
                                       layout=widgets.Layout(width='440px'))
firefighter_strategy=widgets.RadioButtons(options=[('Sem modelo (opcional)','none'),
    ('Usar modelo já treinado','existing'),('Treinar novo modelo com dataset','train')],
    value='none',description='Bombeiro:',style={'description_width':'80px'},
    layout=widgets.Layout(width='460px'))
trained_models={'fire':None,'person':None,'firefighter':None}
fire_trained_info=widgets.HTML('')
person_trained_info=widgets.HTML('')
firefighter_trained_info=widgets.HTML('')
fire_dataset = widgets.Dropdown(description='Dataset fogo:',layout=widgets.Layout(width='680px'))
person_dataset = widgets.Dropdown(description='Dataset pessoa:',layout=widgets.Layout(width='680px'))
firefighter_dataset=widgets.Dropdown(description='Dataset bombeiro:',layout=widgets.Layout(width='680px'))
seed_list = widgets.Dropdown(description='Peso inicial:',layout=widgets.Layout(width='680px'))
rgb_entries = widgets.Dropdown(description='Setup visão RGB:',layout=widgets.Layout(width='680px'))
load_rgb_button=widgets.Button(description='Importar parâmetros RGB',
                              layout=widgets.Layout(width='220px'))
refresh_button = widgets.Button(description='Atualizar listas')
train_fire = widgets.Button(description='Treinar fogo',button_style='info')
train_person = widgets.Button(description='Treinar pessoas',button_style='info')
train_firefighter=widgets.Button(description='Treinar bombeiros/EPI',button_style='info')
stop_training = widgets.Button(description='Interromper treino',button_style='warning')
train_epochs = widgets.BoundedIntText(value=50,min=1,max=1000,description='Épocas:')
train_size = widgets.Dropdown(options=[320,640,960,1280],value=640,description='Imagem:')
train_batch = widgets.BoundedIntText(value=8,min=1,max=128,description='Lote:')
for button,width in ((refresh_button,'150px'),(train_fire,'160px'),
                     (train_person,'175px'),(train_firefighter,'195px'),(stop_training,'185px')):
    button.layout=widgets.Layout(width=width)
fire_existing_panel=widgets.VBox([note(fire_list,'Escolha um arquivo .pt com classe fire, flame, fogo ou chama.')])
person_existing_panel=widgets.VBox([note(person_list,'Escolha um arquivo .pt com classe person, pessoa ou human.')])
firefighter_existing_panel=widgets.VBox([note(firefighter_list,'Classes bombeiro/firefighter ou vestimentas como helmet, capacete, protective_clothing ou EPI.')])
fire_training_panel=widgets.VBox([
    note(fire_dataset,'Dataset YOLO com imagens e rótulos em train e val.'),
    widgets.HBox([train_fire,fire_trained_info])])
person_training_panel=widgets.VBox([
    note(person_dataset,'Dataset YOLO com imagens e rótulos em train e val.'),
    widgets.HBox([train_person,person_trained_info])])
firefighter_training_panel=widgets.VBox([
    note(firefighter_dataset,'Dataset YOLO anotado de bombeiros e/ou vestimentas, com imagens e rótulos train/val.'),
    widgets.HBox([train_firefighter,firefighter_trained_info])])
shared_training_panel=widgets.VBox([
    note(seed_list,'Peso inicial; um modelo COCO é ajustado ao dataset selecionado.'),
    widgets.HBox([train_epochs,train_size,train_batch]),
    stop_training,training_status])

def update_model_panels(_=None):
    fire_existing_panel.layout.display='' if fire_strategy.value=='existing' else 'none'
    fire_training_panel.layout.display='' if fire_strategy.value=='train' else 'none'
    person_existing_panel.layout.display='' if person_strategy.value=='existing' else 'none'
    person_training_panel.layout.display='' if person_strategy.value=='train' else 'none'
    firefighter_existing_panel.layout.display='' if firefighter_strategy.value=='existing' else 'none'
    firefighter_training_panel.layout.display='' if firefighter_strategy.value=='train' else 'none'
    shared_training_panel.layout.display=('' if 'train' in (fire_strategy.value,person_strategy.value,
                                                           firefighter_strategy.value)
                                          else 'none')

fire_strategy.observe(update_model_panels,names='value')
person_strategy.observe(update_model_panels,names='value')
firefighter_strategy.observe(update_model_panels,names='value')
update_model_panels()

param_definitions = [
 ('fire_conf','Conf. chama',.01,.95,.01,.30,'Confiança mínima da chama; 0,30 significa aceitar resultados a partir de 30%.'),
 ('person_conf','Conf. pessoa',.05,.95,.01,.70,'Confiança mínima de pessoa; uma detecção aceita impede o acionamento do jato.'),
 ('firefighter_conf','Conf. bombeiro',.05,.95,.01,.70,'Confiança mínima de bombeiro ou EPI; usada somente se o modelo opcional for selecionado.'),
 ('imgsz_infer','Tamanho YOLO',320,1280,32,640,'Resolução da inferência; 640 equilibra detalhes e velocidade.'),
 ('detect_every_n_frames','Ciclo YOLO',1,30,1,15,'Intervalo-alvo em quadros entre inferências; equivale a 0,6 s a 25 FPS, quando o processamento acompanha a captura.'),
 ('merge_iou','IoU fusão',.05,.95,.05,.45,'Sobreposição mínima para fundir caixas de pessoa/bombeiro; caixas de fogo são recortadas em células sem sobreposição.'),
 ('max_fire_box_area_px','Área máxima caixa',100,10000,100,8000,'Área máxima de cada caixa de fogo em pixels quadrados; chamas grandes são divididas em caixas pequenas, sem sobreposição.'),
 ('route_speed_px_frame','Vel. mira',1,120,1,30,'Velocidade máxima em pixels por quadro do FPS configurado para perseguir o centróide ainda não visitado mais próximo.'),
 ('jet_drop_px','Queda do jato',0,160,1,90,'Curvatura ilustrativa do jato que sai do centro inferior e distância vertical da mira verde acima do impacto.'),
 ('erase_radius_px','Raio virtual',2,120,1,80,'Raio da área preta em pixels quando o jato virtual está acionado, apenas na simulação.'),
 ('restore_seconds','Restituição (s)',.5,20.,.5,20.,'Tempo para a cauda preta desaparecer gradualmente e a imagem original voltar à inferência; padrão da simulação: 20 s.'),
 ('human_safety_radius_px','Dist. pessoa',1,300,1,90,'Margem visual em pixels ao redor da caixa de pessoa; qualquer pessoa detectada bloqueia o jato em toda a imagem.'),
 ('person_memory_seconds','Memória pessoa',.1,10.,.1,2.,'Mantém o bloqueio durante este tempo após a última detecção de pessoa, evitando religamento imediato.'),
]
params_widgets = {}; param_rows=[]
for key,label,low,high,step,default,help_text in param_definitions:
    control_type = widgets.FloatSlider if isinstance(default,float) else widgets.IntSlider
    saved = default
    control = control_type(value=max(low,min(high,saved)),min=low,max=high,step=step,
        description=label,style={'description_width':'140px'},layout=widgets.Layout(width='470px'))
    params_widgets[key]=control
    param_rows.append(note(control,help_text))

simulation_defaults={key:default for key,_,_,_,_,default,_ in param_definitions}
real_defaults={**simulation_defaults,'route_speed_px_frame':5}
mode_defaults={'simulation':simulation_defaults,'real':real_defaults}
mode_param_cache={}
active_param_mode={'value':mode_select.value}

def switch_param_mode(change):
    previous=active_param_mode['value']
    mode_param_cache[previous]={key:control.value for key,control in params_widgets.items()}
    selected=mode_param_cache.get(change['new'],mode_defaults[change['new']])
    for key,control in params_widgets.items():
        control.value=selected[key]
    active_param_mode['value']=change['new']

mode_select.observe(switch_param_mode,names='value')

calibration_panel=widgets.HTML(
    '<div style="padding:12px;border:1px solid #94a3b8;border-radius:7px">'
    '<b>Calibração física da plataforma e da mira</b><br>'
    'Após gravar o firmware de combate real e conectar a câmera e os três servos, '
    'abra o runtime e clique <b>Calibrar câmera e canhão</b>. Registre os quatro '
    'cantos da imagem nas posições 0°, 90° e 180°. Nenhum slider desta seção '
    'move servos físicos.</div>')
calibration_width,calibration_height=640,480

setup_name_label=widgets.HTML('<b>Nome automático:</b> modo, fonte e modelos.')
setup_pick=widgets.Dropdown(description='Salvos:')
setup_save_button=widgets.Button(description='Salvar/atualizar',button_style='success')
setup_load_button=widgets.Button(description='Carregar para editar',button_style='info')
setup_delete_button=widgets.Button(description='Excluir',button_style='danger')
vision_start=widgets.Button(description='Iniciar visão',button_style='success')
vision_stop_button=widgets.Button(description='Parar visão',button_style='warning')
vision_reset=widgets.Button(description='Reiniciar trajetória')
for button,width in ((setup_save_button,'165px'),(setup_load_button,'190px'),
                     (setup_delete_button,'100px'),(vision_start,'145px'),
                     (vision_stop_button,'135px'),
                     (vision_reset,'190px')):
    button.layout=widgets.Layout(width=width)

def setup_key(mode, label):
    return ('S-' if mode=='simulation' else 'R-')+label.strip()

def refresh_setup_options(preferred=None):
    names=[]
    for name in setup_names('vision'):
        try:
            mode=setup_get('vision',name)['vision'].get('combat_mode','simulation')
            if mode==mode_select.value:names.append(name)
        except (KeyError,ValueError,TypeError):continue
    refresh_dropdown(setup_pick,[(name,name) for name in names],preferred)

mode_select.observe(lambda change:refresh_setup_options(),names='value')

def refresh_lists(_=None, preferred=None):
    models=choices(model_paths(),'Modelo')
    fire_guess=preferred or next((str(p.resolve()) for p in model_paths() if 'fire' in p.stem.lower()),'')
    person_guess=next((str(p.resolve()) for p in model_paths() if
                       'person' in p.stem.lower() or p.stem.lower()=='yolo11n'),'')
    refresh_dropdown(fire_list,models,preferred or fire_list.value or fire_guess)
    refresh_dropdown(person_list,models,person_list.value or person_guess)
    fighter_guess=next((str(p.resolve()) for p in model_paths() if
        'firefighter' in p.stem.lower() or 'bombeiro' in p.stem.lower()),'')
    refresh_dropdown(firefighter_list,models,firefighter_list.value or fighter_guess)
    refresh_dropdown(seed_list,models+[('YOLO11n inicial (baixa no treino)','yolo11n.pt')],
                     seed_list.value or 'yolo11n.pt')
    datasets=choices(data_paths(),'Dataset')
    refresh_dropdown(fire_dataset,datasets,fire_dataset.value or str((DATA_DIR/'fire'/'dataset.yaml').resolve()))
    refresh_dropdown(person_dataset,datasets,person_dataset.value or str((DATA_DIR/'person'/'dataset.yaml').resolve()))
    refresh_dropdown(firefighter_dataset,datasets,firefighter_dataset.value or
                     str((DATA_DIR/'firefighter'/'dataset.yaml').resolve()))
    videos=[p for p in VIDEO_DIR.iterdir() if p.suffix.lower() in ('.mp4','.avi','.mov','.mkv')]
    refresh_dropdown(video_list,choices(videos,'Vídeo'))
    legacy=read_legacy()
    names=sorted(legacy,key=lambda name:str(legacy[name].get('saved_at','')),reverse=True)
    previous_rgb=rgb_entries.value or ''
    rgb_entries.options=[('Default','')]+[(name,name) for name in names]
    rgb_entries.value=previous_rgb if previous_rgb in names else ''
    refresh_setup_options()

def apply_legacy(name):
    entry=read_legacy().get(name,{})
    values=entry.get('config',entry)
    if isinstance(values,dict):
        for key,control in params_widgets.items():
            if key in values:
                try: control.value=values[key]
                except Exception: pass

def selected_input():
    if input_mode.value=='video':
        path=resolved_path(video_list)
        if path is None or not path.is_file():
            raise ValueError(f'Coloque um vídeo em {VIDEO_DIR} e selecione-o.')
        return str(path), 'video'
    if input_mode.value=='local':
        return int(camera_list.value),'camera'
    raise ValueError('Selecione uma câmera USB ou um vídeo salvo.')

def stop_setup_camera_preview():
    callback=globals().get('stop_camera_preview')
    if not callable(callback):return
    try:
        callback()
    except AttributeError as exc:
        # Uma execução anterior pode ter sobrescrito o widget desta seção.
        # O callback já interrompeu a captura antes de atualizar esse indicador.
        if not (isinstance(globals().get('video_state'),dict) and
                "'dict' object has no attribute 'value'" in str(exc)):
            raise

def stop_camera_test(_=None):
    camera_test_stop.set()
    camera_test_image.value=b''
    camera_test_status.value=status('Teste da câmera','PENDENTE','Encerrando prévia...')
    def finish(thread=camera_test_thread, stopped=camera_test_stop):
        if thread is not None and thread is not threading.current_thread():
            thread.join(timeout=7)
        if stopped is not camera_test_stop:
            return
        if thread is None or not thread.is_alive():
            camera_test_status.value=status('Teste da câmera','OK','Teste encerrado; câmera liberada.')
        else:
            camera_test_status.value=status('Teste da câmera','FALHA',
                'A captura ainda está fechando; tente novamente após alguns segundos.')
    threading.Thread(target=finish,daemon=True).start()

def test_selected_camera(_):
    global camera_test_stop,camera_test_thread
    camera_test_stop.set()
    if camera_test_thread is not None:
        camera_test_thread.join(timeout=4)
        if camera_test_thread.is_alive():
            camera_test_status.value=status('Teste da câmera','FALHA','A captura anterior ainda está encerrando.')
            return
    if 'vision_stop' in globals():
        vision_stop.set()
    if 'vision_thread' in globals() and vision_thread is not None:
        vision_thread.join(timeout=4)
        if vision_thread.is_alive():
            camera_test_status.value=status('Teste da câmera','FALHA','Pare a visão antes de testar outra câmera.')
            return
    stop_setup_camera_preview()
    try:
        source,kind=selected_input()
        if kind!='camera':raise ValueError('Selecione uma câmera USB.')
        camera_test_stop=threading.Event()
        this_stop=camera_test_stop
        camera_test_image.value=b''
        camera_test_status.value=status('Teste da câmera','PENDENTE',f'Abrindo {network_runtime.redacted_source(source)}...')
        def preview_loop():
            cap=None
            try:
                cap,first,backend,errors=open_capture(source,this_stop,'camera',input_fps.value)
                if this_stop.is_set():return
                if cap is None:
                    camera_test_status.value=status('Teste da câmera','FALHA',
                                                     f'{network_runtime.redacted_source(source)} não forneceu vídeo: {errors}')
                    return
                pending=first
                failures=0
                while not this_stop.is_set():
                    if pending is not None:
                        frame,pending=pending,None
                    else:
                        ok,frame=cap.read()
                        if not ok or frame is None or not frame.size:
                            failures+=1
                            if failures>=10:raise RuntimeError('O fluxo de vídeo parou.')
                            this_stop.wait(.1)
                            continue
                    failures=0
                    h,w=frame.shape[:2]
                    shown=cv2.resize(frame,(320,max(1,round(h*320/w))))
                    ok,jpg=cv2.imencode('.jpg',shown,[cv2.IMWRITE_JPEG_QUALITY,75])
                    if ok:camera_test_image.value=jpg.tobytes()
                    camera_test_status.value=status('Teste da câmera','OK',
                        f'{network_runtime.redacted_source(source)} · {backend} · {w} × {h} pixels')
                    this_stop.wait(.12)
            except Exception as exc:
                camera_test_status.value=status('Teste da câmera','FALHA',
                    f'{type(exc).__name__}: {exc}')
            finally:
                if cap is not None:cap.release()
                if this_stop.is_set():
                    camera_test_image.value=b''
                    camera_test_status.value=status('Teste da câmera','OK','Prévia encerrada; câmera liberada.')
        camera_test_thread=threading.Thread(target=preview_loop,daemon=True)
        camera_test_thread.start()
    except Exception as exc:
        camera_test_status.value=status('Teste da câmera','FALHA',str(exc))

def calibration_fields():
    # A calibração física é salva pelo runtime para uma combinação completa.
    return {'enabled':False}

def section_fields():
    source,source_type=selected_input()
    fire=(resolved_path(fire_list) if fire_strategy.value=='existing' else trained_models['fire'])
    person=(resolved_path(person_list) if person_strategy.value=='existing' else trained_models['person'])
    if fire is None or not Path(fire).is_file():
        raise ValueError('Fogo: selecione um modelo treinado ou conclua o treinamento.')
    if person is None or not Path(person).is_file():
        raise ValueError('Pessoas: selecione um modelo treinado ou conclua o treinamento.')
    firefighter=(None if firefighter_strategy.value=='none' else
                 resolved_path(firefighter_list) if firefighter_strategy.value=='existing' else
                 trained_models['firefighter'])
    if firefighter_strategy.value!='none' and (firefighter is None or not Path(firefighter).is_file()):
        raise ValueError('Bombeiro/EPI: selecione um modelo ou conclua o treinamento opcional.')
    return {**cfg.data['vision'], 'combat_mode':mode_select.value,
      'camera_source':source, 'source_type':input_mode.value,
      'input_fps':input_fps.value,
      'model_strategy':{'fire':fire_strategy.value,'person':person_strategy.value,
                        'firefighter':firefighter_strategy.value},
      'fire_model':within_package(fire) if fire else '',
      'person_model':within_package(person) if person else '',
      'firefighter_model':within_package(firefighter) if firefighter else '',
      'legacy_setup_file':within_package(VISION_FILE),
      'legacy_setup_name':rgb_entries.value or '',
      'params_override':{**{k:v.value for k,v in params_widgets.items()},'input_fps':input_fps.value},
      'calibration_override':calibration_fields(), 'setup_label':'Default',
      'trajectory_export_dir':str(EXPORT_DIR.resolve())}

def save_setup(_):
    global cfg
    try:
        section=section_fields()
        payload={'vision':section}
        name=setup_save('vision',auto_setup_name('vision',payload),payload)
        current=json.loads(config_path.read_text(encoding='utf-8')) if config_path.is_file() else {}
        current['vision']=section
        _json_atomic(config_path,current)
        cfg=AppConfig(config_path)
        refresh_setup_options(name)
        setup_status.value=status('Setup de visão','OK',f'{name} salvo. Selecione-o na combinação oficial ao final do notebook.')
    except Exception as exc:setup_status.value=status('Setup de visão','FALHA',str(exc))

def load_setup(_):
    try:
        if not setup_pick.value:raise ValueError('Selecione um setup salvo.')
        section=setup_get('vision',setup_pick.value)['vision']
        input_fps.value=int(section.get('input_fps',section.get('params_override',{}).get('input_fps',25)))
        mode_select.value=section.get('combat_mode','simulation')
        saved_kind=section.get('source_type','camera')
        source=section.get('camera_source',cfg.camera_source())
        if saved_kind=='video':
            input_mode.value='video'
            refresh_dropdown(video_list,choices([resolved_path(video_list,str(source))],'Vídeo'),str(resolved_path(video_list,str(source))))
        elif isinstance(source, str) and source.lower().startswith(('rtsp://','rtsps://','http://','https://')):
            raise ValueError('Setup antigo de câmera IP: selecione uma câmera USB ou um vídeo salvo.')
        else:
            input_mode.value='local'
            number=int(source)
            if number not in [v for _,v in camera_list.options]:
                camera_list.options=list(camera_list.options)+[(f'Câmera local {number}',number)]
            camera_list.value=number
        for control,key in ((fire_list,'fire_model'),(person_list,'person_model'),
                            (firefighter_list,'firefighter_model')):
            path=section.get(key,'')
            if path:
                resolved=resolved_path(control,path)
                opts=list(control.options)
                if str(resolved) not in [v for _,v in opts]: opts.append((within_package(resolved),str(resolved)))
                control.options=opts;control.value=str(resolved)
        strategy=section.get('model_strategy',{})
        for kind,selector,control in (('fire',fire_strategy,fire_list),
                                      ('person',person_strategy,person_list),
                                      ('firefighter',firefighter_strategy,firefighter_list)):
            if kind=='firefighter' and strategy.get(kind,'none')=='none':
                selector.value='none'
                continue
            saved_path=resolved_path(control)
            if strategy.get(kind)=='train' and saved_path is not None and saved_path.is_file():
                trained_models[kind]=saved_path
                label=(fire_trained_info if kind=='fire' else
                       person_trained_info if kind=='person' else firefighter_trained_info)
                label.value='Modelo salvo: '+html.escape(str(saved_path))
                selector.value='train'
            else:
                selector.value='existing'
        entry=section.get('legacy_setup_name','')
        if entry in [v for _,v in rgb_entries.options]:rgb_entries.value=entry
        for key,control in params_widgets.items():
            if key in section.get('params_override',{}):control.value=section['params_override'][key]
        setup_status.value=status('Setup de visão','OK','Carregado para edição; salve novamente para atualizar.')
    except Exception as exc:setup_status.value=status('Setup de visão','FALHA',str(exc))

def delete_setup(_):
    try:
        name=setup_pick.value
        setup_delete('vision',name)
        refresh_setup_options()
        setup_status.value=status('Setup de visão','OK',f'{name} excluído.')
    except Exception as exc:setup_status.value=status('Setup de visão','FALHA',str(exc))

TRAIN_SCRIPT = r'''
import json, sys
from pathlib import Path
from ultralytics import YOLO
request=json.loads(sys.argv[1])
model=YOLO(request['seed'])
model.train(data=request['dataset'],epochs=request['epochs'],imgsz=request['imgsz'],
            batch=request['batch'],workers=0,project=request['project'],
            name=request['name'],exist_ok=False,plots=True)
print('TREINO_CONCLUIDO',flush=True)
'''

def start_training(kind):
    global train_process
    if train_process is not None and train_process.poll() is None:
        training_status.value=status('Treino','PENDENTE','Há outro treino em andamento.')
        return
    try:
        dataset=resolved_path({'fire':fire_dataset,'person':person_dataset,
                               'firefighter':firefighter_dataset}[kind])
        counts=dataset_report(dataset,kind)
        seed=seed_list.value
        if not seed:raise ValueError('Selecione um peso inicial YOLO.')
        import ultralytics
        run_name=f'{kind}_{dataset.parent.name}_{_vision_datetime.now():%Y%m%d_%H%M%S}'
        # Ultralytics resolve caminhos relativos de acordo com sua pasta de datasets.
        # Crie um YAML de execução com raiz absoluta para treinar a pasta selecionada.
        import yaml
        resolved_data=yaml.safe_load(dataset.read_text(encoding='utf-8'))
        data_root=Path(resolved_data.get('path') or dataset.parent)
        resolved_data['path']=str((dataset.parent/data_root).resolve() if not data_root.is_absolute() else data_root)
        normalized_yaml=EXPORT_DIR/f'{run_name}_dataset.yaml'
        normalized_yaml.write_text(yaml.safe_dump(resolved_data,allow_unicode=True),encoding='utf-8')
        request={'seed':seed,'dataset':str(normalized_yaml),'epochs':int(train_epochs.value),
                 'imgsz':int(train_size.value),'batch':int(train_batch.value),
                 'project':str(RUNS_DIR.resolve()),'name':run_name}
        log=EXPORT_DIR / f'{run_name}.log'
        log_handle=log.open('w',encoding='utf-8')
        try:
            train_process=subprocess.Popen([sys.executable,'-u','-c',TRAIN_SCRIPT,json.dumps(request)],
                cwd=str(PC_DIR),stdout=log_handle,stderr=subprocess.STDOUT)
        finally:
            log_handle.close()
        training_status.value=status('Treino','PENDENTE',
           f'{kind}: {counts}; execução {run_name}; log {log}. O detector de bombeiro é opcional.')
        def monitor(process):
            global train_process
            code=process.wait()
            if code==0:
                best=RUNS_DIR/run_name/'weights'/'best.pt'
                if best.is_file():
                    target=MODELS_DIR/f'{run_name}_best.pt'
                    shutil.copy2(best,target)
                    trained_models[kind]=target.resolve()
                    refresh_lists(preferred=str(target) if kind=='fire' else None)
                    if kind=='person':person_list.value=str(target)
                    if kind=='firefighter':firefighter_list.value=str(target)
                    label={'fire':fire_trained_info,'person':person_trained_info,
                           'firefighter':firefighter_trained_info}[kind]
                    label.value='<b>Modelo treinado:</b> '+html.escape(str(target))
                    training_status.value=status('Treino','OK',f'{kind} salvo em {target}.')
                else:training_status.value=status('Treino','FALHA',f'Pesos best.pt não encontrados; veja {log}.')
            else:
                tail='\n'.join(log.read_text(encoding='utf-8',errors='replace').splitlines()[-8:])
                training_status.value=status('Treino','FALHA',f'Código {code}; {log}; {tail}')
            train_process=None
        threading.Thread(target=monitor,args=(train_process,),daemon=True).start()
    except Exception as exc:
        training_status.value=status('Treino','FALHA',f'{type(exc).__name__}: {exc}')

def stop_train(_):
    if train_process is not None and train_process.poll() is None:
        train_process.terminate()
        training_status.value=status('Treino','PENDENTE','Interrupção solicitada; confira o log.')

def run_vision(session,source,source_type,stop):
    global calibration_width,calibration_height
    cap=None
    try:
        cap,first,backend,errors=open_capture(source,stop,source_type,input_fps.value)
        if cap is None:
            vision_status.value=status('Câmera','FALHA',f'{network_runtime.redacted_source(source)}: {errors}. Selecione outra fonte.')
            return
        calibration_height,calibration_width=first.shape[:2]
        pending=first
        failures=0
        # Arquivos têm relógio próprio. Se a inferência levar mais que 1/FPS,
        # avance o decodificador até o quadro correspondente ao tempo real.
        file_fps=float(input_fps.value) if source_type=='video' else 0.0
        file_frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if source_type=='video' else 0
        video_started=time.monotonic()
        next_frame_index=1  # o primeiro quadro já foi lido por open_capture
        skipped_frames=0
        paused_at=None
        if source_type=='video':
            duration=(file_frames/file_fps) if file_frames>0 else 0.0
            video_timeline.max=max(.1,duration)
            video_timeline.disabled=file_frames<=0
            vision_playback_state['internal']=True
            video_timeline.value=0
            vision_playback_state['internal']=False
            video_time_label.value=f'00:00 / {format_video_time(duration)}'
        while not stop.is_set():
            force_seek_frame=False
            if source_type=='video':
                with vision_playback_lock:
                    requested=vision_playback_state['seek']
                    vision_playback_state['seek']=None
                if requested is not None:
                    wanted=max(0,int(requested*file_fps))
                    if file_frames>0:wanted=min(wanted,file_frames-1)
                    if not cap.set(cv2.CAP_PROP_POS_FRAMES,wanted):
                        raise RuntimeError('Este arquivo não permite avançar para o tempo selecionado.')
                    video_started=time.monotonic()-wanted/file_fps
                    next_frame_index=wanted
                    pending=None
                    paused_at=time.monotonic() if video_pause.value else None
                    force_seek_frame=True
                    session.reset()
                if video_pause.value and not force_seek_frame and pending is None:
                    if paused_at is None:paused_at=time.monotonic()
                    stop.wait(.04)
                    continue
                if not video_pause.value and paused_at is not None:
                    video_started+=time.monotonic()-paused_at
                    paused_at=None
            if pending is not None:
                image,pending=pending,None
            else:
                if source_type=='video':
                    elapsed=time.monotonic()-video_started
                    wanted=next_frame_index if force_seek_frame else int(elapsed*file_fps)
                    if file_frames>0 and wanted>=file_frames:
                        video_started=time.monotonic()
                        wanted=0
                        next_frame_index=0
                        cap.set(cv2.CAP_PROP_POS_FRAMES,0)
                        session.reset()
                    if wanted<next_frame_index and not force_seek_frame:
                        stop.wait(max(0,min(.05,next_frame_index/file_fps-elapsed)))
                        continue
                    to_skip=max(0,wanted-next_frame_index)
                    if to_skip>int(file_fps*2) and cap.set(cv2.CAP_PROP_POS_FRAMES,wanted):
                        skipped_frames+=to_skip
                    else:
                        for _ in range(to_skip):
                            if stop.is_set() or not cap.grab():break
                            skipped_frames+=1
                ok,image=cap.read()
                if not ok or image is None:
                    if source_type=='video':
                        cap.set(cv2.CAP_PROP_POS_FRAMES,0)
                        video_started=time.monotonic()
                        next_frame_index=0
                        paused_at=time.monotonic() if video_pause.value else None
                        session.reset()
                        stop.wait(.02)
                        continue
                    failures+=1
                    if failures>=10:raise RuntimeError('Câmera parou de fornecer quadros.')
                    stop.wait(.1);continue
                if source_type=='video':next_frame_index=wanted+1
            failures=0
            if source_type=='video':
                position=max(0,(next_frame_index-1)/file_fps)
                vision_playback_state['internal']=True
                video_timeline.value=min(video_timeline.max,position)
                vision_playback_state['internal']=False
                video_time_label.value=(f'{format_video_time(position)} / '
                                        f'{format_video_time(duration)}')
            shown,record=session.process(image)
            h,w=shown.shape[:2]
            scaled=cv2.resize(shown,(850,max(1,int(h*850/w)))) if w>850 else shown
            ok,jpg=cv2.imencode('.jpg',scaled)
            if ok:vision_display.value=jpg.tobytes()
            state='FALHA' if record['blocked'] else 'OK' if record['target_valid'] else 'PENDENTE'
            vision_status.value=status('Visão',state,
               f'{session.mode} · {backend} · {record["strategy"]} · centros {len(session.route)} · '
               f'pessoa {record["person_detected"]} · bombeiro/EPI {record["firefighter_detected"]} · '
               f'regiões {len(session.regions)} · jato automático {record["jet_permitted"]} · '
               f'pan/tilt {record["pan_deg"]}/{record["tilt_deg"]} · '
               + (f'vídeo {file_fps:.1f} FPS; {skipped_frames} quadros pulados · ' if source_type=='video' else '')
               + str(session.journal_path))
            results['visao']={'camera':network_runtime.redacted_source(source),'modo':session.mode,'alvo':record['target_valid'],
                              'pessoa':record['person_detected'],'bloqueio':record['blocked'],
                              'bombeiro':record['firefighter_detected'],
                              'jato':record['jet_permitted'],'trajetoria':str(session.journal_path),
                              'fps_configurado':input_fps.value,
                              'quadros_pulados':skipped_frames if source_type=='video' else None}
            if source_type!='video':stop.wait(.03)
    except Exception as exc:
        vision_status.value=status('Visão','FALHA',f'{type(exc).__name__}: {exc}')
    finally:
        if cap is not None:cap.release()
        if source_type=='video':video_timeline.disabled=True
        session.close()

def start_vision(_):
    global vision_thread,vision_stop,vision_session
    vision_stop.set()
    vision_display.value=b''  # evita mostrar último quadro após falha de reconexão
    if vision_thread is not None:
        vision_thread.join(timeout=5)
        if vision_thread.is_alive():
            vision_status.value=status('Visão','FALHA','A câmera anterior ainda está fechando. Aguarde.')
            return
    camera_test_stop.set()
    if camera_test_thread is not None:
        camera_test_thread.join(timeout=4)
        if camera_test_thread.is_alive():
            vision_status.value=status('Câmera','FALHA','A prévia ainda está encerrando. Aguarde.')
            return
    try:
        stop_setup_camera_preview()
    except Exception as exc:
        vision_status.value=status('Prévia anterior','FALHA',str(exc))
        return
    try:
        source,source_type=selected_input()
        fire_path=(resolved_path(fire_list) if fire_strategy.value=='existing' else trained_models['fire'])
        person_path=(resolved_path(person_list) if person_strategy.value=='existing' else trained_models['person'])
        fire=load_model(fire_path,'fire')
        person=load_model(person_path,'person')
        fighter_path=(None if firefighter_strategy.value=='none' else
                      resolved_path(firefighter_list) if firefighter_strategy.value=='existing' else
                      trained_models['firefighter'])
        firefighter=load_model(fighter_path,'firefighter') if fighter_path else None
        if firefighter_strategy.value!='none' and firefighter is None:
            raise ValueError('Selecione ou treine o detector opcional de bombeiro/EPI.')
        params={k:v.value for k,v in params_widgets.items()}
        params['input_fps']=int(input_fps.value)
        vision_session=VisionSession(fire,person,params,calibration_fields(),mode_select.value,
                                     source,firefighter_model=firefighter)
        vision_stop=threading.Event()
        vision_thread=threading.Thread(target=run_vision,
          args=(vision_session,source,source_type,vision_stop),daemon=True)
        vision_status.value=status('Visão','PENDENTE',f'Abrindo {network_runtime.redacted_source(source)}; modelos de fogo e pessoa verificados.')
        vision_thread.start()
    except Exception as exc:
        vision_status.value=status('Pré-condições','FALHA',
          f'{type(exc).__name__}: {exc}. Treine ou selecione os dois modelos antes de iniciar.')

def stop_vision(_):
    vision_stop.set()
    vision_status.value=status('Visão','PENDENTE','Parando e liberando câmera...')

def reset_route(_):
    if vision_session is not None:vision_session.reset()

refresh_button.on_click(refresh_lists)
test_local_button.on_click(test_selected_camera)
stop_camera_test_button.on_click(stop_camera_test)
load_rgb_button.on_click(lambda _:apply_legacy(rgb_entries.value) if rgb_entries.value else None)
train_fire.on_click(lambda _:start_training('fire'))
train_person.on_click(lambda _:start_training('person'))
train_firefighter.on_click(lambda _:start_training('firefighter'))
stop_training.on_click(stop_train)
setup_save_button.on_click(save_setup)
setup_load_button.on_click(load_setup)
setup_delete_button.on_click(delete_setup)
vision_start.on_click(start_vision)
vision_stop_button.on_click(stop_vision)
vision_reset.on_click(reset_route)
refresh_lists()

display(widgets.VBox([
 widgets.HTML('<h3>Visão RGB e combate</h3><p>Escolha o modo antes de iniciar. '
  '<b>Simulação:</b> o vídeo continua sem congelar; as áreas atingidas escurecem temporariamente, '
  'e as próximas inferências analisam o quadro pintado. '
  '<b>Combate real:</b> a tela e a inferência seguem a câmera sem regiões pretas. '
  'Em ambos, pequenas caixas de fogo formam conjuntos delimitados por contornos roxos. '
  'Pessoa ou bombeiro/EPI detectado bloqueia o jato. '
  'Esta célula não abre a válvula física.</p>'),
 note(mode_select,'Modo salvo no setup; o runtime pode ler combat_mode e o arquivo de trajetória exportado.'),
 widgets.HTML('<h4>Entrada de vídeo e arquivos do pacote</h4>'),
 note(input_mode,'Escolha primeiro o tipo de entrada. Apenas o campo correspondente será mostrado.'),
 widgets.HBox([camera_input_controls,camera_test_preview]),
  widgets.HTML('<p>A webcam USB é aberta pelo PC. Vídeos salvos são lidos do diretório de vídeos do pacote e podem ser pausados e posicionados durante a simulação.</p>'),
 widgets.HBox([refresh_button]),
 widgets.HTML('<h4>Detector de fogo</h4><p>Escolha se usará um peso .pt existente ou se treinará um novo modelo. '
              'Aparecem apenas os campos da opção escolhida.</p>'),
 fire_strategy,fire_existing_panel,fire_training_panel,
 widgets.HTML('<h4>Detector de pessoas</h4><p>O peso yolo11n.pt já inclui a classe person. '
              'Você também pode ajustar um modelo com um dataset próprio.</p>'),
 person_strategy,person_existing_panel,person_training_panel,
 widgets.HTML('<h4>Detector opcional de bombeiros e vestimentas</h4><p>Escolha nenhum, um peso já treinado ou um dataset para treinar. '
              'As classes bombeiro, capacete e vestimenta de proteção são aceitas; detecções também bloqueiam o jato. '
              'Fogo e pessoa continuam obrigatórios.</p>'),
 firefighter_strategy,firefighter_existing_panel,firefighter_training_panel,
 shared_training_panel,
 widgets.HTML('<p>Para treinar, coloque imagens e rótulos YOLO nas pastas data/fire, data/person ou data/firefighter '
              '(train e val), ou selecione outro YAML. Os resultados são salvos em runs e models. '
              'A visão exige pesos .pt válidos de fogo e pessoa. O detector de bombeiros é opcional.</p>'),
 widgets.HTML('<h4>Parâmetros da visão e trajetória</h4><p>Estes controles definem os limiares dos detectores, a fusão de caixas sobrepostas e a velocidade da mira. '
              'Chamas detectadas são divididas em caixas pequenas sem sobreposição, com área limitada pelo controle abaixo. '
              'A mira visita o centróide não visitado mais próximo; ao alcançá-lo, força uma nova inferência antes de escolher outro. '
              'O jato liga quando a posição do impacto entra no contorno do conjunto de caixas e desliga ao sair. '
              'A parábola azul parte do centro inferior da imagem; a marca verde mostra a mira acima do impacto. '
              'Na simulação, o rastro escuro desaparece pelo tempo de restituição configurado, inicialmente 20 segundos. '
              'A análise seguinte ocorre sobre esse vídeo ainda pintado. A detecção de pessoa ou bombeiro bloqueia o jato.</p>'),
 widgets.HBox([rgb_entries,load_rgb_button]),
 widgets.HTML('<p>Os valores iniciais abaixo correspondem ao ensaio mostrado na imagem. '
              'Ao alternar para combate real, a velocidade inicial é 5 pixels por quadro; ajustes de cada modo são lembrados separadamente nesta execução. '
              'Os parâmetros de visão são independentes da calibração física de 12 pontos, realizada no runtime. '
              'Ajustes desta seção podem ser salvos em um setup e usados depois na combinação oficial.</p>'),
 *param_rows,
 calibration_panel,
 widgets.HBox([vision_start,vision_stop_button,vision_reset]),vision_status,
 vision_display,
 widgets.HTML('<p><b>Reprodução:</b> para arquivos de vídeo, pause, avance ou volte abaixo da tela única. '
              'Ao saltar no vídeo, os centróides visitados e o rastro são reiniciados. '
              'A reprodução e a detecção continuam enquanto a mira percorre os alvos.</p>'),
 video_controls,
 widgets.HTML('<h4>Salvar parâmetros</h4><p>A lista mostra somente setups do modo atual. '
  'Simulação e combate real têm nomes separados e permanecem selecionáveis na combinação oficial ao final do notebook.</p>'),
 widgets.HBox([setup_pick,setup_name_label]),
 widgets.HBox([setup_load_button,setup_save_button,setup_delete_button]),setup_status,
]))


## 4. Controle Xbox e simulação

Teste o Xbox, associe as entradas aos comandos e salve. O nome do setup é gerado do controle, do limite de RPM e da plataforma; os nomes de cada botão continuam editáveis. A cena visual simula esteiras, mira e câmera sem enviar movimentos físicos ao ESP32.


In [ ]:
# Célula completa: controle Xbox, setups, botões de função e simulação visual.
# Pré-requisitos: cfg, controller, widgets, results,
# setup_names/get/save/delete, config_path, _json_atomic, AppConfig e status_box.
import cv2, numpy as np, math, time, json, html, threading, subprocess, sys, random
from runtime import coupled_manual_pose

# Executar novamente esta célula encerra a leitura anterior sem deixar o joystick aberto.
for _event_name in ('control_stop', 'xbox_watch_stop'):
    if _event_name in globals(): globals()[_event_name].set()
for _thread_name in ('control_thread', 'xbox_watch_thread'):
    _old = globals().get(_thread_name)
    if _old is not None and _old.is_alive(): _old.join(timeout=2)
if 'xbox_read_stop' in globals(): xbox_read_stop.set()
if 'xbox_process' in globals() and xbox_process is not None and xbox_process.poll() is None:
    xbox_process.terminate()
if 'xbox_read_thread' in globals() and xbox_read_thread is not None and xbox_read_thread.is_alive():
    xbox_read_thread.join(timeout=2)

control_stop, xbox_watch_stop = threading.Event(), threading.Event()
control_thread = xbox_watch_thread = None
xbox_pygame_lock = threading.RLock()
bindings_lock = threading.RLock()
xbox_read_stop=threading.Event()
xbox_read_thread=None
xbox_process=None
latest_controls=None
reader_error=''
button_test_active=False
connection_snapshot = {'at':0., 'connected':False, 'name':'', 'index':-1,
                       'axes':0, 'buttons':0, 'hats':0}
awaiting_input = None
capture_baseline = None
suppress_capture = None

# A função de cada comando é fixa; 'Configurar' corrige apenas o índice físico
# indicado pelo driver do controle e registra um nome próprio para o comando.
INPUTS = {
 'lt':          ('LT',       'Aceleração proporcional das esteiras', 'axis',4),
 'drive_x':     ('Stick E X','Direção/curva das esteiras',           'axis',0),
 'drive_y':     ('Stick E Y','Avanço e ré das esteiras',             'axis',1),
 'lb':          ('LB',       'Freio das esteiras enquanto pressionado','button',4),
 'rt':          ('RT',       'Pressão proporcional da água',          'axis',5),
 'rb':          ('RB',       'Avançar atuador; focalizar jato',        'button',5),
 'r3':          ('R3',       'Clique do stick direito: recuar atuador', 'button',9),
 'cannon_x':    ('Stick D X','Mira horizontal; pan acompanha borda', 'axis',2),
 'cannon_y':    ('Stick D Y','Mira vertical; tilt acompanha borda',  'axis',3),
 'mode':        ('X',        'Alternar canhão manual/autônomo',       'button',2),
 'cool_internal':('A',      'Ligar/desligar arrefecimento interno', 'button',0),
 'cool_external':('B',      'Ligar/desligar arrefecimento externo', 'button',1),
 'leds':        ('Y',        'Ligar/desligar LEDs',                   'button',3),
 'camera_left': ('D-pad ←','Girar plataforma para 0° no manual',  'hat',0),
 'camera_right':('D-pad →','Girar plataforma para 180° no manual','hat',0),
 'overlay':     ('D-pad ↑', 'Mostrar/ocultar informações',           'hat',0),
 'link_hold':   ('Start',    'Conectar/desconectar: segurar 2 s',    'button',7),
 'stop':        ('D-pad ↓','Parada de emergência: segurar 1 s',     'hat',0),
 'water':       ('View',     'Ligar/desligar fornecimento de água',  'button',6),
}
DEFAULT_BINDINGS = {k:{'kind':kind,'index':i,**({'direction':[-1,0]} if k=='camera_left' else
                     {'direction':[1,0]} if k=='camera_right' else
                    {'direction':[0,1]} if k=='overlay' else
                    {'direction':[0,-1]} if k=='stop' else {})}
                    for k,(_,_,kind,i) in INPUTS.items()}
_saved_control = cfg.data.get('control', {})
bindings = {k:dict(_saved_control.get('input_bindings',{}).get(k,DEFAULT_BINDINGS[k])) for k in INPUTS}
if int(_saved_control.get('schema_version',0))<3:
    bindings['r3']=dict(DEFAULT_BINDINGS['r3'])
if int(_saved_control.get('schema_version',0))<4:
    for _key in ('camera_left','camera_right','stop'):
        bindings[_key]=dict(DEFAULT_BINDINGS[_key])
for _key,_legacy in [('drive_x','drive_x'),('drive_y','drive_y'),
                     ('cannon_x','cannon_x'),('cannon_y','cannon_y')]:
    if _key not in _saved_control.get('input_bindings',{}) and _legacy in cfg.data.get('xbox_axes',{}):
        bindings[_key]={'kind':'axis','index':int(cfg.data['xbox_axes'][_legacy])}
# O esquema 4 reserva ←/→ para a câmera e ↓ para emergência.
binding_warning=('Direções da plataforma e parada foram atualizadas; teste e salve.'
                 if int(_saved_control.get('schema_version',0))<4 else '')
input_names={k:_saved_control.get('button_names',{}).get(k, label) for k,(label,*_) in INPUTS.items()}
if int(_saved_control.get('schema_version',0))<4:
    for _key in ('stop','camera_left','camera_right'):input_names[_key]=INPUTS[_key][0]
if int(_saved_control.get('schema_version',0))<3:input_names['r3']=INPUTS['r3'][0]

connection_status=widgets.HTML(status_box('Controle Xbox','PENDENTE',
 'A leitura começa ao iniciar o teste de botões.').value,
 layout=widgets.Layout(width='100%'))
control_status=widgets.HTML(status_box('Simulação','PENDENTE',
 'Conecte um Xbox real e clique Iniciar simulação.').value,
 layout=widgets.Layout(width='100%'))
map_status=widgets.HTML(status_box('Mapeamento','PENDENTE',binding_warning).value if binding_warning else '')
speed_limit=widgets.IntSlider(value=max(10,min(80,int(_saved_control.get('speed_limit_rpm',80)))),
 min=10,max=80,step=1,description='Teto esteiras:',
 style={'description_width':'135px'},layout=widgets.Layout(width='565px'))
camera_scan_speed=widgets.IntSlider(value=int(_saved_control.get('camera_scan_speed_deg_s',12)),
 min=2,max=45,description='Varredura auto (°/s):',style={'description_width':'155px'},layout=widgets.Layout(width='540px'))
camera_manual_speed=widgets.IntSlider(value=int(_saved_control.get('camera_manual_speed_deg_s',45)),
 min=5,max=90,description='Giro manual (°/s):',style={'description_width':'155px'},layout=widgets.Layout(width='540px'))
camera_edge_offset=widgets.IntSlider(value=int(_saved_control.get('camera_edge_offset_px',8)),
 min=2,max=30,description='Margem da borda (px):',style={'description_width':'155px'},layout=widgets.Layout(width='540px'))
camera_recenter_step=widgets.IntSlider(value=int(_saved_control.get('camera_recenter_step_deg',20)),
 min=5,max=50,description='Passo recentrar (°):',style={'description_width':'155px'},layout=widgets.Layout(width='540px'))
control_name_label=widgets.HTML('<b>Nome automático:</b> Xbox, RPM e plataforma.')
control_pick=widgets.Dropdown(options=[('Novo setup','')]+[(n,n) for n in setup_names('control')],
 description='Salvos:',layout=widgets.Layout(width='420px'))
control_save=widgets.Button(description='Salvar/atualizar',button_style='success',layout=widgets.Layout(width='155px'))
control_load=widgets.Button(description='Carregar para editar',button_style='info',layout=widgets.Layout(width='170px'))
control_delete=widgets.Button(description='Excluir',button_style='danger',layout=widgets.Layout(width='95px'))
control_start=widgets.Button(description='Iniciar simulação',button_style='success',layout=widgets.Layout(width='165px'))
control_pause=widgets.Button(description='Parar simulação',button_style='warning',layout=widgets.Layout(width='150px'))
scene_image=widgets.Image(format='jpeg',layout=widgets.Layout(width='100%',max_width='1500px'))
reference_panel=widgets.HTML('',layout=widgets.Layout(width='100%'))
state_panel=widgets.HTML('',layout=widgets.Layout(width='100%'))
diagnostic=widgets.HTML('',layout=widgets.Layout(width='100%',height='480px',overflow='auto'))
diagnostic_reset=widgets.Button(description='Limpar leituras',layout=widgets.Layout(width='145px'))
diagnostic_start=widgets.Button(description='Testar todos os botões',button_style='info',layout=widgets.Layout(width='195px'))
diagnostic_stop=widgets.Button(description='Parar teste',button_style='warning',layout=widgets.Layout(width='115px'))
diagnostic_status=widgets.HTML(status_box('Leitura dos botões','PENDENTE',
 'Inicie o teste; cada entrada recebida ficará verde.').value)
diagnostic_seen={key:False for key in INPUTS}
raw_seen={'buttons':set(),'hats':set(),'axes':set()}
raw_snapshot={'buttons':[],'hats':[],'axes':[],'rest_axes':[]}
last_raw_input='Nenhum sinal bruto recebido ainda.'
message=widgets.HTML('<span style="color:#8b5e00">Aguardando controle.</span>',
 layout=widgets.Layout(width='100%'))
input_rows={}

def physical_label(binding):
    kind=binding.get('kind','?'); index=binding.get('index','?')
    if kind=='hat': return f'D-pad {index} {tuple(binding.get("direction",[]))}'
    return f'{"eixo" if kind=="axis" else "botão"} {index}'

def reset_diagnostic(_=None):
    global last_raw_input
    for key in diagnostic_seen:diagnostic_seen[key]=False
    for seen in raw_seen.values():seen.clear()
    last_raw_input='Nenhum sinal bruto recebido ainda.'
    update_diagnostic(zero_controls(),time.monotonic(),bool(connection_snapshot['connected']))

def update_diagnostic(controls,now,available):
    # Verde permanece depois de um recebimento; Limpar leituras reinicia o teste.
    rows=[]
    for key,(label,_,kind,_) in INPUTS.items():
        value=controls.get(key,False)
        active=(abs(float(value))>.13 if kind=='axis' and key not in ('lt','rt') else
                float(value)>.08 if kind=='axis' else bool(value))
        if active:diagnostic_seen[key]=True
        received=diagnostic_seen[key]
        color='#15803d' if received else '#9ca3af'
        state_label='Recebido' if received else 'Aguardando'
        amount=f' · {float(value):+.2f}' if kind=='axis' and active else ''
        rows.append('<div style="padding:5px 3px;border:1px solid #e5e7eb;border-radius:5px;font-size:12px">'
          '<span style="display:inline-block;width:11px;height:11px;border-radius:50%;'
          f'background:{color};margin-right:7px"></span><b>{html.escape(label)}</b> '
          f'<span style="color:{color}">{state_label}{amount}</span>'
          f'<br><small style="margin-left:19px;color:#6b7280">{html.escape(physical_label(bindings[key]))}</small></div>')
    raw_rows=[]
    def raw_chip(label,received,active=False):
        color='#15803d' if received else '#9ca3af'
        background='#dcfce7' if active else '#f9fafb'
        return ('<div style="font-size:11px;padding:4px;border:1px solid #e5e7eb;'
          f'border-radius:4px;background:{background}">'
          f'<span style="color:{color}">●</span> {html.escape(label)} '
          f'<small>{"recebido" if received else "aguardando"}</small></div>')
    for i,v in enumerate(raw_snapshot['buttons']):
        raw_rows.append(raw_chip(f'Botão {i}',i in raw_seen['buttons'],bool(v)))
    for i,hat in enumerate(raw_snapshot['hats']):
        for direction,label in (((-1,0),'←'),((1,0),'→'),((0,1),'↑'),((0,-1),'↓')):
            active=(direction[0]!=0 and hat[0]==direction[0] or
                    direction[1]!=0 and hat[1]==direction[1])
            raw_rows.append(raw_chip(f'D-pad {i} {label}',(i,direction) in raw_seen['hats'],active))
    for i,v in enumerate(raw_snapshot['axes']):
        active=(i<len(raw_snapshot['rest_axes']) and abs(v-raw_snapshot['rest_axes'][i])>.14)
        raw_rows.append(raw_chip(f'Eixo {i}: {v:+.2f}',i in raw_seen['axes'],active))
    hold=state['link_hold_since']
    progress=(min(2.,max(0.,now-hold)) if hold is not None and controls.get('link_hold') else 0.)
    simulation_running=control_thread is not None and control_thread.is_alive()
    link=('Conexão simulada ativa' if state['connected'] else
          f'Start detectado: {progress:.1f}/2,0 s' if controls.get('link_hold') and simulation_running else
          'Start recebido; contagem de 2 s ocorre na simulação' if controls.get('link_hold') else
          'Aguardando Start')
    if controls.get('stop') and state.get('stop_hold_since') is not None:
        link+=f' · Parada: {min(1.,max(0.,now-state["stop_hold_since"])):.1f}/1,0 s'
    info='OK: Xbox disponível' if available else ('Falha: '+reader_error if reader_error else 'Aguardando leitura do Xbox.')
    diagnostic.value=('<div style="background:#f8fafc;border:1px solid #cbd5e1;'
      'border-radius:8px;padding:12px;font-family:Arial,sans-serif">'
      '<b>Teste de todos os comandos</b><br><small>'+html.escape(info)+'</small><br>'
      '<span style="color:#155e75;font-size:12px">'+html.escape(link)+'</span><br>'
      '<span style="color:#92400e;font-size:11px">'+html.escape(last_raw_input)+'</span><hr>'+
      '<b style="font-size:12px">Funções configuradas</b>'
      '<div style="display:grid;grid-template-columns:repeat(3,minmax(170px,1fr));gap:5px">'+
      ''.join(rows)+'</div><br><b style="font-size:12px">Todas as entradas físicas detectadas</b>'
      '<div style="display:grid;grid-template-columns:repeat(6,minmax(95px,1fr));gap:4px">'+
      (''.join(raw_rows) if raw_rows else '<small>Aguardando contagem de botões do pygame.</small>')+
      '</div></div>')

def mark_diagnostic(controls):
    for key,(_,_,kind,_) in INPUTS.items():
        value=controls.get(key,False)
        if (abs(float(value))>.13 if kind=='axis' and key not in ('lt','rt') else
                float(value)>.08 if kind=='axis' else bool(value)):
            diagnostic_seen[key]=True

diagnostic_reset.on_click(reset_diagnostic)

def update_input_row(key):
    row=input_rows[key]
    b=bindings[key]
    row['physical'].value=f'<span style="color:#255da4">{html.escape(physical_label(b))}</span>'

def start_capture(key):
    global awaiting_input,capture_baseline
    with bindings_lock:
        awaiting_input=key
        capture_baseline=None
    start_button_test(None)
    map_status.value=status_box('Captura','TESTANDO',
       f'Mova/aperte {INPUTS[key][0]} até o fim do curso; aguarde a leitura.').value

for _key,(_label,_meaning,_kind,_index) in INPUTS.items():
    _color='#dbeafe' if _kind=='axis' else '#fee2e2' if _key in ('stop','link_hold') else '#e9f7ed'
    _name=widgets.Text(value=str(input_names[_key]),placeholder='Nome deste comando',
                       layout=widgets.Layout(width='42%'))
    _physical=widgets.HTML('',layout=widgets.Layout(width='105px'))
    _capture=widgets.Button(description='Configurar',layout=widgets.Layout(width='95px',height='27px'))
    _capture.on_click(lambda _,key=_key:start_capture(key))
    _meaning_html=widgets.HTML('<span style="font-size:11px">'+html.escape(_meaning)+
       '</span><br><small style="color:#375273;font-size:10px">Comando pré-definido</small>',
       layout=widgets.Layout(width='100%'))
    _badge=widgets.HTML('<div style="background:'+_color+';border-radius:5px;padding:3px 6px;'
      'font-size:12px;font-weight:bold">'+html.escape(_label)+'</div>',
      layout=widgets.Layout(width='83px'))
    input_rows[_key]={'name':_name,'physical':_physical,'configure':_capture,
                      'widget':widgets.VBox([
                        widgets.HBox([_badge,_meaning_html],layout=widgets.Layout(width='100%')),
                        widgets.HBox([_name,_physical,_capture],
                                     layout=widgets.Layout(width='100%',align_items='center'))],
                        layout=widgets.Layout(width='100%',border='1px solid #cbd5e1',
                                              padding='4px',min_height='73px'))}
    update_input_row(_key)

# A aquisição isolada impede leituras concorrentes: SDL vive em
# outro processo, enquanto este kernel só interpreta as leituras serializadas.
XBOX_READER_SCRIPT = r'''
import json, pygame, sys, time
pygame.init()
pygame.joystick.init()
index=int(sys.argv[1])
count=pygame.joystick.get_count()
if not 0 <= index < count and count==1:
    index=0  # Um único controle presente pode ser selecionado automaticamente.
if not 0 <= index < count:
    print('ERR:' + f'pygame detectou {count} controle(s); índice selecionado {index} indisponível. Use o índice do Xbox detectado nesta seção.',flush=True)
    sys.exit(2)
joy=pygame.joystick.Joystick(index)
joy.init()
print('INFO:' + json.dumps({'name':joy.get_name(),'index':index,'axes':joy.get_numaxes(),
 'buttons':joy.get_numbuttons(),'hats':joy.get_numhats()}),flush=True)
try:
    while True:
        pygame.event.pump()
        packet={'axes':[round(float(joy.get_axis(i)),4) for i in range(joy.get_numaxes())],
          'buttons':[int(joy.get_button(i)) for i in range(joy.get_numbuttons())],
          'hats':[list(joy.get_hat(i)) for i in range(joy.get_numhats())]}
        print('DATA:'+json.dumps(packet,separators=(',',':')),flush=True)
        time.sleep(.05)
finally:
    joy.quit()
    pygame.joystick.quit()
    pygame.quit()
'''

def probe_xbox_connection():
    with xbox_pygame_lock:
        snap=connection_snapshot.copy()
    good=snap['connected'] and time.monotonic()-snap['at']<.8
    if good:
        label=(f'{snap["name"]} · índice {snap["index"]} · {snap["axes"]} eixos · '
               f'{snap["buttons"]} botões · {snap["hats"]} D-pads. Dados recebidos às '
               +time.strftime('%H:%M:%S')+'.')
    else:label=reader_error or 'Aguardando dados do controle Xbox.'
    connection_status.value=status_box('Controle Xbox','OK' if good else 'FALHA' if reader_error else 'TESTANDO',label).value

def ensure_xbox_reader():
    global xbox_read_thread,xbox_read_stop,reader_error
    if xbox_read_thread is not None and xbox_read_thread.is_alive():return
    if 'xbox_monitor' in globals() and xbox_monitor.get('process') is not None and \
            'stop_xbox_preview' in globals():
        stop_xbox_preview()  # Libera uma leitura anterior antes da nova aquisição.
    xbox_read_stop=threading.Event();reader_error=''
    xbox_read_thread=threading.Thread(target=xbox_read_loop,args=(xbox_read_stop,),daemon=True)
    xbox_read_thread.start()

def start_button_test(_):
    global button_test_active
    button_test_active=True
    diagnostic_status.value=status_box('Leitura dos botões','TESTANDO',
        'Pressione os botões e mova os eixos; os sinais recebidos ficam verdes.').value
    ensure_xbox_reader()

def stop_button_test(_):
    global button_test_active
    button_test_active=False
    if control_thread is None or not control_thread.is_alive():
        xbox_read_stop.set()
        if xbox_process is not None and xbox_process.poll() is None:xbox_process.terminate()
    diagnostic_status.value=status_box('Leitura dos botões','PENDENTE','Teste interrompido.').value

# A captura é feita na mesma thread de leitura, impedindo duas consultas concorrentes ao SDL.
def capture_input(axes,pressed,hats,previous_pressed,previous_hats):
    global awaiting_input,capture_baseline,suppress_capture
    with bindings_lock:
        key=awaiting_input
        if key is None:return False
        if capture_baseline is None:
            capture_baseline=list(axes)
            return True
        candidates=[]
        for i in sorted(pressed-previous_pressed):candidates.append({'kind':'button','index':i})
        for i,hat in enumerate(hats):
            before=previous_hats[i] if i<len(previous_hats) else (0,0)
            if hat!=(0,0) and hat!=before:
                candidates.append({'kind':'hat','index':i,'direction':list(hat)})
        for i,value in enumerate(axes):
            base=capture_baseline[i] if i<len(capture_baseline) else 0.
            if abs(value-base)>.48:
                candidates.append({'kind':'axis','index':i,
                                   **({'rest':base,'active':value} if key in ('lt','rt') else {})})
        desired=INPUTS[key][2]
        allowed=('hat','button') if key in ('stop','camera_left','camera_right','overlay') else (desired,)
        candidates=[c for c in candidates if c['kind'] in allowed]
        if not candidates:return True
        candidate=candidates[0]
        suppress_capture=dict(candidate)
        physical=(candidate['kind'],candidate['index'],tuple(candidate.get('direction',[])))
        duplicate=[k for k,b in bindings.items() if k!=key and
          (b.get('kind'),b.get('index'),tuple(b.get('direction',[])))==physical]
        if duplicate:
            # Uma troca entre dois eixos/botões ocupados não exige uma entrada livre.
            other=duplicate[0]
            bindings[other]=dict(bindings[key])
            bindings[key]=candidate
            update_input_row(key);update_input_row(other)
            awaiting_input=None;capture_baseline=None
            map_status.value=status_box('Entradas trocadas','OK',
               f'{INPUTS[key][0]} recebeu {physical_label(candidate)}; '
               f'{INPUTS[other][0]} recebeu {physical_label(bindings[other])}. '
               'Confira as duas linhas antes de salvar.').value
        else:
            bindings[key]=candidate
            update_input_row(key)
            awaiting_input=None;capture_baseline=None
            map_status.value=status_box('Entrada capturada','OK',
               f'{INPUTS[key][0]} = {physical_label(candidate)}. Você pode editar o nome e salvar.').value
        return True

def _analog(key,axes):
    b=bindings[key];i=int(b.get('index',-1))
    return float(axes[i]) if b.get('kind')=='axis' and 0<=i<len(axes) else 0.

def _trigger(key,axes):
    b=bindings[key];i=int(b.get('index',-1))
    if b.get('kind')!='axis' or not 0<=i<len(axes):return 0.
    if b.get('rest_pending'):return 0.
    raw=float(axes[i]);rest=float(b.get('rest',-1.))
    active=float(b.get('active',1.))
    if abs(active-rest)<.1:return 0.
    return max(0.,min(1.,(raw-rest)/(active-rest)))

def _pressed(key,pressed,hats):
    b=bindings[key];i=int(b.get('index',-1))
    if b.get('kind')=='button':return i in pressed
    if b.get('kind')=='hat' and 0<=i<len(hats):
        direction=b.get('direction',[0,0]);return tuple(direction)==hats[i]
    return False

def xbox_read_loop(stop):
    global xbox_process,latest_controls,last_raw_input,reader_error,suppress_capture
    previous_pressed=set();previous_hats=[];previous_axes=[];info={};had_data=False;last_probe=0.
    try:
        index=int(controller.value) if controller.value is not None else -1
        process=subprocess.Popen([sys.executable,'-u','-c',XBOX_READER_SCRIPT,str(index)],
            stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True,bufsize=1)
        xbox_process=process
        for line in process.stdout:
            if stop.is_set():break
            line=line.strip()
            if line.startswith('ERR:'):
                raise RuntimeError(line[4:])
            if line.startswith('INFO:'):
                info=json.loads(line[5:])
                actual=int(info.get('index',index))
                if actual!=index:
                    options=list(controller.options)
                    if not any((entry[1] if isinstance(entry,tuple) else entry)==actual for entry in options):
                        options.append((f'{info["name"]} (detectado automaticamente)',actual))
                        controller.options=options
                    controller.value=actual
                connection_status.value=status_box('Controle Xbox','OK',
                    f'{info["name"]}: dispositivo aberto; aguardando leituras.').value
                continue
            if not line.startswith('DATA:'):continue
            data=json.loads(line[5:]);now=time.monotonic();had_data=True
            axes=[float(x) for x in data.get('axes',[])]
            pressed={i for i,b in enumerate(data.get('buttons',[])) if b}
            hats=[tuple(h) for h in data.get('hats',[])]
            if len(raw_snapshot['rest_axes'])!=len(axes):
                raw_snapshot['rest_axes']=list(axes)
            raw_snapshot.update(axes=list(axes),buttons=list(data.get('buttons',[])),hats=list(hats))
            raw_seen['buttons'].update(pressed)
            for i,h in enumerate(hats):
                if h[0]:raw_seen['hats'].add((i,(int(math.copysign(1,h[0])),0)))
                if h[1]:raw_seen['hats'].add((i,(0,int(math.copysign(1,h[1])))))
            raw_seen['axes'].update(i for i,v in enumerate(axes)
                if abs(v-raw_snapshot['rest_axes'][i])>.14)
            newly_pressed=sorted(pressed-previous_pressed)
            changed_hats=[(i,h) for i,h in enumerate(hats) if h!=(0,0) and
                          (i>=len(previous_hats) or h!=previous_hats[i])]
            changed_axes=[i for i,v in enumerate(axes) if i<len(previous_axes) and
                          abs(v-previous_axes[i])>.35]
            if newly_pressed:
                i=newly_pressed[0]
                related=[INPUTS[k][0] for k,b in bindings.items() if b.get('kind')=='button'
                         and b.get('index')==i]
                last_raw_input=f'Último sinal físico: botão {i} → '+(', '.join(related) if related else 'não configurado')
            elif changed_hats:
                last_raw_input=f'Último sinal físico: D-pad {changed_hats[0][0]} {changed_hats[0][1]}'
            elif changed_axes:
                last_raw_input=f'Último sinal físico: eixo {changed_axes[0]} = {axes[changed_axes[0]]:+.2f}'
            captured=capture_input(axes,pressed,hats,previous_pressed,previous_hats)
            with bindings_lock:
                for key in ('lt','rt'):
                    b=bindings[key];idx=int(b.get('index',-1))
                    if b.get('kind')=='axis' and 'rest' not in b and 0<=idx<len(axes):
                        base=axes[idx];b.update(rest=base,active=1.,rest_pending=base>.6)
                    if b.get('rest_pending') and 0<=idx<len(axes) and axes[idx]<.5:
                        b['rest']=axes[idx];b['rest_pending']=False
                controls={key:(_analog(key,axes) if key in ('drive_x','drive_y','cannon_x','cannon_y')
                    else _trigger(key,axes) if key in ('lt','rt') else _pressed(key,pressed,hats))
                    for key in INPUTS}
            diagnostic_controls=dict(controls)
            if button_test_active:mark_diagnostic(diagnostic_controls)
            previous_pressed=set(pressed);previous_hats=list(hats);previous_axes=list(axes)
            if suppress_capture is not None:
                b=suppress_capture;i=int(b['index'])
                still=(i in pressed if b['kind']=='button' else
                       i<len(hats) and hats[i]==tuple(b.get('direction',[])) if b['kind']=='hat' else
                       i<len(axes) and abs(axes[i]-float(b.get('rest',0)))>.12)
                if not still:suppress_capture=None
            if captured or suppress_capture is not None:controls=zero_controls()
            with xbox_pygame_lock:
                latest_controls=dict(controls)
                connection_snapshot.update(at=now,connected=True,name=info.get('name','Xbox'),
                    index=info.get('index',index),axes=len(axes),buttons=len(data.get('buttons',[])),hats=len(hats))
            if now-last_probe>1.:
                probe_xbox_connection()
                last_probe=now
            if button_test_active and (newly_pressed or changed_hats or changed_axes or
                 not hasattr(xbox_read_loop,'last_display') or now-xbox_read_loop.last_display>.1):
                update_diagnostic(diagnostic_controls,now,True)
                xbox_read_loop.last_display=now
                diagnostic_status.value=status_box('Leitura dos botões','OK',
                    'Dados chegando do pygame; pressione cada entrada para marcar Recebido.').value
        if not stop.is_set():
            details=(process.stderr.read()[-300:] if process.stderr else '').strip()
            raise RuntimeError(reader_error or details or
                'O processo de leitura terminou sem dados; verifique o controle/driver.')
    except Exception as exc:
        if not stop.is_set():
            reader_error=f'{type(exc).__name__}: {exc}'
            connection_status.value=status_box('Controle Xbox','FALHA',reader_error).value
            diagnostic_status.value=status_box('Leitura dos botões','FALHA',reader_error).value
            update_diagnostic(zero_controls(),time.monotonic(),False)
    finally:
        with xbox_pygame_lock:
            latest_controls=None
            connection_snapshot.update(at=time.monotonic(),connected=False,name='',index=-1,
                                      axes=0,buttons=0,hats=0)
        process=xbox_process;xbox_process=None
        if process is not None and process.poll() is None:
            process.terminate()
            try:process.wait(timeout=2)
            except subprocess.TimeoutExpired:process.kill()

def zero_controls():
    return {k:False for k in INPUTS if INPUTS[k][2]!='axis'} | {
     'drive_x':0.,'drive_y':0.,'cannon_x':0.,'cannon_y':0.,'lt':0.,'rt':0.}

def random_field_objects():
    """Dois marcos fixos por ensaio, gerados fora do chassi e do painel da mira."""
    rng=random.SystemRandom()
    first=(rng.randint(300,790),rng.randint(100,310))
    while True:
        second=(rng.randint(135,950),rng.randint(110,700))
        if math.dist(first,second)>330 and math.dist(second,(750,470))>185:
            break
    return [{'name':name,'x':xy[0],'y':xy[1],'color':color}
            for name,xy,color in [('A',first,(90,218,251)),
                                  ('B',second,(208,132,245))]]

def new_control_state():
    return {'objects':random_field_objects(),'connected':False,'emergency':False,'mode':'manual',
      'cool_internal':False,'cool_external':False,'leds':False,'water':False,
      'overlay':False,'camera_pan_deg':90.,'scan_direction':1,
      'last_recenter':0.,'focus':0.,'pan':0.,'tilt':45.,
      'aim_x':0.,'aim_y':0.,'x':750.,'y':470.,'heading':0.,
      'left_rpm':0.,'right_rpm':0.,'v_mps':0.,'omega':0.,
      'throttle':0.,'pressure':0.,'link_hold_since':None,'hold_latched':False,
      'stop_hold_since':None,'stop_latched':False,
      'previous':{},'hint':'Segure Start por 2 s para conectar o robô (simulado).'}
state=new_control_state()

def _deadzone(value,threshold=.08):
    value=float(value)
    return 0. if abs(value)<threshold else max(-1.,min(1.,value))

def apply_control(s, c, now, dt, rpm_limit, geometry):
    """Simula a máquina de estados; não envia sinais ao ESP32."""
    dt=max(0.,min(.12,float(dt)))
    prev=s['previous'];rising=lambda key: bool(c.get(key)) and not prev.get(key,False)
    if c.get('stop'):
        if s['stop_hold_since'] is None:s['stop_hold_since']=now
        if not s['stop_latched'] and now-s['stop_hold_since']>=1.:
            s.update(connected=False,emergency=True,water=False,leds=False,
                     cool_internal=False,cool_external=False,throttle=0.,pressure=0.,
                     left_rpm=0.,right_rpm=0.)
            s['hint']='PARADA DE EMERGÊNCIA: solte a seta para baixo e segure Start 2 s para rearmar.'
            s['hold_latched']=True
            s['stop_latched']=True
    else:
        s['stop_hold_since']=None;s['stop_latched']=False
    start=bool(c.get('link_hold'))
    if start:
        if s['link_hold_since'] is None:s['link_hold_since']=now
        if not s['hold_latched'] and now-s['link_hold_since']>=2.:
            neutral=(abs(float(c.get('drive_x',0)))<.08 and abs(float(c.get('drive_y',0)))<.08
                      and float(c.get('lt',0))<.08 and float(c.get('rt',0))<.08
                      and not c.get('stop'))
            if not neutral:
                s['hint']='Solte LT/RT e a seta para baixo; centralize os sticks para conectar/rearmar.'
            elif s['connected']:
                s.update(connected=False,water=False,pressure=0.,throttle=0.,
                         cool_internal=False,cool_external=False,leds=False)
                s['hint']='Conexão cortada. Segure Start por 2 s para reconectar.'
                s['hold_latched']=True
            else:
                s['connected']=True;s['emergency']=False
                s['hint']='Conectado (simulado). View libera água; LT acelera; LB freia.'
                s['hold_latched']=True
    else:s['link_hold_since']=None;s['hold_latched']=False
    if not s['connected']:
        s['throttle']=s['pressure']=s['v_mps']=s['omega']=0.
        s['left_rpm']=s['right_rpm']=0.
        if not s['emergency'] and not start and (float(c.get('lt',0))>.08 or
             float(c.get('rt',0))>.08 or any(rising(k) for k in ('water','mode','rb','r3'))):
            s['hint']='Primeiro segure Start por 2 s para conectar.'
    else:
        for key,field in (('mode','mode'),('water','water'),('cool_internal','cool_internal'),
                          ('cool_external','cool_external'),('leds','leds'),('overlay','overlay')):
            if not rising(key):continue
            if key=='mode':s['mode']='auto' if s['mode']=='manual' else 'manual'
            else:s[field]=not s[field]
            s['hint']=f'{INPUTS[key][0]}: {INPUTS[key][1]}.'
        if c.get('rb') and c.get('r3'):
            s['hint']='Solte RB ou R3: focalizar e dispersar são comandos opostos.'
        elif c.get('rb'):s['focus']=min(1.,s['focus']+.35*dt)
        elif c.get('r3'):s['focus']=max(0.,s['focus']-.35*dt)
        throttle=max(0.,min(1.,float(c.get('lt',0))))
        if c.get('lb'):
            if throttle>.08:s['hint']='LB acionado: solte o freio para acelerar.'
            throttle=0.
        x=_deadzone(c.get('drive_x',0));y=-_deadzone(c.get('drive_y',0))
        s['throttle']=throttle
        left=throttle*max(-1.,min(1.,y+x))
        right=throttle*max(-1.,min(1.,y-x))
        s['left_rpm']=rpm_limit*left;s['right_rpm']=rpm_limit*right
        pressure=max(0.,min(1.,float(c.get('rt',0))))
        s['pressure']=pressure if s['water'] else 0.
        if pressure>.08 and not s['water']:
            s['hint']='Aperte View para liberar água antes de pressionar RT.'
        radius=float(geometry.get('sprocket_radius_m') or 0)
        width=float(geometry.get('track_width_m') or 0)
        if radius>0 and width>0:
            vl=left*rpm_limit*2*math.pi*radius/60
            vr=right*rpm_limit*2*math.pi*radius/60
            s['v_mps']=(vl+vr)/2;s['omega']=(vr-vl)/width
            # Ângulo positivo na tela aponta à direita; yaw cinemático positivo é anti-horário.
            s['heading']-=s['omega']*dt
            travel=s['v_mps']*115*dt
        else:
            s['v_mps']=s['omega']=0.
            s['heading']-=(right-left)*1.2*dt
            travel=(left+right)*42*dt
        s['x']=(s['x']+math.sin(s['heading'])*travel-110)%1280+110
        s['y']=(s['y']-math.cos(s['heading'])*travel-90)%650+90
        if s['mode']=='manual':
            direction=int(c.get('camera_right',False))-int(c.get('camera_left',False))
            (s['pan'],s['tilt'],s['camera_pan_deg'],s['aim_x'],s['aim_y']) = coupled_manual_pose(
                s['pan'],s['tilt'],s['camera_pan_deg'],
                _deadzone(c.get('cannon_x',0)),_deadzone(c.get('cannon_y',0)),
                direction,dt,camera_manual_speed.value,camera_edge_offset.value,370.,
                camera_recenter_step.value)
        if s['mode']=='auto' and not c.get('stop'):
            s['camera_pan_deg']+=s['scan_direction']*camera_scan_speed.value*dt
            if s['camera_pan_deg']>=180:s['camera_pan_deg']=180.;s['scan_direction']=-1
            elif s['camera_pan_deg']<=0:s['camera_pan_deg']=0.;s['scan_direction']=1
    s['previous']={k:bool(c.get(k)) for k in INPUTS if INPUTS[k][2]!='axis'}
    return s

# Composição em pixels: figura única para usar toda a largura disponível.
def _line(img,a,b,color,thick=1):
    cv2.line(img,tuple(map(int,a)),tuple(map(int,b)),color,thick,cv2.LINE_AA)
def _text(img,value,point,scale=.5,color=(220,230,238),thick=1):
    cv2.putText(img,str(value),tuple(map(int,point)),cv2.FONT_HERSHEY_SIMPLEX,
                scale,color,thick,cv2.LINE_AA)
def _ray(origin,heading,distance):
    return (origin[0]+math.sin(heading)*distance,origin[1]-math.cos(heading)*distance)
def _cone(overlay,origin,angle,length,half_angle,color):
    points=[(int(origin[0]),int(origin[1]))]
    for a in np.linspace(angle-half_angle,angle+half_angle,17):
        points.append(tuple(map(int,_ray(origin,float(a),length))))
    cv2.fillPoly(overlay,[np.asarray(points,np.int32)],color)

def project_field_objects(s, sx0, sx1, sy0, sy1):
    """Projeta objetos fixos do mapa no quadro de 120° da câmera."""
    camera_angle=float(s['heading'])+math.radians(float(s['camera_pan_deg'])-90.)
    half_fov=math.radians(60.)
    projected=[]
    for obj in s['objects']:
        dx=float(obj['x'])-float(s['x']);dy=float(obj['y'])-float(s['y'])
        bearing=math.atan2(dx,-dy)
        relative=(bearing-camera_angle+math.pi)%(2*math.pi)-math.pi
        if abs(relative)>half_fov:continue
        distance=math.hypot(dx,dy)
        px=int((sx0+sx1)/2+relative/half_fov*(sx1-sx0)/2)
        py=int(sy1-18-min(1.,distance/1050.)*(sy1-sy0-37))
        projected.append((obj,px,py))
    return projected

def draw_control_scene(s,c):
    image=np.full((790,1500,3),(29,35,45),dtype=np.uint8)
    # O grid começa logo abaixo do título, sem a faixa vazia vista na imagem anterior.
    for x in range(30,1500,80):_line(image,(x,37),(x,790),(42,51,62))
    for y in range(37,790,80):_line(image,(0,y),(1500,y),(42,51,62))
    _text(image,'SIMULACAO DO ROBO E DO CANHAO',(18,26),.62,(235,244,250),2)
    pos=(float(s['x']),float(s['y']));h=float(s['heading']);front=_ray(pos,h,60)
    camera_angle=h+math.radians(float(s['camera_pan_deg'])-90.)
    half_fov=math.radians(60.)
    field=image.copy()
    _cone(field,pos,camera_angle,1900.,half_fov,(218,225,231))
    image=cv2.addWeighted(field,.11,image,.89,0)
    for edge in (camera_angle-half_fov,camera_angle+half_fov):
        _line(image,pos,_ray(pos,edge,1900),(230,238,245),1)
    _line(image,pos,_ray(pos,camera_angle,135),(235,245,253),1)
    for obj in s['objects']:
        point=(int(obj['x']),int(obj['y']))
        cv2.circle(image,point,17,obj['color'],-1,cv2.LINE_AA)
        cv2.circle(image,point,20,(240,245,251),1,cv2.LINE_AA)
        _text(image,'OBJETO '+obj['name'],(point[0]+22,point[1]-9),.44,obj['color'])
    # Cones transparentes: três faróis frontais e dois laterais.
    lights=image.copy()
    if s['leds']:
        for shift in (-.2,0,.2):_cone(lights,front,h+shift,190,.15,(30,225,250))
        for side in (-1,1):_cone(lights,pos,h+side*math.pi/2,135,.23,(30,225,250))
    image=cv2.addWeighted(lights,.24,image,.76,0)
    # Cabos de comunicação e água seguem a traseira, conforme os dois estados.
    rear=_ray(pos,h+math.pi,58)
    lateral=np.array([math.cos(h),math.sin(h)])
    for active,shift,source,color in (
        (s['connected'],-10,(0,690),(65,222,105)),
        (s['water'] and s['connected'],10,(0,718),(245,166,48))):
        if not active:continue
        target=np.array(rear)+lateral*shift
        bend1=(max(45,int(target[0]*.36)),int(source[1]*.75+target[1]*.25))
        bend2=(max(60,int(target[0]*.75)),int(source[1]*.20+target[1]*.80))
        path=np.asarray([source,bend1,bend2,tuple(target.astype(int))],np.int32)
        cv2.polylines(image,[path],False,(18,29,35),8,cv2.LINE_AA)
        cv2.polylines(image,[path],False,color,4,cv2.LINE_AA)
        cv2.circle(image,tuple(target.astype(int)),5,color,-1,cv2.LINE_AA)
    # Carrinho de esteiras rotacionado pelo rumo integrado.
    ux=np.array([math.sin(h),-math.cos(h)]);side=np.array([math.cos(h),math.sin(h)])
    center=np.array(pos)
    def footprint(half_w,half_l,offset=0):
        p=center+side*offset
        return np.array([p-side*half_w-ux*half_l,p+side*half_w-ux*half_l,
                         p+side*half_w+ux*half_l,p-side*half_w+ux*half_l],np.int32)
    cv2.fillPoly(image,[footprint(34,55,-35),footprint(34,55,35)],(72,84,95))
    cv2.fillPoly(image,[footprint(42,43)],(116,142,159))
    cv2.polylines(image,[footprint(42,43)],True,(212,228,236),2,cv2.LINE_AA)
    _line(image,pos,front,(238,245,250),3)
    cv2.circle(image,tuple(map(int,pos)),8,(241,249,253),-1,cv2.LINE_AA)
    # Arrefecimento: partículas azuis ao redor (externo) ou sobre o chassi (interno).
    particles=image.copy()
    if s['cool_external']:
        for i in range(110):
            angle=i*2.39996
            radius=69+39*(.5+.5*math.sin(i*5.71))
            point=_ray(pos,angle,radius)
            cv2.circle(particles,tuple(map(int,point)),2+i%3,(255,178,52),-1,cv2.LINE_AA)
    if s['cool_internal']:
        for i in range(58):
            local_x=32*math.sin(i*2.39996)
            local_y=35*math.sin(i*3.828)
            point=center+side*local_x+ux*local_y
            cv2.circle(particles,tuple(point.astype(int)),2+i%3,(255,205,95),-1,cv2.LINE_AA)
    if s['cool_external'] or s['cool_internal']:
        image=cv2.addWeighted(particles,.72,image,.28,0)
    # Vetor laranja: aceleração e direção; cone azul: jato e focalização.
    drive_amount=float(s['throttle'])*min(1.,math.hypot(
        float(c.get('drive_x',0)),float(c.get('drive_y',0))))
    heading_vector=h+math.atan2(float(c.get('drive_x',0)),
                           max(.1,abs(float(c.get('drive_y',0)))))
    if float(c.get('drive_y',0))>0:heading_vector+=math.pi
    if drive_amount>.01:
        tip=_ray(pos,heading_vector,170*drive_amount)
        cv2.arrowedLine(image,tuple(map(int,pos)),tuple(map(int,tip)),(40,170,255),
                        3,cv2.LINE_AA,tipLength=.16)
    if s['water'] and s['connected']:
        pressure=float(s['pressure'])
        blue_angle=h+math.radians(float(s['pan']))
        spread=.035+.38*(1.-float(s['focus']))
        length=115+220*pressure
        jet=image.copy()
        _cone(jet,front,blue_angle,length,spread,(245,167,48))
        image=cv2.addWeighted(jet,.48 if pressure>.02 else .15,
                              image,.52 if pressure>.02 else .85,0)
        for edge in (blue_angle-spread,blue_angle+spread):
            _line(image,front,_ray(front,edge,length),(255,193,92),2 if pressure>.02 else 1)
    # Estados e sensores aparecem sempre no painel lateral, fora da imagem.
    # Janela de mira, escala de panorama e inclinação; retículo contido em ±62%.
    bx0,by0,bx1,by1=1040,205,1480,500
    cv2.rectangle(image,(bx0,by0),(bx1,by1),(16,28,43),-1)
    cv2.rectangle(image,(bx0,by0),(bx1,by1),(170,188,210),2)
    horizon=int(350+(float(s['tilt'])-45.)*1.4)
    _line(image,(bx0+25,horizon),(bx1-25,horizon),(140,170,195),2)
    _text(image,'CAMERA 120° / MIRA',(bx0+12,by0+24),.48,(223,236,250))
    sx0,sx1=bx0+47,bx1-23
    sy0,sy1=by0+44,by1-57
    for deg in range(-60,61,10):
        tick=int(sx0+(deg+60)*(sx1-sx0)/120)
        major=deg%30==0
        _line(image,(tick,sy1),(tick,sy1+(9 if major else 4)),
              (152,177,201) if major else (82,105,130),1)
        if major:_text(image,f'{deg:+d}',(tick-14,by1-30),.43,(186,202,222))
    # Projeção angular: girar a câmera desloca os objetos no quadro menor;
    # deslocar ou girar o robô muda a direção e a distância relativas.
    for obj,px,py in project_field_objects(s,sx0,sx1,sy0,sy1):
        cv2.circle(image,(px,py),9,obj['color'],-1,cv2.LINE_AA)
        cv2.circle(image,(px,py),11,(245,250,255),1,cv2.LINE_AA)
        _text(image,obj['name'],(px+13,py+5),.43,(244,247,252))
    for deg in range(0,91,10):
        tick=int(sy1-deg*(sy1-sy0)/90)
        _line(image,(sx0-(9 if deg%30==0 else 4),tick),(sx0,tick),
              (152,177,201) if deg%30==0 else (82,105,130),1)
        _text(image,str(deg),(bx0+9,tick+4),.44,(186,202,222))
    pan_tick=int((sx0+sx1)/2+s['aim_x']*(sx1-sx0)/2)
    tilt_tick=int(sy1-s['tilt']*(sy1-sy0)/90)
    cv2.circle(image,(pan_tick,sy1+2),4,(45,211,245),-1,cv2.LINE_AA)
    cv2.circle(image,(sx0-3,tilt_tick),4,(45,211,245),-1,cv2.LINE_AA)
    _text(image,f'CANHAO {s["pan"]:+.0f}  CAMERA {s["camera_pan_deg"]:.0f}',(bx0+115,by1-9),.42,(244,196,95))
    cv2.rectangle(image,(bx0+44,by0+38),(bx1-19,by1-54),(96,122,149),1)
    mx=int((sx0+sx1)/2+s['aim_x']*(sx1-sx0)/2)
    my=int((sy0+sy1)/2+s['aim_y']*(sy1-sy0)/2)
    cv2.circle(image,(mx,my),10,(50,230,120),2,cv2.LINE_AA)
    _line(image,(mx-23,my),(mx+23,my),(70,235,125),2)
    _line(image,(mx,my-23),(mx,my+23),(70,235,125),2)
    _text(image,f'TILT {s["tilt"]:.0f} graus',(bx0+308,by0+55),.42,(216,226,241))
    if s['overlay']:
        _text(image,'Vetor laranja: tracao   Cone azul: jato   Cabos: verde=comunicacao, azul=agua',
              (20,765),.48)
        _text(image,'Pontos azuis: arrefecimento',(1110,765),.42)
    image=cv2.resize(image,(1080,569),interpolation=cv2.INTER_AREA)
    ok,encoded=cv2.imencode('.jpg',image,[cv2.IMWRITE_JPEG_QUALITY,85])
    return encoded.tobytes() if ok else b''

def update_sim_panels(s,c,now):
    refs=[]
    for key,(label,meaning,kind,_) in INPUTS.items():
        custom=input_rows[key]['name'].value.strip() or label
        # Função em destaque; índice físico fica somente no teste de botões.
        active=(abs(float(c.get(key,0)))>.08 if kind=='axis' else bool(c.get(key)))
        bg='#e7f7e9' if active else '#f8fafc'
        refs.append('<span style="display:inline-block;padding:2px 5px;background:'+bg+
                    ';border:1px solid #e5e7eb;border-radius:4px;font-size:10px" title="'+
                    html.escape(meaning,quote=True)+'">'+html.escape(custom)+' · '+
                    html.escape(meaning)+'</span>')
    if not reference_panel.value:
        reference_panel.value=('<details style="font-family:Arial,sans-serif;background:#f8fafc;'
        'border:1px solid #cbd5e1;padding:3px 6px;border-radius:5px">'
        '<summary style="font-size:11px;cursor:pointer">Referência dos comandos (abrir/fechar)</summary>'
        '<div style="display:flex;flex-wrap:wrap;gap:3px;margin-top:4px">'+
        ''.join(refs)+'</div></details>')
    telemetry=results.get('enlace',{}) if isinstance(results,dict) else {}
    if not isinstance(telemetry,dict):telemetry={}
    last=telemetry.get('latest',telemetry)
    if not isinstance(last,dict):last={}
    temperature=last.get('temperature_c',last.get('temperature','--'))
    humidity=last.get('humidity_pct',last.get('humidity','--'))
    link_elapsed=(f'{min(2.,now-s["link_hold_since"]):.1f}/2,0 s'
                  if s['link_hold_since'] is not None else '0/2,0 s')
    stop_elapsed=(f'{min(1.,now-s["stop_hold_since"]):.1f}/1,0 s'
                  if s['stop_hold_since'] is not None else '0/1,0 s')
    on=lambda value:'Ligado' if value else 'Desligado'
    toggle=lambda value:'green' if value else 'red'
    moving=bool(s['throttle']>.02 and
                (abs(float(c.get('drive_x',0)))>.08 or abs(float(c.get('drive_y',0)))>.08))
    aim_active=bool(s['mode']=='manual' and
        (abs(float(c.get('cannon_x',0)))>.08 or abs(float(c.get('cannon_y',0)))>.08))
    rows=[
      ('Conexão simulada',on(s['connected']),toggle(s['connected'])),
      ('Start pressionado',link_elapsed,'green' if c.get('link_hold') else ''),
      ('Parada de emergência',on(s['emergency']),'red' if s['emergency'] else 'green'),
      ('Seta para baixo pressionada',stop_elapsed,'red' if c.get('stop') else ''),
      ('Modo do canhão','Automático' if s['mode']=='auto' else 'Manual',
                         'green' if s['mode']=='auto' else 'red'),
      ('Fornecimento de água',on(s['water']),toggle(s['water'])),
      ('Pressão comandada',f'{100*s["pressure"]:.0f}%',
                           'green' if s['pressure']>.02 else 'red' if c.get('rt',0)>.08 else ''),
      ('Aceleração LT',f'{100*s["throttle"]:.0f}%',
                       'green' if s['throttle']>.02 else 'red' if c.get('lb') else ''),
      ('Freio LB',on(c.get('lb')),'red' if c.get('lb') else ''),
      ('Limite das esteiras',f'{speed_limit.value} RPM',''),
      ('Esteira esquerda',f'{s["left_rpm"]:+.1f} RPM','green' if moving else ''),
      ('Esteira direita',f'{s["right_rpm"]:+.1f} RPM','green' if moving else ''),
      ('Velocidade linear',f'{s["v_mps"]:+.2f} m/s','green' if moving else ''),
      ('Giro do robô',f'{s["omega"]:+.2f} rad/s','green' if moving else ''),
      ('Posição no quadro',f'({s["x"]:.0f}, {s["y"]:.0f}) px',''),
      ('Rumo',f'{math.degrees(s["heading"])%360:.0f}°',''),
      ('Stick esquerdo X/Y',f'{float(c.get("drive_x",0)):+.2f} / {float(c.get("drive_y",0)):+.2f}',
                              'green' if moving else ''),
      ('Stick direito X/Y',f'{float(c.get("cannon_x",0)):+.2f} / {float(c.get("cannon_y",0)):+.2f}',
                              'green' if aim_active else ''),
      ('Pan do canhão',f'{s["pan"]:+.1f}°','green' if aim_active else ''),
      ('Tilt do canhão',f'{s["tilt"]:.1f}°','green' if aim_active else ''),
      ('Mira na janela',f'{s["aim_x"]:+.2f} / {s["aim_y"]:+.2f}',
                         'green' if aim_active else ''),
      ('Focalização do jato',f'{100*s["focus"]:.0f}%',
                              'green' if c.get('rb') else 'red' if c.get('r3') else ''),
      ('Avanço RB',on(c.get('rb')),'green' if c.get('rb') else ''),
      ('Recuo R3',on(c.get('r3')),'red' if c.get('r3') else ''),
      ('Arrefecimento interno',on(s['cool_internal']),toggle(s['cool_internal'])),
      ('Arrefecimento externo',on(s['cool_external']),toggle(s['cool_external'])),
      ('LEDs',on(s['leds']),toggle(s['leds'])),
      ('Plataforma da câmera',f'{s["camera_pan_deg"]:.1f}° (0–180)',
         'green' if c.get('camera_left') or c.get('camera_right') else ''),
      ('Legendas na imagem',on(s['overlay']),toggle(s['overlay'])),
      ('Temperatura medida',str(temperature),''),('Umidade medida',str(humidity),'')]
    colors={'green':('#e8f6e9','#12652d'),'red':('#fdeaea','#a02b2b'),
            '':('#ffffff','#273444')}
    lines=[]
    for label,value,tone in rows:
        bg,fg=colors[tone]
        lines.append('<div style="display:flex;justify-content:space-between;gap:6px;'
          f'background:{bg};color:{fg};border-bottom:1px solid #e5e7eb;'
          'padding:2px 5px;min-height:17px">'
          f'<span>{html.escape(label)}</span><b style="text-align:right">{html.escape(value)}</b></div>')
    state_panel.value=('<div style="font:12px Arial,sans-serif;background:#f8fafc;'
        'border:1px solid #cbd5e1;padding:6px;border-radius:6px">'
        '<b style="font-size:14px">Estados e parâmetros de controle</b>'+''.join(lines)+'</div>')

def control_loop(stop):
    global state
    last=time.monotonic();last_status=0.;last_panel=0.
    try:
        while not stop.is_set():
            now=time.monotonic();dt=min(.12,now-last);last=now
            with xbox_pygame_lock:
                snap=connection_snapshot.copy()
                fresh=snap['connected'] and now-snap['at']<.35
                controls=dict(latest_controls) if fresh and latest_controls is not None else zero_controls()
            if not fresh and state['connected']:
                state=new_control_state()
                state['hint']='Leitura do Xbox interrompida: comandos neutralizados. Reconecte e segure Start.'
            apply_control(state,controls,now,dt,int(speed_limit.value),cfg.data.get('geometry',{}))
            if now-last_panel>.15:
                update_sim_panels(state,controls,now)
                last_panel=now
            if now-last_status>.8:
                control_status.value=status_box('Simulação','OK' if fresh else 'FALHA' if reader_error else 'TESTANDO',
                    f'{snap["name"]}: dados recebidos; segure Start por 2 s.' if fresh else
                    reader_error or 'Aguardando dados do controle; veja o teste acima.').value
                last_status=now
            message.value='<div style="background:#fff8dd;padding:8px;border-radius:5px">'+html.escape(state['hint'])+'</div>'
            scene_image.value=draw_control_scene(state,controls)
            results['controle']={'fonte':'xbox' if fresh else 'indisponível',
                'conexao_simulada':state['connected'],'parada':state['emergency'],
                'modo':state['mode'],'rpm_estimadas':[round(state['left_rpm'],2),round(state['right_rpm'],2)],
                'pan_tilt_comandados':[round(state['pan'],1),round(state['tilt'],1)],
                'pressao_virtual':round(state['pressure'],3),'limite_rpm':int(speed_limit.value)}
            stop.wait(.05)
    except Exception as exc:
        control_status.value=status_box('Simulação','FALHA',f'{type(exc).__name__}: {exc}').value

def start_control(_):
    global control_stop,control_thread,state
    control_stop.set()
    if control_thread is not None and control_thread.is_alive():
        control_thread.join(timeout=2)
        if control_thread.is_alive():
            control_status.value=status_box('Simulação','FALHA','A leitura anterior ainda está encerrando.').value
            return
    state=new_control_state()
    ensure_xbox_reader()
    control_stop=threading.Event()
    control_thread=threading.Thread(target=control_loop,args=(control_stop,),daemon=True)
    control_thread.start()

def stop_control(_):
    control_stop.set()
    control_status.value=status_box('Simulação','PENDENTE','Leitura interrompida.').value
    if not button_test_active:
        xbox_read_stop.set()
        if xbox_process is not None and xbox_process.poll() is None:xbox_process.terminate()

def refresh_control_names(selected=''):
    names=setup_names('control')
    control_pick.options=[('Novo setup','')]+[(n,n) for n in names]
    control_pick.value=selected if selected in names else ''

def validate_bindings():
    with bindings_lock:
        signatures=[]
        for key,(label,_,kind,_) in INPUTS.items():
            b=bindings[key]
            allowed=('hat','button') if key in ('stop','camera_left','camera_right','overlay') else (kind,)
            if b.get('kind') not in allowed or int(b.get('index',-1))<0:
                raise ValueError(f'Configure a entrada física de {label}.')
            signatures.append((kind,int(b['index']),tuple(b.get('direction',[]))))
        if len(signatures)!=len(set(signatures)):
            raise ValueError('Há índices físicos duplicados. Configure uma entrada diferente em cada linha.')
        return {key:dict(bindings[key]) for key in INPUTS}

def save_control(_):
    global cfg
    try:
        if controller.value is None or int(controller.value)<0:
            raise ValueError('Conecte um controle Xbox e inicie seu teste nesta seção.')
        mapped=validate_bindings()
        axes={k:int(mapped[k]['index']) for k in ('drive_x','drive_y','cannon_x','cannon_y')}
        legacy={k:int(mapped[k]['index']) for k in
                ('mode','cool_internal','cool_external','leds')}
        # O runtime legado só admite botões, não D-pad nem retenção por 1 s.
        # Evite associar por engano a emergência ao botão A (índice do hat 0).
        legacy.update(stop=65533,jet=65535,rearm=65534)
        control_settings={**cfg.data.get('control',{}),'schema_version':4,
            'speed_limit_rpm':int(speed_limit.value),'input_bindings':mapped,
            'camera_scan_speed_deg_s':camera_scan_speed.value,
            'camera_manual_speed_deg_s':camera_manual_speed.value,
            'camera_edge_offset_px':camera_edge_offset.value,
            'camera_recenter_step_deg':camera_recenter_step.value,
            'button_names':{k:input_rows[k]['name'].value.strip() or INPUTS[k][0] for k in INPUTS}}
        payload={'xbox_axes':axes,'xbox_buttons':legacy,'xbox_index':int(controller.value),
                 'control':control_settings}
        name=setup_save('control',auto_setup_name('control',payload),payload)
        data=json.loads(config_path.read_text(encoding='utf-8'))
        data.update(payload);_json_atomic(config_path,data);cfg=AppConfig(config_path)
        refresh_control_names(name)
        map_status.value=status_box('Setup de controle','OK',
                 f'{name} salvo com nomes, entradas físicas e teto de {speed_limit.value} RPM.').value
    except Exception as exc:
        map_status.value=status_box('Setup de controle','FALHA',str(exc)).value

def load_control(_):
    try:
        name=control_pick.value
        if not name:raise ValueError('Selecione um setup salvo.')
        data=setup_get('control',name)
        ctl=data.get('control',{})
        old_axes=data.get('xbox_axes',{})
        with bindings_lock:
            for key in INPUTS:
                fallback=({'kind':'axis','index':int(old_axes[key])} if key in old_axes else
                          DEFAULT_BINDINGS[key])
                bindings[key]=dict(ctl.get('input_bindings',{}).get(key,fallback))
                if (int(ctl.get('schema_version',0))<3 and key=='r3') or (
                     int(ctl.get('schema_version',0))<4 and key in ('stop','camera_left','camera_right')):
                    bindings[key]=dict(DEFAULT_BINDINGS[key])
                input_rows[key]['name'].value=(INPUTS[key][0] if key in ('r3','stop') and
                    int(ctl.get('schema_version',0))<3 else
                    ctl.get('button_names',{}).get(key,INPUTS[key][0]))
                update_input_row(key)
        speed_limit.value=int(ctl.get('speed_limit_rpm',80))
        camera_scan_speed.value=int(ctl.get('camera_scan_speed_deg_s',12))
        camera_manual_speed.value=int(ctl.get('camera_manual_speed_deg_s',45))
        camera_edge_offset.value=int(ctl.get('camera_edge_offset_px',8))
        camera_recenter_step.value=int(ctl.get('camera_recenter_step_deg',20))
        map_status.value=status_box('Setup de controle','OK',
                 f'{name} carregado; edite nomes, capture entradas e salve novamente.').value
    except Exception as exc:
        map_status.value=status_box('Setup de controle','FALHA',str(exc)).value

def delete_control(_):
    try:
        name=control_pick.value
        if not name:raise ValueError('Selecione um setup salvo.')
        setup_delete('control',name);refresh_control_names()
        map_status.value=status_box('Setup de controle','OK',f'{name} excluído.').value
    except Exception as exc:
        map_status.value=status_box('Setup de controle','FALHA',str(exc)).value

diagnostic_start.on_click(start_button_test);diagnostic_stop.on_click(stop_button_test)
control_start.on_click(start_control);control_pause.on_click(stop_control)
control_save.on_click(save_control);control_load.on_click(load_control);control_delete.on_click(delete_control)
scene_image.value=draw_control_scene(state,zero_controls())
update_diagnostic(zero_controls(),time.monotonic(),False)
update_sim_panels(state,zero_controls(),time.monotonic())

# Ocupa toda a largura disponível do notebook clássico e do JupyterLab.
display(widgets.HTML('<style>.container {width:98% !important; max-width:1600px !important;} '
                     '.jp-NotebookPanel-notebook {max-width: none !important;}</style>'))
display(widgets.VBox([
 widgets.HTML('<h3>1. Comandos e teste do Xbox</h3><p>O teste já verifica '
              'a conexão. Pressione cada botão e mova joysticks, gatilhos e D-pad. São listadas '
              'todas as entradas físicas que o pygame detectar, inclusive as não configuradas. '
              'Os sinais recebidos ficam verdes até clicar Limpar leituras. Use o índice mostrado '
              'para corrigir uma associação nos cartões abaixo.</p>'),
 widgets.HBox([diagnostic_start,diagnostic_stop,diagnostic_reset]),
 connection_status,diagnostic_status,diagnostic,
 widgets.HTML('<p><b>Associação de comandos</b> — os comandos '
              'têm função fixa. Dê um nome e capture o índice físico em cada cartão. '
              'Azul: eixos; vermelho: Start e parada. D-pad para baixo por 1 s aciona a parada; '
              'R3 é o clique do stick direito. Entradas repetidas são recusadas.</p>'),
 widgets.GridBox([input_rows[k]['widget'] for k in INPUTS],
     layout=widgets.Layout(width='100%',grid_template_columns='repeat(3, minmax(0, 1fr))',
                           grid_gap='6px')),map_status,
   widgets.HTML('<h3>2. Setups de controle</h3><p>Carregue, edite ou exclua um setup salvo.</p>'),
  widgets.HBox([control_pick,control_name_label],layout=widgets.Layout(width='100%')),
  widgets.HBox([control_load,control_save,control_delete]),speed_limit,
widgets.HTML('<h3>3. Simulação visual do robô e do canhão</h3><p>É necessário um Xbox conectado. '
              'Todos os atuadores começam desligados. Segure Start por 2 s; LT acelera, '
              'o stick esquerdo guia, LB freia; View libera água e RT regula pressão. '
              'O stick direito move a mira; ←/→ giram a plataforma no modo manual. '
              'Segurar D-pad para baixo por 1 s interrompe e exige rearme. '
              'A imagem é uma simulação; não envia comandos ao ESP32.</p>'),
 widgets.HBox([control_start,control_pause]),control_status,message,
 widgets.HBox([
   widgets.VBox([scene_image,reference_panel],layout=widgets.Layout(width='70%')),
   widgets.VBox([state_panel],layout=widgets.Layout(width='29%'))],
   layout=widgets.Layout(width='100%',align_items='flex-start'))
],layout=widgets.Layout(width='100%',max_width='1550px')))


## 5. Hardware do protótipo

Selecione os motores, o driver BTS7960/DRV8833/L298N e MG996R/SG90 independentemente para pan, tilt e plataforma. O foco usa SG90. Selecione o formato dos dados de cada DHT: no GPIO16 foi confirmado DHT22; o GPIO17 ainda não foi medido e usará a mesma decodificação DHT22 inicialmente. **Você pode cadastrar tudo antes da montagem.** As caixas de saídas físicas e da válvula registram o que deverá funcionar no perfil de combate real; se a combinação usar visão em simulação, seu firmware desliga essas saídas automaticamente. Salvar ou compilar não aciona componentes. Para gravar, selecione a COM detectada com o ESP32 ligado por USB. O nome do setup é automático.


In [ ]:
# Seleção explícita dos componentes do protótipo; sem cadastro manual de motores/driver.
import html, json
import ipywidgets as widgets
from IPython.display import display
import profile_manager
from actuator_catalog import hardware_pairs
from serial.tools import list_ports
if not FORM_SAVED:
    raise RuntimeError('Salve a conexão na seção 2 antes de configurar o hardware.')
_hw=cfg.data.get('hardware',{})
hw_pick=widgets.Dropdown(options=[('Default','')],description='Salvos:',layout=widgets.Layout(width='490px'))
hw_name_label=widgets.HTML('<b>Nome automático:</b> motor, driver selecionado, servos e saídas.')
hw_motor=widgets.Dropdown(options=[('Motor DC 3–6 V','dc_3_6v'),('Mabuchi 12 V','bts7960_12v')],
                          value=_hw.get('drive_kind','dc_3_6v'),description='Esteiras:',layout=widgets.Layout(width='510px'))
hw_driver=widgets.Dropdown(options=[('BTS7960 · dois módulos','bts7960'),
                                    ('DRV8833 · ponte dupla','drv8833'),
                                    ('L298N · ponte dupla','l298n')],
                           value=_hw.get('drive_driver_id','bts7960') if _hw.get('drive_driver_id','bts7960') in ('bts7960','drv8833','l298n') else 'bts7960',
                           description='Driver:',layout=widgets.Layout(width='510px'))
servo_options=[('MG996R','mg996r'),('SG90','sg90')]
hw_pan=widgets.Dropdown(options=servo_options,value=_hw.get('pan_servo_kind',_hw.get('cannon_motor_id','mg996r')),description='Canhão pan:',
                        layout=widgets.Layout(width='350px'))
hw_tilt=widgets.Dropdown(options=servo_options,value=_hw.get('tilt_servo_kind',_hw.get('cannon_motor_id','mg996r')),description='Canhão tilt:',
                         layout=widgets.Layout(width='350px'))
hw_camera=widgets.Dropdown(options=servo_options,value=_hw.get('camera_servo_kind','sg90'),description='Plataforma:',
                           layout=widgets.Layout(width='350px'))
hw_focus=widgets.Dropdown(options=[('SG90 · foco','sg90')],value='sg90',description='Foco:',
                          layout=widgets.Layout(width='350px'))
hw_dispersed=widgets.BoundedIntText(value=int(_hw.get('focus_dispersed_deg',175)),min=0,max=180,
                         description='Disperso (°):',layout=widgets.Layout(width='260px'))
hw_focused=widgets.BoundedIntText(value=int(_hw.get('focus_focused_deg',5)),min=0,max=180,
                         description='Focalizado (°):',layout=widgets.Layout(width='260px'))
hw_dht_internal=widgets.Dropdown(options=['DHT11','DHT22'],value=_hw.get('dht_internal_type','DHT22'),description='Dados interno:',layout=widgets.Layout(width='310px'))
hw_dht_external=widgets.Dropdown(options=['DHT11','DHT22'],value=_hw.get('dht_external_type','DHT22'),description='Dados externo:',layout=widgets.Layout(width='310px'))
hw_real=widgets.Checkbox(value=bool(_hw.get('physical_outputs',False)),
                         description='Prever saídas físicas para o perfil real')
hw_jet=widgets.Checkbox(value=bool(_hw.get('jet_output_enabled',False)),
                        description='Prever válvula física do jato no perfil real')
hw_upload=widgets.Dropdown(options=[('Selecione a COM para gravar','auto')],
              description='Gravar via USB:',layout=widgets.Layout(width='500px'))
hw_note=widgets.HTML();hw_status=widgets.HTML()
def hw_refresh_ports(_=None):
    ports=list(list_ports.comports());value=str(_hw.get('upload_port','auto'))
    hw_upload.options=[('Selecione a COM para gravar','auto')]+[(f'{p.device} · {p.description}',p.device) for p in ports]
    hw_upload.value=value if value in [v for _,v in hw_upload.options] else 'auto'

def hw_show(_=None):
    model='DC 3–6 V' if hw_motor.value=='dc_3_6v' else 'Mabuchi 12 V'
    supply={'bts7960':('dois módulos BTS7960, um por motor; GPIO25/26 PWM + GPIO27 EN, '
                        'GPIO32/33 PWM + GPIO23 EN; fonte ≥5,5 V (para motor 3–6 V, até 6 V)'),
            'drv8833':('um módulo DRV8833 de dois canais; AIN1/AIN2 GPIO25/26, BIN1/BIN2 GPIO32/33; '
                        'nSLEEP em nível ativo; fonte 2,7–10,8 V e corrente dentro dos limites do módulo'),
            'l298n':('um módulo L298N de dois canais; IN1/IN2 GPIO25/26, ENA PWM GPIO27; '
                      'IN3/IN4 GPIO32/33, ENB PWM GPIO23. Remova jumpers ENA/ENB; lógica 5 V; '
                      'a queda de tensão interna pode reduzir muito a tensão no motor de 3–6 V')}[hw_driver.value]
    errors, warnings, _=hardware_pairs({'drive_motor_id':'dc_3_6v' if hw_motor.value=='dc_3_6v' else 'mabuchi_12v',
        'drive_driver_id':hw_driver.value,'cannon_motor_id':hw_pan.value,'cannon_driver_id':'mg996r_pwm'},physical=False)
    message=('<b style="color:#b42318">Incompatível: '+html.escape('; '.join(errors))+'</b>' if errors else
             '<b>Seleção compatível sob condições:</b> '+html.escape(model+' + '+supply+'.'))
    hw_note.value=(message+'<br>Pan GPIO18, tilt GPIO19, plataforma GPIO4 e foco GPIO14: MG/SG usam PWM RC 50 Hz; todos precisam de fonte externa e GND comum. '
       'DHT11 interno e externo: decodificação DHT22 confirmada no GPIO16 e padrão também no GPIO17, ainda não medido; relés: GPIO22/13. '
       'Sem medição de corrente da peça, confira corrente de partida antes do acionamento físico. '
       'Wi-Fi local/USB segue a seção 2: <b>'+html.escape(cfg.data.get('esp',{}).get('transport','wifi_local'))+'</b>.')

def hw_refresh(selected=''):
    names=setup_names('hardware')
    hw_pick.options=[('Default','')]+[(x,x) for x in names]
    hw_pick.value=selected if selected in names else ''

def hw_payload():
    if hw_dispersed.value==hw_focused.value:raise ValueError('Extremos disperso e focalizado devem ser diferentes.')
    motor_id='dc_3_6v' if hw_motor.value=='dc_3_6v' else 'mabuchi_12v'
    hw={'drive_kind':hw_motor.value,'drive_motor_id':motor_id,'drive_driver_id':hw_driver.value,
        'cannon_kind':'mg996r','cannon_motor_id':hw_pan.value,'cannon_driver_id':'mg996r_pwm',
        'pan_servo_kind':hw_pan.value,'tilt_servo_kind':hw_tilt.value,
        'camera_servo_kind':hw_camera.value,'focus_actuator_model':'SG90',
        'dht_internal_type':hw_dht_internal.value,'dht_external_type':hw_dht_external.value,
        'focus_driver_model':'PWM direto do ESP32 (sinal, não alimentação)',
        'focus_dispersed_deg':hw_dispersed.value,'focus_focused_deg':hw_focused.value,
        'interface_board_id':'none','gear_ratio':1.,
        'physical_outputs':hw_real.value,'jet_output_enabled':hw_jet.value,
        'upload_port':hw_upload.value}
    if hw_jet.value and not hw_real.value:raise ValueError('Válvula física requer saídas físicas habilitadas.')
    errors,warnings,_=hardware_pairs(hw,physical=hw_real.value)
    if errors:raise ValueError(' | '.join(errors))
    return {'esp':{'transport':cfg.data.get('esp',{}).get('transport','wifi_local'),
                   'serial_port':cfg.data.get('esp',{}).get('serial_port','auto'),
                   'host':'192.168.4.1'},'hardware':hw},warnings

def hw_save(_):
    try:
        payload,warnings=hw_payload()
        name=setup_save('hardware',auto_setup_name('hardware',payload),payload)
        hw_refresh(name)
        hw_status.value=status_box('Hardware','OK','Salvo. '+(' · '.join(warnings) if warnings else
                    'Monte conforme o quadro de conexões na seção 8.')).value
    except Exception as exc:hw_status.value=status_box('Hardware','FALHA',str(exc)).value

def hw_load(_):
    try:
        if not hw_pick.value:raise ValueError('Escolha um setup salvo.')
        data=setup_get('hardware',hw_pick.value).get('hardware',{})
        hw_motor.value=data.get('drive_kind','dc_3_6v')
        hw_driver.value=data.get('drive_driver_id','bts7960')
        hw_pan.value=data.get('pan_servo_kind',data.get('cannon_motor_id','mg996r'))
        hw_tilt.value=data.get('tilt_servo_kind',data.get('cannon_motor_id','mg996r'))
        hw_camera.value=data.get('camera_servo_kind','mg996r')
        hw_dht_internal.value=data.get('dht_internal_type','DHT22')
        hw_dht_external.value=data.get('dht_external_type','DHT22')
        hw_dispersed.value=int(data.get('focus_dispersed_deg',175))
        hw_focused.value=int(data.get('focus_focused_deg',5))
        hw_real.value=bool(data.get('physical_outputs',False))
        hw_jet.value=bool(data.get('jet_output_enabled',False))
        upload=str(data.get('upload_port','auto'))
        hw_upload.value=upload if upload in [v for _,v in hw_upload.options] else 'auto'
        hw_status.value=status_box('Hardware','OK','Carregado para edição.').value
    except Exception as exc:hw_status.value=status_box('Hardware','FALHA',str(exc)).value

def hw_delete(_):
    try:
        if not hw_pick.value:raise ValueError('Default não é um setup salvo.')
        setup_delete('hardware',hw_pick.value);hw_refresh()
        hw_status.value=status_box('Hardware','OK','Excluído.').value
    except Exception as exc:hw_status.value=status_box('Hardware','FALHA',str(exc)).value
hw_b_ports=widgets.Button(description='Atualizar COM');hw_b_ports.on_click(hw_refresh_ports)
hw_b_save=widgets.Button(description='Salvar setup',button_style='success');hw_b_save.on_click(hw_save)
hw_b_load=widgets.Button(description='Carregar para editar');hw_b_load.on_click(hw_load)
hw_b_delete=widgets.Button(description='Excluir salvo',button_style='danger');hw_b_delete.on_click(hw_delete)
hw_refresh();hw_refresh_ports();hw_show();hw_motor.observe(hw_show,names='value');hw_driver.observe(hw_show,names='value')
display(widgets.VBox([widgets.HBox([hw_pick,hw_name_label]),widgets.HBox([hw_b_load,hw_b_save,hw_b_delete]),
    widgets.HBox([hw_motor,hw_driver]),hw_note,
    widgets.HTML('<b>Servos independentes (sinal RC PWM; fonte externa)</b>'),
    widgets.HBox([hw_pan,hw_tilt,hw_camera,hw_focus]),
    widgets.HTML('Foco SG90: padrão focalizado 5° e disperso 175°. RB focaliza; R3 dispersa. Cada outro eixo aceita MG996R ou SG90.'),
    widgets.HBox([hw_dispersed,hw_focused]),widgets.HBox([hw_dht_internal,hw_dht_external]),hw_real,hw_jet,
    widgets.HBox([hw_upload,hw_b_ports]),hw_status]))


## 6. Combinação oficial

Escolha os setups de cada grupo ou **Default**, salve a combinação e defina-a como oficial. Os nomes dos grupos são criados automaticamente a partir dos componentes e parâmetros; a combinação reúne os quatro nomes separados por `*`. Uma combinação de simulação pode usar o cadastro físico futuro: as saídas permanecem desligadas no firmware de simulação. A montagem será verificada na seção 8.


In [ ]:
# Seção 6 — selecionar até quatro setups; ausência usa os defaults de fábrica.
import datetime as _final_dt
import ipywidgets as widgets
from IPython.display import display

FINAL_SECTIONS=(('connection','Conexão ESP32'),('vision','Visão e jato'),
                ('control','Xbox e velocidade'),('hardware','ESP32 e motores'))
final_section_pickers={}
for key,title in FINAL_SECTIONS:
    names=setup_names(key)
    final_section_pickers[key]=widgets.Dropdown(
        options=[('Default (sem setup salvo)','')]+[(n,n) for n in names],
        description=title+':',style={'description_width':'170px'},
        layout=widgets.Layout(width='100%'))
final_name=widgets.Text(description='Nome completo:',disabled=True,
    style={'description_width':'120px'},layout=widgets.Layout(width='100%'))
final_pick=widgets.Dropdown(options=[('Nova combinação','')],description='Salvas:',
    style={'description_width':'120px'},layout=widgets.Layout(width='100%'))
final_status=widgets.HTML(layout=widgets.Layout(width='100%'))
final_official=widgets.HTML(layout=widgets.Layout(width='100%'))
final_load=widgets.Button(description='Carregar para editar',button_style='info',
    layout=widgets.Layout(min_width='190px',height='38px'))
final_save=widgets.Button(description='Salvar combinação',button_style='success',
    layout=widgets.Layout(min_width='195px',height='38px'))
final_delete=widgets.Button(description='Excluir combinação',button_style='danger',
    layout=widgets.Layout(min_width='175px',height='38px'))
final_activate=widgets.Button(description='Definir como setup oficial',button_style='primary',
    layout=widgets.Layout(min_width='225px',height='38px'))

def final_message(title,state,message):
    final_status.value=status_box(title,state,message).value

def final_refresh_official():
    name=official_name()
    final_official.value=status_box('Atualmente oficial','OK' if name else 'PENDENTE',
        name or 'Nenhum. Salve e ative uma combinação para usar os programas independentes.').value

def final_selections():
    return {section:pick.value or '' for section,pick in final_section_pickers.items()}

def final_segment(section,choice):
    return choice or 'default'

def final_auto_name(selections):
    # '*' separa as quatro seções. Cada seção não salva aparece como default.
    return '*'.join(final_segment(section,selections[section]) for section,_ in FINAL_SECTIONS)

def final_preview(_=None):
    try:final_name.value=final_auto_name(final_selections())
    except Exception as exc:final_name.value=f'Corrija o setup selecionado: {exc}'

for picker in final_section_pickers.values():
    picker.observe(final_preview,names='value')

def refresh_combinations(selected=''):
    names=setup_names('combinations')
    options=[('Nova combinação','')]
    for name in names:
        saved=setup_get('combinations',name).get('saved_at','')
        options.append((f'{name}  ·  {saved[:19]}' if saved else name,name))
    final_pick.options=options
    final_pick.value=selected if selected in names else ''
    final_preview()

def save_combination(_):
    try:
        selected=final_selections()
        assemble_combination(selected)  # Valida defaults e escolhas salvas.
        name=final_auto_name(selected)
        now=_final_dt.datetime.now().astimezone()
        setup_save('combinations',name,{'sections':selected,'saved_at':now.isoformat()})
        refresh_combinations(name)
        final_message('Combinação','OK',
            f'{name} salvo. Clique em Definir como setup oficial para preparar PC e ESP32.')
    except Exception as exc:
        final_message('Combinação','FALHA',str(exc))

def load_combination(_):
    try:
        name=final_pick.value
        if not name:raise ValueError('Selecione uma combinação salva.')
        selected=setup_get('combinations',name)['sections']
        for section,picker in final_section_pickers.items():
            value=selected.get(section,'')
            if value not in [item for _,item in picker.options]:
                raise ValueError(f'Setup {value!r} ausente em {section}.')
            picker.value=value
        final_name.value=name
        final_message('Combinação','OK',f'{name} carregado. Altere as seleções e salve.')
    except Exception as exc:final_message('Combinação','FALHA',str(exc))

def delete_combination(_):
    try:
        name=final_pick.value
        if not name:raise ValueError('Selecione a combinação a excluir.')
        setup_delete('combinations',name)
        refresh_combinations();final_refresh_official()
        final_message('Combinação','OK',f'{name} excluído.')
    except Exception as exc:final_message('Combinação','FALHA',str(exc))

def activate_official(_):
    try:
        name=final_pick.value
        if not name:raise ValueError('Salve e selecione uma combinação primeiro.')
        from profile_manager import prepare
        _,data=prepare(name)
        data['_official']['activated_at']=_final_dt.datetime.now(
            _final_dt.timezone.utc).isoformat()
        _json_atomic(SETUPS_DIR/'official'/'active.json',data)
        final_refresh_official()
        warning=(' Calibre a RPM máxima para comandar tração física.'
                 if float(data.get('geometry',{}).get('output_max_rpm',0))==0 else '')
        notice=(' Componentes cadastrados para montagem futura; no perfil de simulação, firmware e runtime não acionam saídas físicas.'
                if data['_official'].get('outputs_deferred') else '')
        final_message('Setup oficial','OK',f'{name} ativado para os runtimes.{warning}{notice}')
    except Exception as exc:final_message('Setup oficial','FALHA',str(exc))

final_load.on_click(load_combination)
final_save.on_click(save_combination)
final_delete.on_click(delete_combination)
final_activate.on_click(activate_official)
refresh_combinations();final_refresh_official()
display(widgets.VBox([
    widgets.HTML('<h3>7. Combinação completa</h3><p>Selecione os setups salvos de cada seção. '
        'Se uma lista estiver vazia ou mantida em Default, o pacote aplicará os parâmetros padrão '
        'sem criar um setup salvo. Os quatro trechos do nome são separados por *; os trechos '
        'selecionados incluem conexão, fonte de vídeo, modo e motor/driver. Uma nova gravação '
        'com as mesmas seleções atualiza a data do registro.</p>'),
    final_official,*final_section_pickers.values(),final_name,final_pick,
    widgets.HBox([final_load,final_save,final_delete,final_activate],
        layout=widgets.Layout(flex_flow='row wrap',grid_gap='8px')),
    final_status],layout=widgets.Layout(width='100%',max_width='1400px')))


## 7. Preparar runtime e gravar firmware

**Preparar runtimes** gera configurações para todas as combinações completas salvas. A simulação sempre desliga as saídas físicas, mesmo que os componentes estejam cadastrados para uso futuro; o perfil real usa as autorizações do setup. **Compilar** não exige componentes conectados. **Gravar** exige ESP32, cabo USB e uma COM detectada; use **Atualizar COM** se a porta não aparecer. Depois da montagem, confira firmware e cada componente na seção 8.


In [ ]:
# Seção 7 — preparar runtimes e firmware com partição adequada para ESP32 de 4 MB.
import re
import threading
import html
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from profile_manager import names as profile_names, prepare_all, assemble, run_pio

fw_choice = widgets.Dropdown(description='Setup completo:',
    style={'description_width':'135px'}, layout=widgets.Layout(width='620px'))
fw_port = widgets.Dropdown(description='Porta USB:',options=[('Selecione a COM detectada','')],
    style={'description_width':'110px'},layout=widgets.Layout(width='480px'))
fw_port_refresh=widgets.Button(description='Atualizar COM',layout=widgets.Layout(width='140px'))
fw_port_row = widgets.HBox([fw_port,fw_port_refresh])
fw_port_row.layout.display = 'none'
fw_refresh = widgets.Button(description='Atualizar combinações',
    layout=widgets.Layout(min_width='190px'))
fw_export = widgets.Button(description='Preparar runtimes', button_style='success',
    layout=widgets.Layout(min_width='190px'))
fw_build = widgets.Button(description='Compilar firmware selecionado', button_style='info',
    layout=widgets.Layout(min_width='255px'))
fw_upload = widgets.Button(description='Gravar firmware somente por cabo USB', button_style='warning',
    layout=widgets.Layout(min_width='290px'))
fw_status = widgets.HTML(layout=widgets.Layout(width='100%'))
fw_connection = widgets.HTML('<b>Gravação:</b> aguardando operação.')
fw_progress = widgets.IntProgress(value=0, min=0, max=100, description='Gravação:',
    layout=widgets.Layout(width='100%', display='none'))
fw_log = widgets.HTML(layout=widgets.Layout(width='100%', height='230px', overflow='auto'))
_fw_busy = threading.Lock()
_fw_lines = []

def fw_ensure_partition():
    """Usa uma partição real de 3 MB, sem OTA, na flash de 4 MB."""
    ini = Path(ROOT) / 'esp32' / 'platformio.ini'
    original = ini.read_text(encoding='utf-8')
    header = re.search(r'(?m)^\[env:esp32dev\]\s*$', original)
    if not header:
        raise ValueError(f'Não encontrei [env:esp32dev] em {ini}')
    start = header.end()
    next_section = re.search(r'(?m)^\s*\[[^\]]+\]', original[start:])
    end = start + next_section.start() if next_section else len(original)
    section = original[start:end]
    for key, value in (
        ('board_build.partitions', 'huge_app.csv'),
        ('board_upload.maximum_size', '3145728'),
    ):
        pattern = rf'(?m)^[ \t]*{re.escape(key)}[ \t]*=[^\r\n]*'
        replacement = f'{key} = {value}'
        if re.search(pattern, section):
            section = re.sub(pattern, lambda _: replacement, section, count=1)
        else:
            section = '\n' + replacement + section
    updated = original[:start] + section + original[end:]
    if updated != original:
        ini.write_text(updated, encoding='utf-8')
    return ini

def fw_refresh_ports(_=None,preferred=''):
    from serial.tools import list_ports
    ports=list(list_ports.comports())
    current=preferred or fw_port.value or ''
    fw_port.options=[('Selecione a COM detectada','')]+[(f'{p.device} · {p.description}',p.device) for p in ports]
    fw_port.value=current if current in [v for _,v in fw_port.options] else ''

def fw_refresh_choices(_=None):
    choices = profile_names()
    previous = fw_choice.value
    fw_choice.options = [(name, name) for name in choices]
    fw_choice.value = previous if previous in choices else choices[0] if choices else None
    if fw_choice.value:
        data = assemble(fw_choice.value)
        fw_refresh_ports(preferred=str(data.get('hardware', {}).get('upload_port') or
                                      data.get('esp', {}).get('serial_port') or ''))
        fw_status.value = status_box('Combinação completa', 'OK',
            f"{fw_choice.value}: {data['_official']['profile']}." +
            (' Componentes previstos; simulação mantém todas as saídas físicas desligadas.'
             if data['_official'].get('outputs_deferred') else '')).value
    else:
        fw_status.value = status_box('Combinação completa', 'PENDENTE',
            'Salve uma combinação completa na seção 6.').value

def fw_print(line):
    _fw_lines.append(line)
    del _fw_lines[:-250]
    fw_log.value = ('<pre style="font-size:11px;white-space:pre-wrap;overflow-wrap:anywhere">'
                    + html.escape('\n'.join(_fw_lines[-150:])) + '</pre>')
    if 'Connecting' in line:
        fw_connection.value = '<b>Connecting...</b> aguardando resposta do ESP32.'
    if 'Chip is ESP32' in line:
        fw_connection.value = '<b>ESP32 reconhecido.</b> Solte BOOT.'
    match = re.search(r'Writing at[^\n]*\((\d+)\s*%\)', line)
    if match:
        fw_progress.value = min(100, int(match.group(1)))

def fw_start(action):
    if action == 'upload' and fw_port_row.layout.display == 'none':
        fw_port_row.layout.display = ''
        fw_refresh_ports()
    if action == 'upload' and not fw_port.value:
        fw_status.value = status_box('Porta de gravação USB', 'PENDENTE',
            'Conecte o ESP32 pelo cabo, clique Atualizar COM e selecione a porta detectada.').value
        return
    if not _fw_busy.acquire(blocking=False):
        return
    fw_export.disabled = fw_build.disabled = fw_upload.disabled = True
    fw_progress.layout.display = '' if action == 'upload' else 'none'
    fw_progress.value = 0
    _fw_lines.clear()
    def worker():
        try:
            if action == 'export':
                report = prepare_all()
                deferred=[name for name in report['simulacao']
                          if assemble(name)['_official'].get('outputs_deferred')]
                result = (f"Simulação: {report['simulacao']}; combate real: "
                          f"{report['combate_real']}. Pendências: {report['erros'] or 'nenhuma'}. "
                          f"Cadastro físico adiado nestas simulações: {deferred or 'nenhum'}.")
            else:
                if not fw_choice.value:
                    raise ValueError('Escolha uma combinação completa.')
                ini = fw_ensure_partition()
                fw_print(f'Partição de 3 MB configurada em: {ini}')
                fw_print('Tabela huge_app.csv: 3 MB para firmware, sem atualização OTA.')
                binary = run_pio(fw_choice.value, action == 'upload', fw_port.value, fw_print)
                result = (f'Firmware {fw_choice.value} gravado em {fw_port.value}.'
                          if action == 'upload' else f'Firmware compilado: {binary}')
                if action == 'upload':
                    fw_progress.value = 100
            fw_status.value = status_box('Runtimes e firmware', 'OK', result).value
        except Exception as exc:
            fw_status.value = status_box('Runtimes e firmware', 'FALHA',
                                        f'{type(exc).__name__}: {exc}').value
        finally:
            fw_export.disabled = fw_build.disabled = fw_upload.disabled = False
            _fw_busy.release()
    threading.Thread(target=worker, daemon=True).start()

def fw_changed(change):
    if change['new']:
        try:
            data = assemble(change['new'])
            fw_refresh_ports(preferred=str(data.get('hardware', {}).get('upload_port') or
                                          data.get('esp', {}).get('serial_port') or ''))
            fw_status.value = status_box('Combinação completa', 'OK',
                f"{change['new']} ({data['_official']['profile']}) selecionada."+
                (' Saídas físicas desligadas no perfil de simulação; cadastro mantido.'
                 if data['_official'].get('outputs_deferred') else '')).value
        except Exception as exc:
            fw_status.value = status_box('Combinação completa', 'FALHA', str(exc)).value

fw_choice.observe(fw_changed, names='value')
fw_refresh.on_click(fw_refresh_choices)
fw_port_refresh.on_click(fw_refresh_ports)
fw_export.on_click(lambda _: fw_start('export'))
fw_build.on_click(lambda _: fw_start('build'))
fw_upload.on_click(lambda _: fw_start('upload'))
try:
    fw_refresh_choices()
except Exception as exc:
    fw_status.value = status_box('Configuração', 'FALHA', str(exc)).value

display(widgets.VBox([
    widgets.HTML('<b>Escolha uma combinação completa.</b> Preparar runtimes gera os '
                 'arquivos de configuração; Compilar cria o firmware com partição de 3 MB. '
                 'Ao clicar Gravar, selecione uma COM detectada para a gravação por cabo; a comunicação '
                 'operacional permanece conforme o setup Wi-Fi ou USB.'),
    widgets.HBox([fw_choice, fw_refresh]),
    widgets.HBox([fw_connection]), fw_port_row,
    widgets.HBox([fw_export, fw_build, fw_upload]),
    fw_status, fw_progress,
    widgets.HTML('Antes de gravar: feche o monitor serial, segure BOOT, pressione e solte EN; '
                 'mantendo BOOT, clique Gravar. Solte BOOT quando aparecer Chip is ESP32.'),
    fw_log
], layout=widgets.Layout(width='100%')))


## 8. Conexões e testes físicos, um componente por vez

O quadro lista os componentes previstos, seus terminais e os GPIO do setup oficial. Você pode verificar **ESP32/firmware** pela COM USB e consultar DHTs disponíveis mesmo antes de montar os atuadores. Os testes de motor, servo e relé exigem firmware de combate real com saídas habilitadas, montagem conferida e área livre. Na simulação esses pulsos ficam bloqueados. Um comando aceito pelo ESP32 não comprova movimento físico: confira cada componente visualmente.


In [ ]:
# Diagnóstico individual por USB do perfil oficial. Nunca interprete ACK como movimento medido.
import html, json, threading, time
import ipywidgets as widgets
from IPython.display import display
from serial.tools import list_ports
import serial
import profile_manager

s8_name=official_name()
s8_data=None;s8_error=''
try:
    if not s8_name:raise ValueError('Defina uma combinação oficial na seção 6.')
    _,s8_data=profile_manager.prepare(s8_name)
except Exception as exc:s8_error=str(exc)
s8_hw=(s8_data or {}).get('hardware',{})
s8_esp=(s8_data or {}).get('esp',{})
s8_physical=bool(s8_hw.get('physical_outputs'))
s8_version_ok=False;s8_busy=threading.Lock()
s8_color='laranja (sinal), vermelho (+), marrom ou preto (GND); confirme o chicote'
s8_motor=s8_hw.get('drive_motor_model','motor DC do setup')
s8_driver=s8_hw.get('drive_driver_model','BTS7960')
s8_drive_id=s8_hw.get('drive_driver_id','bts7960')
if s8_drive_id=='drv8833':
    s8_left='AIN1←GPIO25; AIN2←GPIO26; AOUT1/2→motor E; nSLEEP ativo (3,3 V); VM fonte, GND comum'
    s8_right='BIN1←GPIO32; BIN2←GPIO33; BOUT1/2→motor D; mesmo módulo, nSLEEP ativo'
elif s8_drive_id=='l298n':
    s8_left='IN1←GPIO25; IN2←GPIO26; ENA PWM←GPIO27 (remova jumper ENA); OUT1/2→motor E; VSS lógica 5 V; Vs fonte, GND comum'
    s8_right='IN3←GPIO32; IN4←GPIO33; ENB PWM←GPIO23 (remova jumper ENB); OUT3/4→motor D; mesmo módulo L298N'
else:
    s8_left='RPWM←GPIO25; LPWM←GPIO26; R_EN e L_EN←GPIO27; B+/B− fonte; M+/M− motor E; VCC lógica, GND comum'
    s8_right='RPWM←GPIO32; LPWM←GPIO33; R_EN e L_EN←GPIO23; segundo BTS; B+/B− fonte; M+/M− motor D'
s8_rows=[
 ('ESP32 ↔ PC','USB/COM ou AP local','USB dados / Wi-Fi; gravar somente por USB'),
 ('Motor/driver esquerdo',s8_motor+' / '+s8_driver,
     s8_left),
 ('Motor/driver direito',s8_motor+' / '+s8_driver,
     s8_right),
 ('Servo canhão pan',s8_hw.get('pan_servo_model',s8_hw.get('cannon_motor_model','MG996R')),
     'GPIO18→fio de sinal; '+s8_color),
 ('Servo canhão tilt',s8_hw.get('tilt_servo_model',s8_hw.get('cannon_motor_model','MG996R')),
     'GPIO19→fio de sinal; '+s8_color),
 ('Servo plataforma da câmera',s8_hw.get('camera_servo_model',s8_hw.get('camera_servo_kind','SG90').upper()),
     'GPIO4→fio de sinal; '+s8_color),
 ('Microservo foco',s8_hw.get('focus_actuator_model','SG90'),
     'GPIO14→fio laranja (sinal); vermelho +4,8–6 V; marrom GND; confirme o chicote'),
 ('DHT interno','DHT11 · dados '+s8_hw.get('dht_internal_type','DHT22'),'GPIO16→DATA; VCC e GND segundo o módulo; cores não padronizadas'),
 ('DHT externo','DHT11 · dados '+s8_hw.get('dht_external_type','DHT22'),'GPIO17→DATA; VCC e GND segundo o módulo; cores não padronizadas'),
 ('Relé arrefecimento interno','módulo relé','GPIO22→IN; VCC/GND; válvula no COM/NO conforme esquema'),
 ('Relé arrefecimento externo','módulo relé','GPIO13→IN; VCC/GND; válvula no COM/NO conforme esquema'),
 ('Relé válvula do jato','módulo relé','GPIO21→IN; VCC/GND; COM/NO conforme esquema; teste automático não energiza este canal')]
s8_table=widgets.HTML('<div style="overflow-x:auto;max-height:350px;overflow-y:auto">'
    '<table style="border-collapse:collapse;width:100%;font:12px Arial">'
    '<tr style="background:#e5edf5"><th>Componente</th><th>Modelo oficial</th><th>ESP32 → terminal/fio</th></tr>'+
    ''.join('<tr><td style="border-bottom:1px solid #ddd;padding:5px"><b>'+html.escape(a)+'</b></td>'
      '<td style="border-bottom:1px solid #ddd;padding:5px">'+html.escape(b)+'</td>'
      '<td style="border-bottom:1px solid #ddd;padding:5px">'+html.escape(c)+'</td></tr>'
      for a,b,c in s8_rows)+'</table></div>')
s8_notice=widgets.HTML(status_box('Setup oficial','PENDENTE' if s8_error else 'OK',
                         s8_error or s8_name+' · saídas físicas '+('habilitadas' if s8_physical else 'desligadas neste perfil; cadastro preservado para o perfil real')).value)
s8_ports=widgets.Dropdown(description='COM para diagnóstico:',options=[('Atualizar portas','')],
        style={'description_width':'155px'},layout=widgets.Layout(width='520px'))
s8_confirm=widgets.Checkbox(value=False,description='Conferi alimentação, fiação, área livre e mecanismo desacoplado')
s8_states={};s8_buttons={}
s8_specs=[
 ('link','ESP32 e firmware','VERSION',None,0,False),
 ('internal','DHT interno','STATUS',None,0,False),
 ('external','DHT externo','STATUS',None,0,False),
 ('motor_l','Motor esquerdo','MOTOR_L',.15,350,True),
 ('motor_r','Motor direito','MOTOR_R',.15,350,True),
 ('pan','Servo pan','SERVO_PAN',100,550,True),
 ('tilt','Servo tilt','SERVO_TILT',100,550,True),
 ('camera','Servo plataforma','SERVO_CAMERA',100,550,True),
 ('focus','Microservo foco SG90','SERVO_FOCUS',175,550,True),
 ('cool_i','Relé interno','COOL_INT',1,300,True),
 ('cool_e','Relé externo','COOL_EXT',1,300,True)]

def s8_refresh(_=None):
    ports=list(list_ports.comports())
    s8_ports.options=[('Escolha uma COM','')]+[(f'{p.device} · {p.description}',p.device) for p in ports]
    expected=str(s8_hw.get('upload_port') or s8_esp.get('serial_port') or '')
    s8_ports.value=expected if expected in [v for _,v in s8_ports.options] else ''

def s8_set(key,state,detail):
    s8_states[key].value=status_box(next(row[1] for row in s8_specs if row[0]==key),state,detail).value

def s8_run(key,command,value,duration,physical):
    global s8_version_ok
    if not s8_busy.acquire(False):
        s8_set(key,'PENDENTE','Outro teste está em curso.');return
    try:
        if s8_error:raise RuntimeError(s8_error)
        if physical and not s8_physical:
            raise RuntimeError('Setup de simulação: saídas físicas estão desabilitadas no firmware.')
        if physical and not s8_confirm.value:
            raise RuntimeError('Confira as ligações, a fonte e a área livre; marque a confirmação antes do pulso.')
        if key!='link' and not s8_version_ok:
            raise RuntimeError('Clique Verificar ESP/firmware primeiro; a assinatura precisa corresponder ao setup oficial.')
        port=s8_ports.value
        if not port:raise RuntimeError('Selecione a COM USB do ESP32. Feche o runtime/monitor serial.')
        s8_set(key,'TESTANDO',f'Abrindo {port} e aguardando resposta...')
        with serial.Serial(port,460800,timeout=.15,write_timeout=1) as dev:
            time.sleep(1.8)  # algumas placas reiniciam ao abrir a COM
            try:dev.reset_input_buffer()
            except Exception:pass
            if key!='link':
                dev.write(b'VERSION\n');dev.flush()
                verify_until=time.monotonic()+2
                version_line=''
                while time.monotonic()<verify_until:
                    line=dev.readline().decode('utf-8','replace').strip()
                    if line.startswith('VERSION,'):
                        version_line=line;break
                parts=version_line.split(',')
                if len(parts)<5 or parts[-3]!=s8_data['_official']['firmware_signature']:
                    raise RuntimeError('Esta COM não responde com a assinatura do firmware oficial.')
            wire=(command if key=='link' or command=='STATUS' else
                  f'TEST {command} {value} {duration}')+'\n'
            dev.write(wire.encode('ascii'));dev.flush()
            deadline=time.monotonic()+(4 if command=='STATUS' else 3.5)
            lines=[]
            while time.monotonic()<deadline:
                line=dev.readline().decode('utf-8','replace').strip()
                if line:
                    lines.append(line)
                    if (command=='VERSION' and line.startswith('VERSION,')) or (
                         command=='STATUS' and line.startswith('SENSOR,')) or (
                         command not in ('VERSION','STATUS') and
                         (line=='DIAG DONE' or line.startswith('DIAG REFUSED'))):break
            if command=='VERSION':
                ver=next((x for x in lines if x.startswith('VERSION,')),None)
                if not ver:raise RuntimeError('Sem VERSION do ESP32. Confirme firmware, cabo e COM.')
                parts=ver.split(',')
                expected=s8_data['_official']['firmware_signature']
                s8_version_ok=len(parts)>=5 and parts[-3]==expected
                if not s8_version_ok:raise RuntimeError('Firmware de outra combinação/versão. Compile e grave a seleção oficial.')
                s8_set(key,'OK','ESP32 respondeu pela COM e assinatura corresponde à combinação oficial.')
            elif command=='STATUS':
                report=next((x for x in lines if x.startswith('SENSOR,')),None)
                if not report:raise RuntimeError('Sem resposta SENSOR; verifique firmware e COM.')
                parts=report.split(',');idx=1 if key=='internal' else 4
                model_idx,error_idx=(7,8) if key=='internal' else (9,10)
                if len(parts)<15:
                    raise RuntimeError('Resposta SENSOR de versão antiga: compile e grave o firmware oficial atualizado.')
                model,code=parts[model_idx],int(parts[error_idx])
                raw_idx=11 if key=='internal' else 13
                raw=f'{parts[raw_idx]} °C / {parts[raw_idx+1]} % UR'
                causes={0:'amostra válida',1:'timeout de leitura',
                        2:'checksum inválido',3:'valor decodificado fora do domínio do sensor',
                        4:'aguardando primeira amostra'}
                if parts[idx]!='1':
                    raise RuntimeError(f'{model}: {causes.get(code,"erro "+str(code))}; '
                        f'decodificado: {raw}.')
                temperature=float(parts[idx+1]);humidity=float(parts[idx+2])
                s8_set(key,'OK',f'{model}: {temperature:.1f} °C, {humidity:.1f} % UR.')
            else:
                refusal=next((x for x in lines if x.startswith('DIAG REFUSED')),None)
                if refusal:raise RuntimeError(refusal)
                if not any(x.startswith('DIAG START,') for x in lines) or 'DIAG DONE' not in lines:
                    raise RuntimeError('Pulso não foi confirmado por START e DONE; feche a central e confira o firmware.')
                s8_set(key,'OK','Pulso aceito pelo ESP32; confirme visualmente o componente. Sem feedback físico.')
    except Exception as exc:s8_set(key,'FALHA',str(exc))
    finally:s8_busy.release()

s8_items=[]
for key,title,command,value,duration,physical in s8_specs:
    btn=widgets.Button(description='Verificar ESP/firmware' if key=='link' else 'Testar '+title,
                       button_style='info' if key=='link' else '',layout=widgets.Layout(width='225px'))
    state=widgets.HTML(status_box(title,'PENDENTE','Ainda não testado.').value,
                       layout=widgets.Layout(width='75%'))
    s8_buttons[key]=btn;s8_states[key]=state
    btn.on_click(lambda _,k=key,c=command,v=value,d=duration,p=physical:
        threading.Thread(target=s8_run,args=(k,c,v,d,p),daemon=True).start())
    s8_items.append(widgets.HBox([btn,state],layout=widgets.Layout(width='100%')))
s8_update=widgets.Button(description='Atualizar COM');s8_update.on_click(s8_refresh)
s8_refresh()
display(widgets.VBox([s8_notice,s8_table,
    widgets.HTML('<b>Sequência:</b> conferir quadro → selecionar COM → verificar firmware → DHT → pulsos individuais.'),
    widgets.HBox([s8_ports,s8_update]),s8_confirm,*s8_items],
    layout=widgets.Layout(width='100%')))


## 9. Abrir a central do operador

Abra o runtime em processo separado com a combinação completa. Na janela inicial, confira componentes e versão do firmware; no combate real, calibre pan, tilt e plataforma conforme os modelos MG996R/SG90 escolhidos, com câmera USB ao vivo. A central usa a fonte de vídeo escolhida na seção 3 e envia os comandos ao ESP32 por USB ou Wi-Fi local. Para o uso posterior, inicie diretamente `BLAZE_Simulacao.cmd` ou `BLAZE_Combate_Real.cmd`.


In [ ]:
# Seção 9 — abre o runtime independente com uma combinação completa.
import subprocess
import sys
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from profile_manager import names, prepare

runtime_status = widgets.HTML(layout=widgets.Layout(width='100%'))
runtime_sim_button = widgets.Button(description='Abrir runtime de simulação',
    button_style='success',layout=widgets.Layout(min_width='250px'))
runtime_real_button = widgets.Button(description='Abrir runtime de combate real',
    button_style='warning',layout=widgets.Layout(min_width='270px'))
runtime_log_files = []

def _open_runtime(profile):
    try:
        available = names(profile)
        if not available:
            raise ValueError('Nenhuma combinação completa de ' + profile +
                ' está salva. Crie-a na seção 6; para combate real habilite as saídas físicas no hardware.')
        script = Path(ROOT) / 'pc_operador' / ('runtime_simulacao.py' if profile=='simulacao'
                                               else 'runtime_combate_real.py')
        log_dir = Path(ROOT) / 'output' / 'runtime_logs'
        log_dir.mkdir(parents=True,exist_ok=True)
        logfile = log_dir / (profile+'.log')
        stream = logfile.open('a',encoding='utf-8')
        flags = getattr(subprocess,'CREATE_NO_WINDOW',0) if sys.platform=='win32' else 0
        process = subprocess.Popen([sys.executable,str(script)],cwd=str(script.parent),
            stdout=stream,stderr=subprocess.STDOUT,creationflags=flags)
        stream.close()
        runtime_log_files.append((process,logfile))
        runtime_status.value = status_box('Central independente','OK',
            f'Processo {process.pid} iniciado. Escolha um setup completo na janela externa. Log: {logfile}').value
    except Exception as exc:
        runtime_status.value = status_box('Central independente','FALHA',
            f'{type(exc).__name__}: {exc}').value

runtime_sim_button.on_click(lambda _:_open_runtime('simulacao'))
runtime_real_button.on_click(lambda _:_open_runtime('combate_real'))
display(widgets.VBox([
    widgets.HTML('Os dois programas abaixo também podem ser executados diretamente, sem abrir este '
        'notebook. Em cada janela você escolhe <b>uma combinação completa</b>, abre a central, '
        'compila ou grava o firmware correspondente no ESP32.'),
    widgets.HBox([runtime_sim_button,runtime_real_button]),runtime_status
],layout=widgets.Layout(width='100%')))


## 10. Organizar e publicar no GitHub

**Conferir envio** clona temporariamente a branch atual e mostra cada inclusão ou exclusão. O botão **Enviar revisão ao GitHub** substitui os arquivos atuais da branch e publica o programa em `Protótipo V0.0/` por um commit normal, mesmo quando este pacote veio de um ZIP sem `.git`. O histórico anterior permanece disponível.

Preencha a URL do repositório e, quando necessário, o nome e e-mail do commit. O Git for Windows cuida da autenticação. Modelos treinados e setups portáteis são incluídos; vídeos locais, senhas e firmware compilado ficam no PC. Para selecionar pastas, comparar ou baixar arquivos e editar a estrutura existente do repositório, abra `scripts/gerenciar_github.py` no VS Code. Veja `docs/gerenciar_github.md`.


In [ ]:
"""Substituição revisável do GitHub em Protótipo V0.0, sem clone manual."""
import copy
import datetime
import json
import os
import re
import shutil
import subprocess
import tempfile
from pathlib import Path

import ipywidgets as widgets
from IPython.display import clear_output, display

GIT_ROOT = ROOT.resolve()
GIT_TARGET = 'Protótipo V0.0'
GIT_FILES = [
    '.gitignore', 'README.md', 'BLAZE_Simulacao.cmd', 'BLAZE_Combate_Real.cmd',
    'Abrir_Gerenciador_GitHub.cmd',
    'pc_operador/teste_configuracao.ipynb', 'pc_operador/runtime.py',
    'pc_operador/actuator_catalog.py', 'pc_operador/profile_manager.py',
    'pc_operador/package_variant.py', 'pc_operador/launcher.py',
    'pc_operador/central_operador.py', 'pc_operador/runtime_simulacao.py',
    'pc_operador/runtime_combate_real.py', 'pc_operador/config.example.json',
    'pc_operador/requirements.txt', 'esp32/platformio.ini',
    *[f'pc_operador/parameters_setups/defaults/{name}.json' for name in
      ('connection', 'vision', 'control', 'hardware')],
    *[p.relative_to(GIT_ROOT).as_posix() for base, pattern in
      (('scripts', '*'), ('docs', '*.md'), ('docs', '*.jpg'),
       ('.vscode', '*.json'), ('pc_operador/data', '**/dataset.yaml'),
       ('esp32/src', '*'), ('esp32/include', '*'), ('pc_operador/models', '*.pt'))
      for p in sorted((GIT_ROOT / base).glob(pattern)) if p.is_file() and
      p.name not in ('secrets.h', 'setup_generated.h')],
]
GIT_USER_FILES = [
    f'pc_operador/parameters_setups/user/{name}.json'
    for name in ('connection', 'vision', 'control', 'hardware', 'combinations',
                 'camera_cannon_calibrations')
]
GIT_SECRET_KEYS = ('password', 'ssid', 'token', 'secret', 'api_key', 'bt_pin', 'credential')
GIT_ABSOLUTE = re.compile(r'^(?:[A-Za-z]:[\\/]|/|\\\\)')
GIT_GITHUB = re.compile(r'^(?:https://github\.com/[^/@\s]+/[^/@\s]+|git@github\.com:[^/@\s]+/[^/@\s]+?)(?:\.git)?$')

def git_exec(*args, cwd=GIT_ROOT, check=True):
    result = subprocess.run(['git', '-C', str(cwd), *map(str, args)],
                            text=True, capture_output=True, check=False)
    if check and result.returncode:
        raise RuntimeError((result.stderr or result.stdout).strip() or 'Comando Git falhou.')
    return result.stdout.strip()

def git_detect_remote():
    if not shutil.which('git'):
        return ''
    top = git_exec('rev-parse', '--show-toplevel', check=False)
    if top and Path(top).resolve() == GIT_ROOT:
        return git_exec('remote', 'get-url', 'origin', check=False)
    return ''

def git_config(key):
    return git_exec('config', '--get', key, check=False) if shutil.which('git') else ''

def git_sources():
    return sorted({p for p in GIT_FILES + GIT_USER_FILES
                   if (GIT_ROOT / p).is_file()})

def git_portable_json(relative):
    """Prepara um JSON para outro PC sem alterar o setup desta instalação."""
    data = copy.deepcopy(json.loads((GIT_ROOT / relative).read_text(encoding='utf-8')))
    notes = []
    def clean(value, key=''):
        if isinstance(value, dict):
            result = {}
            for k, item in value.items():
                if any(secret in k.casefold() for secret in GIT_SECRET_KEYS):
                    if item not in ('', None, {}, []):notes.append('credenciais locais omitidas')
                    continue
                if k == 'trajectory_export_dir':
                    result[k] = '../output/visao'
                    continue
                result[k] = clean(item, k)
            if isinstance(result.get('vision'), dict):
                vision=result['vision']
                for model_key, fallback in [('fire_model','fire_yolo_best.pt'),
                                            ('person_model','yolo11n.pt'),
                                            ('firefighter_model','')]:
                    raw=str(vision.get(model_key) or '')
                    if raw:
                        name=raw.replace('\\','/').split('/')[-1]
                        local=GIT_ROOT / 'pc_operador/models' / name
                        if local.is_file():
                            vision[model_key]='models/'+name
                        else:
                            vision[model_key]='models/'+fallback if fallback else ''
                            notes.append(f'{model_key}: modelo local não incluído; usado padrão')
                source=str(vision.get('camera_source',''))
                if (vision.get('source_type')=='video' or GIT_ABSOLUTE.match(source) or
                        source.lower().startswith(('rtsp://','rtsps://','http://','https://'))):
                    vision['source_type']='local'
                    vision['camera_source']=0
                    notes.append('vídeo local substituído por câmera USB 0 na cópia')
                vision['trajectory_export_dir']='../output/visao'
            return result
        if isinstance(value, list):return [clean(v,key) for v in value]
        if isinstance(value, str) and key in (
                'fire_model','person_model','firefighter_model','camera_source'):
            return value  # normalizado acima pela seção vision
        if isinstance(value, str) and re.match(r'^\w+://[^/]*@',value):
            notes.append(f'{key}: URL com credencial omitida')
            return ''
        if isinstance(value, str) and GIT_ABSOLUTE.match(value):
            notes.append(f'{key}: caminho local omitido')
            return ''
        return value
    portable=clean(data)
    return portable, sorted(set(notes))

def git_target_prefix(clone):
    """Todos os arquivos do programa ficarão nesta pasta da branch."""
    return GIT_TARGET

def git_stage_copy(clone, prefix, sources):
    staged, notes = [], []
    # A exclusão fica apenas no clone temporário até o usuário apertar Enviar.
    tracked=[item for item in git_exec('ls-files','-z',cwd=clone).split('\0') if item]
    if tracked:git_exec('rm','-r','-f','--',*tracked,cwd=clone)
    for relative in sources:
        destination=clone/prefix/relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        if relative in GIT_USER_FILES:
            portable, file_notes=git_portable_json(relative)
            destination.write_text(json.dumps(portable,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
            notes.extend(file_notes)
        else:
            shutil.copy2(GIT_ROOT/relative,destination)
        staged.append((Path(prefix)/relative).as_posix())
    # Os arquivos antigos são removidos do commit; -f inclui os modelos do pacote.
    git_exec('add', '-f', '--', *staged, cwd=clone)
    return staged, sorted(set(notes))

git_output=widgets.Output(layout=widgets.Layout(max_height='230px',overflow_y='auto'))
git_state=widgets.HTML()
git_url=widgets.Text(value=git_detect_remote(),description='GitHub:',
                     placeholder='https://github.com/organização/repositório.git',
                     style={'description_width':'85px'},layout=widgets.Layout(width='660px'))
git_url_row=widgets.HBox([git_url])
git_name=widgets.Text(value=git_config('user.name'),description='Nome Git:',
                      layout=widgets.Layout(width='320px'))
git_email=widgets.Text(value=git_config('user.email'),description='E-mail Git:',
                       layout=widgets.Layout(width='400px'))
git_identity_row=widgets.HBox([git_name,git_email])
git_check=widgets.Button(description='Conferir envio',button_style='info',
                         layout=widgets.Layout(width='170px'))
git_send=widgets.Button(description='Enviar revisão ao GitHub',button_style='success',
                        layout=widgets.Layout(width='240px'))
git_send.disabled=True
git_checked={'fingerprint':None}

def git_fingerprint():
    return ((git_detect_remote() or git_url.value.strip()).rstrip('/'),
            git_name.value.strip(),git_email.value.strip(),
            tuple((path,(GIT_ROOT/path).stat().st_size,(GIT_ROOT/path).stat().st_mtime_ns)
                  for path in git_sources()))

def git_mark_dirty(_=None):
    git_checked['fingerprint']=None
    git_send.disabled=True
    git_refresh_fields()
    git_state.value='<b>Conferência pendente.</b> Clique Conferir envio antes de publicar.'

def git_refresh_fields():
    if git_detect_remote():git_url_row.layout.display='none'
    else:git_url_row.layout.display='flex'
    git_identity_row.layout.display=('none' if git_config('user.name') and
                                     git_config('user.email') else 'flex')

def git_preflight():
    errors, notes=[], []
    if not shutil.which('git'):
        errors.append('Git não encontrado. Instale o Git for Windows para enviar ao GitHub.')
        return errors,notes
    remote=(git_detect_remote() or git_url.value.strip()).rstrip('/')
    if not remote or not GIT_GITHUB.fullmatch(remote):
        errors.append('Informe a URL HTTPS ou SSH do repositório GitHub.')
    if not git_name.value.strip() or not git_email.value.strip():
        errors.append('Informe nome e e-mail do autor do commit; os demais campos são automáticos.')
    required=['pc_operador/teste_configuracao.ipynb', 'pc_operador/runtime.py',
              'pc_operador/profile_manager.py','pc_operador/central_operador.py',
              'esp32/src/main.cpp', 'pc_operador/models/fire_yolo_best.pt',
              'pc_operador/models/yolo11n.pt']
    for path in required:
        if not (GIT_ROOT/path).is_file():errors.append('Arquivo do pacote ausente: '+path)
    for path in GIT_USER_FILES:
        if not (GIT_ROOT/path).is_file():continue
        try:
            _,changes=git_portable_json(path)
            notes.extend(f'{path}: {change}' for change in changes)
        except (OSError,ValueError,TypeError) as exc:
            errors.append(f'{path}: JSON inválido: {exc}')
    return errors,sorted(set(notes))

def git_show_check(_=None):
    git_refresh_fields()
    git_checked['fingerprint']=None
    git_send.disabled=True
    with git_output:
        clear_output()
        errors,notes=git_preflight()
        if errors:
            for item in errors:print('PENDENTE:',item)
            git_state.value='<b>Envio pendente.</b> Preencha apenas os campos acima indicados.'
        else:
            remote=(git_detect_remote() or git_url.value.strip()).rstrip('/')
            print('Repositório:',remote)
            print('Arquivos locais incluídos:',len(git_sources()),
                  '· modelos treinados incluídos automaticamente.')
            print('Nenhum vídeo, credencial local ou firmware compilado será enviado.')
            if _ is None:
                print('Clique Conferir envio para examinar a branch e o destino remoto.')
                git_state.value='<b>Conferência pendente.</b>'
            else:
                try:
                    git_state.value='<b>Conferindo branch e arquivos remotos...</b>'
                    with tempfile.TemporaryDirectory(prefix='blaze-git-check-') as temp:
                        clone=Path(temp)/'clone'
                        result=subprocess.run(['git','clone','--depth','1','--quiet',remote,str(clone)],
                                              capture_output=True,text=True,check=False)
                        if result.returncode:
                            raise RuntimeError('Não foi possível acessar o repositório. Confira URL e autenticação. '
                                +(result.stderr or '').strip()[-500:])
                        git_exec('config','core.quotePath','false',cwd=clone)
                        branch=git_exec('branch','--show-current',cwd=clone,check=False) or 'main'
                        if not git_exec('branch','--show-current',cwd=clone,check=False):
                            git_exec('checkout','-B',branch,cwd=clone)
                        prefix=git_target_prefix(clone)
                        tracked=[item for item in git_exec('ls-files','-z',cwd=clone).split('\0') if item]
                        _,adjustments=git_stage_copy(clone,prefix,git_sources())
                        changed=git_exec('diff','--cached','--name-status',cwd=clone).splitlines()
                        print('Branch:',branch,'· pasta:',prefix+'/' if prefix else 'raiz')
                        print('Alterações preparadas:',len(changed),
                              '· arquivos atuais removidos:',len(tracked))
                        for path in changed:print('  ',path)
                        notes.extend(adjustments)
                    git_checked['fingerprint']=git_fingerprint()
                    git_send.disabled=False
                    git_state.value='<b>Conferido.</b> Revise destino e arquivos acima antes de enviar.'
                except Exception as exc:
                    print('FALHA NA CONFERÊNCIA:',exc)
                    git_state.value='<b>Conferência falhou.</b>'
        for item in notes:print('AJUSTE NA CÓPIA:',item)

def git_publish(_):
    git_send.disabled=True
    with git_output:
        clear_output()
        try:
            errors,notes=git_preflight()
            if errors:raise RuntimeError(' | '.join(errors))
            if git_checked['fingerprint']!=git_fingerprint():
                raise RuntimeError('Dados ou arquivos mudaram desde a conferência. Clique Conferir envio novamente.')
            remote=(git_detect_remote() or git_url.value.strip()).rstrip('/')
            git_state.value='<b>Obtendo a branch do repositório...</b>'
            with tempfile.TemporaryDirectory(prefix='blaze-git-') as temp:
                clone=Path(temp)/'clone'
                result=subprocess.run(['git','clone','--depth','1','--quiet',remote,str(clone)],
                                      capture_output=True,text=True,check=False)
                if result.returncode:
                    raise RuntimeError('Não foi possível acessar o repositório. '
                        'Confira URL, autenticação Git e sua permissão de escrita. '
                        +(result.stderr or '').strip()[-500:])
                git_exec('config','core.quotePath','false',cwd=clone)
                branch=git_exec('branch','--show-current',cwd=clone,check=False)
                if not branch:
                    branch='main';git_exec('checkout','-B',branch,cwd=clone)
                prefix=git_target_prefix(clone)
                git_state.value='<b>Preparando arquivos portáteis...</b>'
                staged,changes=git_stage_copy(clone,prefix,git_sources())
                if not git_exec('diff','--cached','--name-only',cwd=clone):
                    print('Sem modificações para enviar. Branch remota:',branch)
                    git_state.value='<b>Repositório atualizado.</b>'
                    return
                git_exec('config','user.name',git_name.value.strip(),cwd=clone)
                git_exec('config','user.email',git_email.value.strip(),cwd=clone)
                destination=(prefix+'/' if prefix else 'raiz do repositório')
                print('Destino:',destination,'· branch:',branch,'· arquivos:',len(staged))
                for item in sorted(set(notes+changes)):print('AJUSTE NA CÓPIA:',item)
                git_exec('commit','-m','Organizar BLAZE em Protótipo V0.0',cwd=clone)
                git_state.value='<b>Enviando commit ao GitHub...</b>'
                git_exec('push','-u','origin','HEAD',cwd=clone)
                print('Revisão enviada. Credenciais, vídeos e firmware compilado ficaram neste PC.')
                git_state.value='<b>Envio concluído.</b>'
        except Exception as exc:
            git_state.value='<b>Envio não concluído.</b> Corrija a mensagem abaixo e tente novamente.'
            print('FALHA:',exc)
        finally:
            git_checked['fingerprint']=None
            git_send.disabled=True

git_url.observe(git_mark_dirty,names='value')
git_name.observe(git_mark_dirty,names='value')
git_email.observe(git_mark_dirty,names='value')
git_check.on_click(git_show_check)
git_send.on_click(git_publish)
git_refresh_fields()
display(widgets.HTML('<h3>Conferir e substituir o conteúdo do GitHub</h3><p>Confira a lista: '
                     'todos os arquivos atuais da branch serão removidos e a cópia portátil '
                     'será adicionada em <b>Protótipo V0.0/</b>. O envio ocorre somente após '
                     'clicar Enviar revisão. Para baixar, comparar, mover ou excluir arquivos '
                     'individualmente, abra <code>scripts/gerenciar_github.py</code> no VS Code.</p>'),
        git_url_row,git_identity_row,widgets.HBox([git_check,git_send]),git_state,git_output)
git_show_check()
